<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/13_Synthetic_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# =============================================================================
# NOTEBOOK 13 — SYNTHETIC DATA GENERATION
# Research: SPP-GAN — Privacy-Preserving Synthetic Tabular Data Generation
# Format: 26 independent Google Colab code sections, aligned with Notebook 12
# =============================================================================

# =============================================================================
# SECTION 1 — HEADER & SCOPE
# =============================================================================

print("=" * 100)
print("NOTEBOOK 13 — SYNTHETIC DATA GENERATION")
print("=" * 100)

NOTEBOOK_NAME = "Notebook 13 — Synthetic Data Generation"
FRAMEWORK_NAME = "SPP-GAN"
DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

print("Framework :", FRAMEWORK_NAME)
print("Datasets  :", DATASET_IDS)
print("Purpose   : Generate, restore, validate, fingerprint, and persist synthetic data.")

NOTEBOOK 13 — SYNTHETIC DATA GENERATION
Framework : SPP-GAN
Datasets  : ['adult_income', 'bank_marketing', 'diabetes_130us']
Purpose   : Generate, restore, validate, fingerprint, and persist synthetic data.


In [31]:
# =============================================================================
# SECTION 2 — LOAD CONFIGURATION
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
import random
import time
import gc
import platform
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

print()
print("-" * 100)
print("GOOGLE DRIVE VERIFICATION")
print("-" * 100)

if not MYDRIVE_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_ROOT), force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            "Google Drive mount failed. Mount Drive in Colab and rerun Section 2."
        ) from exc

if not MYDRIVE_ROOT.exists() or not MYDRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"MyDrive is unavailable: {MYDRIVE_ROOT}")

if not PROJECT_ROOT.exists() or not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Canonical SPP-GAN project root not found: {PROJECT_ROOT}"
    )

NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"
NB12_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_12"

NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
NB13_CONFIG_ROOT = NB13_ROOT / "configuration"
NB13_SYNTHETIC_ROOT = NB13_ROOT / "synthetic_datasets"
NB13_METADATA_ROOT = NB13_ROOT / "metadata"
NB13_MANIFEST_ROOT = NB13_ROOT / "manifests"
NB13_VALIDATION_ROOT = NB13_ROOT / "validation"
NB13_LOG_ROOT = NB13_ROOT / "logs"

for directory in [
    NB13_CONFIG_ROOT,
    NB13_SYNTHETIC_ROOT,
    NB13_METADATA_ROOT,
    NB13_MANIFEST_ROOT,
    NB13_VALIDATION_ROOT,
    NB13_LOG_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 2025
GENERATION_REPETITIONS = 1
SAMPLE_SIZE_OVERRIDE = None       # None = use persisted Notebook 12 training-row count
GENERATION_BATCH_SIZE = 1024      # bounded batches for Colab RAM/GPU compatibility
SAVE_PARQUET = False

print(f"✓ MyDrive     : {MYDRIVE_ROOT}")
print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Notebook 02 : {NB02_ROOT}")
print(f"✓ Notebook 12 : {NB12_ROOT}")
print(f"✓ Notebook 13 : {NB13_ROOT}")

if not NB02_ROOT.exists():
    raise FileNotFoundError(f"Notebook 02 root not found: {NB02_ROOT}")
if not NB12_ROOT.exists():
    raise FileNotFoundError(f"Notebook 12 root not found: {NB12_ROOT}")


----------------------------------------------------------------------------------------------------
GOOGLE DRIVE VERIFICATION
----------------------------------------------------------------------------------------------------
✓ MyDrive     : /content/drive/MyDrive
✓ Project root: /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 02 : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Notebook 12 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12
✓ Notebook 13 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13


In [34]:
# ==================================================================================================
# SECTION 3 — DISCOVER TRAINED MODELS
# ==================================================================================================

print("=" * 100)
print("SECTION 3 — DISCOVER TRAINED MODELS")
print("=" * 100)

from pathlib import Path
import json
import pandas as pd

# --------------------------------------------------------------------------------------------------
# 1. Validate Required Paths
# --------------------------------------------------------------------------------------------------

assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"
assert NB12_ROOT.exists(), f"Notebook 12 root does not exist: {NB12_ROOT}"

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Notebook 12  : {NB12_ROOT}")

# --------------------------------------------------------------------------------------------------
# 2. Define Dataset Registry
# --------------------------------------------------------------------------------------------------

DATASET_NAMES = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

# --------------------------------------------------------------------------------------------------
# 3. Define Model Discovery Roots
# --------------------------------------------------------------------------------------------------

MODEL_SEARCH_ROOTS = [
    NB12_ROOT / "models",
    NB12_ROOT / "checkpoints",
    NB12_ROOT / "artifacts",
    NB12_ROOT / "training",
    PROJECT_ROOT / "models",
]

MODEL_EXTENSIONS = {
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".joblib",
}

# --------------------------------------------------------------------------------------------------
# 4. Discover Model Artifacts
# --------------------------------------------------------------------------------------------------

MODEL_CANDIDATES = {dataset: [] for dataset in DATASET_NAMES}

for search_root in MODEL_SEARCH_ROOTS:
    if not search_root.exists():
        continue

    for artifact_path in search_root.rglob("*"):
        if not artifact_path.is_file():
            continue

        if artifact_path.suffix.lower() not in MODEL_EXTENSIONS:
            continue

        path_text = str(artifact_path).lower()

        if any(
            excluded in path_text
            for excluded in [
                "__pycache__",
                "/.ipynb_checkpoints/",
                "/temp/",
                "/tmp/",
            ]
        ):
            continue

        for dataset in DATASET_NAMES:
            if dataset in path_text:
                MODEL_CANDIDATES[dataset].append(artifact_path)
                break

# Remove duplicate paths while preserving deterministic ordering.
for dataset in DATASET_NAMES:
    MODEL_CANDIDATES[dataset] = sorted(
        set(MODEL_CANDIDATES[dataset]),
        key=lambda path: str(path).lower(),
    )

# --------------------------------------------------------------------------------------------------
# 5. Identify Final-Generator Candidates
# --------------------------------------------------------------------------------------------------

def model_candidate_priority(path):
    """
    Rank likely final-generator artifacts ahead of intermediate checkpoints.
    This ranks candidates only; it does not establish the empirically selected
    model or identify the best training repetition.
    """
    name = path.name.lower()
    parts = [part.lower() for part in path.parts]

    if name in {"generator_final.pt", "generator_final.pth"}:
        return 0

    if name in {"final_generator.pt", "final_generator.pth"}:
        return 1

    if name in {"generator_best.pt", "generator_best.pth"}:
        return 2

    if name in {"best_generator.pt", "best_generator.pth"}:
        return 3

    if "final_model" in name or "best_model" in name:
        return 4

    if "generator" in name and "checkpoint" not in name:
        return 5

    if any("checkpoint" in part for part in parts):
        return 7

    return 6


for dataset in DATASET_NAMES:
    MODEL_CANDIDATES[dataset] = sorted(
        MODEL_CANDIDATES[dataset],
        key=lambda path: (
            model_candidate_priority(path),
            str(path).lower(),
        ),
    )

# --------------------------------------------------------------------------------------------------
# 6. Search for Notebook 12 Model Registries and Manifests
# --------------------------------------------------------------------------------------------------

REGISTRY_SEARCH_ROOTS = [
    NB12_ROOT / "manifests",
    NB12_ROOT / "metadata",
    NB12_ROOT / "configuration",
    NB12_ROOT / "registries",
    NB12_ROOT / "artifacts",
    NB12_ROOT,
    PROJECT_ROOT / "registries",
]

REGISTRY_EXTENSIONS = {".csv", ".json", ".parquet"}

REGISTRY_KEYWORDS = (
    "registry",
    "manifest",
    "final_model",
    "model_selection",
    "training_summary",
)

registry_files = []

for search_root in REGISTRY_SEARCH_ROOTS:
    if not search_root.exists():
        continue

    for file_path in search_root.rglob("*"):
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in REGISTRY_EXTENSIONS:
            continue

        path_text = str(file_path).lower()

        if any(
            excluded in path_text
            for excluded in [
                "__pycache__",
                "/.ipynb_checkpoints/",
                "/checkpoints/",
            ]
        ):
            continue

        if any(keyword in file_path.name.lower() for keyword in REGISTRY_KEYWORDS):
            registry_files.append(file_path)

registry_files = sorted(
    set(registry_files),
    key=lambda path: str(path).lower(),
)

# Prefer recognized registry filenames, then other potentially relevant metadata.
PREFERRED_REGISTRY_NAMES = {
    "final_model_registry.csv",
    "notebook_12_model_manifest.csv",
    "notebook_12b_final_model_registry.csv",
}

preferred_registry_files = [
    path
    for path in registry_files
    if path.name.lower() in PREFERRED_REGISTRY_NAMES
]

MODEL_REGISTRY_PATH = (
    preferred_registry_files[0]
    if preferred_registry_files
    else None
)

MODEL_REGISTRY_AVAILABLE = MODEL_REGISTRY_PATH is not None
MODEL_REGISTRY_DF = None

if MODEL_REGISTRY_AVAILABLE:
    try:
        suffix = MODEL_REGISTRY_PATH.suffix.lower()

        if suffix == ".csv":
            MODEL_REGISTRY_DF = pd.read_csv(MODEL_REGISTRY_PATH)

        elif suffix == ".parquet":
            MODEL_REGISTRY_DF = pd.read_parquet(MODEL_REGISTRY_PATH)

        elif suffix == ".json":
            with open(MODEL_REGISTRY_PATH, "r", encoding="utf-8") as file:
                registry_data = json.load(file)

            if isinstance(registry_data, list):
                MODEL_REGISTRY_DF = pd.DataFrame(registry_data)
            elif isinstance(registry_data, dict):
                if isinstance(registry_data.get("models"), list):
                    MODEL_REGISTRY_DF = pd.DataFrame(registry_data["models"])
                else:
                    MODEL_REGISTRY_DF = pd.json_normalize(registry_data)
            else:
                raise ValueError("Unsupported JSON registry structure.")

        print(f"✓ Recognized model registry: {MODEL_REGISTRY_PATH}")
        print(f"✓ Registry rows            : {len(MODEL_REGISTRY_DF):,}")

    except Exception as exc:
        MODEL_REGISTRY_AVAILABLE = False
        MODEL_REGISTRY_DF = None
        print(f"⚠ Registry could not be loaded: {exc}")

else:
    print("⚠ No recognized final-model registry was found.")
    print("  Discovered metadata candidates:")

    if registry_files:
        for path in registry_files[:30]:
            print(f"  - {path}")
    else:
        print("  - No registry/manifest/selection/training-summary files found.")

    print(
        "  Candidate model artifacts will be reported, but the selected "
        "training repetition must be established before generation."
    )

# --------------------------------------------------------------------------------------------------
# 7. Build Model Discovery Summary
# --------------------------------------------------------------------------------------------------

summary_rows = []

for dataset in DATASET_NAMES:
    candidates = MODEL_CANDIDATES[dataset]

    final_generator_candidates = [
        path
        for path in candidates
        if path.name.lower() in {
            "generator_final.pt",
            "generator_final.pth",
            "final_generator.pt",
            "final_generator.pth",
        }
    ]

    summary_rows.append(
        {
            "dataset": dataset,
            "candidate_count": len(candidates),
            "final_generator_candidate_count": len(final_generator_candidates),
            "top_candidate": str(candidates[0]) if candidates else None,
            "registry_available": MODEL_REGISTRY_AVAILABLE,
            "registry_path": (
                str(MODEL_REGISTRY_PATH)
                if MODEL_REGISTRY_AVAILABLE
                else None
            ),
            "discovery_status": (
                "CANDIDATES_FOUND"
                if candidates
                else "NO_MODEL_ARTIFACTS_FOUND"
            ),
        }
    )

MODEL_DISCOVERY_SUMMARY = pd.DataFrame(summary_rows)

# --------------------------------------------------------------------------------------------------
# 8. Display Discovery Results
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("MODEL DISCOVERY SUMMARY")
print("-" * 100)

print(MODEL_DISCOVERY_SUMMARY.to_string(index=False))

for dataset in DATASET_NAMES:
    candidates = MODEL_CANDIDATES[dataset]

    print("\n" + "-" * 100)
    print(f"{dataset.upper()} — MODEL CANDIDATES")
    print("-" * 100)
    print(f"Total candidates: {len(candidates)}")

    if candidates:
        print("Highest-priority candidates:")

        for path in candidates[:10]:
            print(f"  - {path}")

    else:
        print("⚠ No model artifacts found for this dataset.")

# --------------------------------------------------------------------------------------------------
# 9. Save Discovery Summary
# --------------------------------------------------------------------------------------------------

NB13_LOG_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_DISCOVERY_SUMMARY_PATH = (
    NB13_LOG_ROOT / "model_discovery_summary.csv"
)

MODEL_DISCOVERY_SUMMARY.to_csv(
    MODEL_DISCOVERY_SUMMARY_PATH,
    index=False,
)

print("\n" + "-" * 100)
print("SECTION 3 COMPLETED")
print("-" * 100)
print(f"✓ Discovery summary saved: {MODEL_DISCOVERY_SUMMARY_PATH}")
print(
    "⚠ Candidate ranking is not final model selection. "
    "Use Notebook 12's documented selection criterion or authoritative "
    "training artifact to choose the generator and repetition."
)

SECTION 3 — DISCOVER TRAINED MODELS
✓ Project root : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 12  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12
✓ Recognized model registry: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/registries/notebook_12B_final_model_registry.csv
✓ Registry rows            : 15

----------------------------------------------------------------------------------------------------
MODEL DISCOVERY SUMMARY
----------------------------------------------------------------------------------------------------
       dataset  candidate_count  final_generator_candidate_count                                                                                                                top_candidate  registry_available                                                                                                          registry_path discovery_status
  adult_income              176                                5   

In [38]:
# ==================================================================================================
# SECTION 4 — VALIDATE MODEL ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE MODEL ARTIFACTS")
print("=" * 100)

import hashlib
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------------------------------------
# 1. Define SHA-256 Fingerprinting
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 2. Validate Notebook 12 Registry
# --------------------------------------------------------------------------------------------------

assert (
    MODEL_REGISTRY_AVAILABLE
    and isinstance(MODEL_REGISTRY_DF, pd.DataFrame)
    and not MODEL_REGISTRY_DF.empty
), "Notebook 12 final-model registry is unavailable or empty."

required_registry_columns = {
    "dataset",
    "repetition_id",
    "status",
    "final_model_path",
    "final_model_sha256",
    "final_model_size_bytes",
}

missing_columns = required_registry_columns.difference(
    MODEL_REGISTRY_DF.columns
)

assert not missing_columns, (
    f"Registry is missing required columns: {sorted(missing_columns)}"
)

registry = MODEL_REGISTRY_DF.copy()

registry["dataset"] = (
    registry["dataset"].astype(str).str.strip().str.lower()
)

registry["status"] = (
    registry["status"].astype(str).str.strip().str.upper()
)

registry["repetition_id"] = pd.to_numeric(
    registry["repetition_id"],
    errors="coerce",
)

registry["final_model_size_bytes"] = pd.to_numeric(
    registry["final_model_size_bytes"],
    errors="coerce",
)

# Validate that the registry contains exactly one record for each dataset/repetition.
assert not registry.duplicated(
    subset=["dataset", "repetition_id"]
).any(), "Duplicate dataset/repetition records found in the model registry."

# This section validates every registered final model. It does not arbitrarily
# declare one repetition to be the best; all PASS repetitions remain identifiable.
MODEL_ARTIFACTS = {}
MODEL_VALIDATION_ROWS = []

# --------------------------------------------------------------------------------------------------
# 3. Validate Every Registered Final Generator
# --------------------------------------------------------------------------------------------------

for _, record in registry.iterrows():
    dataset = record["dataset"]
    repetition_id = record["repetition_id"]
    recorded_path = str(record["final_model_path"]).strip()

    if dataset not in DATASET_NAMES:
        continue

    artifact_path = Path(recorded_path)

    exists = artifact_path.is_file()
    actual_size = artifact_path.stat().st_size if exists else None

    expected_size = record["final_model_size_bytes"]
    expected_sha256 = str(record["final_model_sha256"]).strip().lower()

    actual_sha256 = (
        sha256_file(artifact_path)
        if exists and actual_size > 0
        else None
    )

    size_matches = (
        actual_size is not None
        and pd.notna(expected_size)
        and actual_size == int(expected_size)
    )

    hash_matches = (
        actual_sha256 is not None
        and expected_sha256
        and actual_sha256 == expected_sha256
    )

    registry_pass = record["status"] == "PASS"
    nonempty_file = exists and actual_size > 0

    if (
        registry_pass
        and nonempty_file
        and size_matches
        and hash_matches
    ):
        validation_status = "VALIDATED"
    elif not exists:
        validation_status = "MISSING_FILE"
    elif not nonempty_file:
        validation_status = "EMPTY_FILE"
    elif not size_matches:
        validation_status = "SIZE_MISMATCH"
    elif not hash_matches:
        validation_status = "SHA256_MISMATCH"
    else:
        validation_status = "REGISTRY_STATUS_NOT_PASS"

    MODEL_VALIDATION_ROWS.append(
        {
            "dataset": dataset,
            "repetition_id": (
                int(repetition_id)
                if pd.notna(repetition_id)
                else None
            ),
            "artifact_path": recorded_path,
            "registry_status": record["status"],
            "exists": exists,
            "size_bytes": actual_size,
            "expected_size_bytes": (
                int(expected_size)
                if pd.notna(expected_size)
                else None
            ),
            "size_matches": size_matches,
            "sha256": actual_sha256,
            "expected_sha256": expected_sha256,
            "sha256_matches": hash_matches,
            "selection_source": "NOTEBOOK_12_FINAL_MODEL_REGISTRY",
            "status": validation_status,
        }
    )

    if validation_status == "VALIDATED":
        MODEL_ARTIFACTS.setdefault(dataset, {})[
            int(repetition_id)
        ] = artifact_path


# --------------------------------------------------------------------------------------------------
# 4. Build Validation Report
# --------------------------------------------------------------------------------------------------

MODEL_VALIDATION_DF = pd.DataFrame(MODEL_VALIDATION_ROWS)

NB13_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_VALIDATION_PATH = (
    NB13_VALIDATION_ROOT / "model_artifact_validation.csv"
)

MODEL_VALIDATION_DF.to_csv(
    MODEL_VALIDATION_PATH,
    index=False,
)

print("\n" + "-" * 100)
print("MODEL ARTIFACT VALIDATION SUMMARY")
print("-" * 100)

if MODEL_VALIDATION_DF.empty:
    print("⚠ No registered model artifacts matched the configured datasets.")
else:
    display(MODEL_VALIDATION_DF)

print("\nValidated repetitions by dataset:")

for dataset in DATASET_NAMES:
    repetitions = sorted(MODEL_ARTIFACTS.get(dataset, {}).keys())

    print(
        f"  {dataset}: {len(repetitions)} validated "
        f"repetition(s) {repetitions}"
    )

print(f"\n✓ Validation report saved: {MODEL_VALIDATION_PATH}")

# --------------------------------------------------------------------------------------------------
# 5. Verify Dataset Coverage
# --------------------------------------------------------------------------------------------------

missing_datasets = [
    dataset
    for dataset in DATASET_NAMES
    if not MODEL_ARTIFACTS.get(dataset)
]

if missing_datasets:
    print("\n⚠ Some datasets have no validated generator artifacts:")
    for dataset in missing_datasets:
        print(f"  - {dataset}")
else:
    print("\n✓ All configured datasets have validated generator artifacts.")

print(
    "\nNote: All valid registered repetitions are retained. "
    "The generation section must explicitly determine whether to generate "
    "from every repetition or from a documented selected repetition."
)

SECTION 4 — VALIDATE MODEL ARTIFACTS

----------------------------------------------------------------------------------------------------
MODEL ARTIFACT VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------


,dataset,repetition_id,artifact_path,registry_status,exists,size_bytes,expected_size_bytes,size_matches,sha256,expected_sha256,sha256_matches,selection_source,status
0,adult_income,1,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,3147dc0dd607f0b242d35daed9e4c7a17830f9589fe29b...,3147dc0dd607f0b242d35daed9e4c7a17830f9589fe29b...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
1,adult_income,2,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,60e8d6409ddd8f5f2e9a82cd758829a736e46c4921913e...,60e8d6409ddd8f5f2e9a82cd758829a736e46c4921913e...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
2,adult_income,3,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,48e62c40e789f84822f4c26b75137464996fbff3274b8e...,48e62c40e789f84822f4c26b75137464996fbff3274b8e...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
3,adult_income,4,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712309,2712309,True,183431ff4d80d2d1228f9adad8470986db81c1d5b664d3...,183431ff4d80d2d1228f9adad8470986db81c1d5b664d3...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
4,adult_income,5,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,aa1828fd4b41c1450c7c447f1c4206c1a95213bf9fe0fa...,aa1828fd4b41c1450c7c447f1c4206c1a95213bf9fe0fa...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
5,bank_marketing,1,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,5bb3cc6fcea00cb21f417257963552a16c5e7faaa149ba...,5bb3cc6fcea00cb21f417257963552a16c5e7faaa149ba...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
6,bank_marketing,2,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,e9055c31fab00a8be54c818a8cde714572680908ae9c75...,e9055c31fab00a8be54c818a8cde714572680908ae9c75...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
7,bank_marketing,3,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,797dd47e7c97106cce5d02bc52da26bbc9408567a37877...,797dd47e7c97106cce5d02bc52da26bbc9408567a37877...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
8,bank_marketing,4,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379573,2379573,True,fed615ef19110561dddae97a6d67d63950dcb67adece08...,fed615ef19110561dddae97a6d67d63950dcb67adece08...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
9,bank_marketing,5,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,9ef74c66d380fbbbb5e33c900c00e23f74dcd7c76e9005...,9ef74c66d380fbbbb5e33c900c00e23f74dcd7c76e9005...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED



Validated repetitions by dataset:
  adult_income: 5 validated repetition(s) [1, 2, 3, 4, 5]
  bank_marketing: 5 validated repetition(s) [1, 2, 3, 4, 5]
  diabetes_130us: 5 validated repetition(s) [1, 2, 3, 4, 5]

✓ Validation report saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/model_artifact_validation.csv

✓ All configured datasets have validated generator artifacts.

Note: All valid registered repetitions are retained. The generation section must explicitly determine whether to generate from every repetition or from a documented selected repetition.


In [40]:
# ==================================================================================================
# SECTION 5 — LOAD PREPROCESSORS/SCHEMAS
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — LOAD PREPROCESSORS/SCHEMAS")
print("=" * 100)

import json
import joblib
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------------------------------------
# 1. Define Notebook 02 Artifact Search Roots
# --------------------------------------------------------------------------------------------------

PREPROCESSOR_ROOT_CANDIDATES = [
    NB02_ROOT / "preprocessors",
    NB02_ROOT / "artifacts" / "preprocessors",
    NB02_ROOT / "artifacts",
    NB02_ROOT,
]

SCHEMA_ROOT_CANDIDATES = [
    NB02_ROOT / "schemas",
    NB02_ROOT / "artifacts" / "schemas",
    NB02_ROOT / "artifacts",
    NB02_ROOT,
]

NATIVE_ROOT_CANDIDATES = [
    NB02_ROOT / "native",
    NB02_ROOT / "processed",
    NB02_ROOT / "datasets",
    NB02_ROOT / "splits",
    NB02_ROOT / "artifacts",
    NB02_ROOT,
]

# --------------------------------------------------------------------------------------------------
# 2. Discover Candidate Artifacts Without Arbitrary Selection
# --------------------------------------------------------------------------------------------------

def discover_dataset_artifacts(
    roots,
    dataset,
    extensions,
    keywords=None,
):
    """
    Return all matching artifact paths in deterministic order.
    Multiple matches are preserved so that ambiguous files are not silently
    treated as the canonical artifact.
    """
    keywords = [word.lower() for word in (keywords or [])]
    candidates = set()

    for root in roots:
        if not root.exists():
            continue

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in extensions:
                continue

            path_text = str(path).lower()

            if dataset.lower() not in path_text:
                continue

            if keywords and not any(
                word in path.name.lower() for word in keywords
            ):
                continue

            if any(
                excluded in path_text
                for excluded in [
                    "/.ipynb_checkpoints/",
                    "/__pycache__/",
                    "/temp/",
                    "/tmp/",
                ]
            ):
                continue

            candidates.add(path.resolve())

    return sorted(candidates, key=lambda path: str(path).lower())


# --------------------------------------------------------------------------------------------------
# 3. Initialize Artifact Registries
# --------------------------------------------------------------------------------------------------

PREPROCESSORS = {}
PREPROCESSOR_PATHS = {}
PREPROCESSOR_CANDIDATES = {}

SCHEMAS = {}
SCHEMA_PATHS = {}
SCHEMA_CANDIDATES = {}

REFERENCE_TRAIN = {}
REFERENCE_PATHS = {}
REFERENCE_CANDIDATES = {}

ARTIFACT_DISCOVERY_ROWS = []

# --------------------------------------------------------------------------------------------------
# 4. Discover Preprocessors, Schemas, and Reference Datasets
# --------------------------------------------------------------------------------------------------

for dataset in DATASET_IDS:

    # Search for serialized preprocessing artifacts.
    prep_candidates = discover_dataset_artifacts(
        PREPROCESSOR_ROOT_CANDIDATES,
        dataset,
        {".joblib", ".pkl", ".pickle"},
        ["preprocessor", "transformer", "encoder"],
    )

    # Search for schema-related JSON files.
    schema_candidates = discover_dataset_artifacts(
        SCHEMA_ROOT_CANDIDATES,
        dataset,
        {".json"},
        ["schema", "feature", "manifest"],
    )

    # Search for potential reference datasets.
    native_candidates = discover_dataset_artifacts(
        NATIVE_ROOT_CANDIDATES,
        dataset,
        {".csv"},
        ["native", "processed", "train", "dataset"],
    )

    PREPROCESSOR_CANDIDATES[dataset] = prep_candidates
    SCHEMA_CANDIDATES[dataset] = schema_candidates
    REFERENCE_CANDIDATES[dataset] = native_candidates

    # Only automatically load when exactly one candidate exists.
    # Multiple candidates require an explicit canonical-path decision.
    prep_path = (
        prep_candidates[0]
        if len(prep_candidates) == 1
        else None
    )

    schema_path = (
        schema_candidates[0]
        if len(schema_candidates) == 1
        else None
    )

    native_path = (
        native_candidates[0]
        if len(native_candidates) == 1
        else None
    )

    PREPROCESSOR_PATHS[dataset] = prep_path
    SCHEMA_PATHS[dataset] = schema_path
    REFERENCE_PATHS[dataset] = native_path

    # Load preprocessor.
    if prep_path is not None:
        try:
            PREPROCESSORS[dataset] = joblib.load(prep_path)
            prep_status = "LOADED"
        except Exception as exc:
            prep_status = f"LOAD_FAILED: {exc}"
    elif not prep_candidates:
        prep_status = "NOT_FOUND"
    else:
        prep_status = f"AMBIGUOUS: {len(prep_candidates)} candidates"

    # Load schema.
    if schema_path is not None:
        try:
            with open(schema_path, "r", encoding="utf-8") as handle:
                SCHEMAS[dataset] = json.load(handle)
            schema_status = "LOADED"
        except Exception as exc:
            schema_status = f"LOAD_FAILED: {exc}"
    elif not schema_candidates:
        schema_status = "NOT_FOUND"
    else:
        schema_status = f"AMBIGUOUS: {len(schema_candidates)} candidates"

    # Load reference dataset.
    if native_path is not None:
        try:
            REFERENCE_TRAIN[dataset] = pd.read_csv(
                native_path,
                low_memory=False,
            )
            reference_status = "LOADED"
        except Exception as exc:
            reference_status = f"LOAD_FAILED: {exc}"
    elif not native_candidates:
        reference_status = "NOT_FOUND"
    else:
        reference_status = f"AMBIGUOUS: {len(native_candidates)} candidates"

    ARTIFACT_DISCOVERY_ROWS.append(
        {
            "dataset": dataset,
            "preprocessor_candidates": len(prep_candidates),
            "preprocessor_path": str(prep_path) if prep_path else None,
            "preprocessor_status": prep_status,
            "schema_candidates": len(schema_candidates),
            "schema_path": str(schema_path) if schema_path else None,
            "schema_status": schema_status,
            "reference_candidates": len(native_candidates),
            "reference_path": str(native_path) if native_path else None,
            "reference_status": reference_status,
        }
    )

    print(f"\n{dataset}")
    print(f"  Preprocessor: {prep_status}")
    print(f"  Schema      : {schema_status}")
    print(f"  Reference   : {reference_status}")

    for category, paths in [
        ("Preprocessor candidates", prep_candidates),
        ("Schema candidates", schema_candidates),
        ("Reference candidates", native_candidates),
    ]:
        if paths:
            print(f"  {category}:")
            for path in paths[:10]:
                print(f"    - {path}")

# --------------------------------------------------------------------------------------------------
# 5. Load Notebook 12 Configuration Snapshot
# --------------------------------------------------------------------------------------------------

NB12_CONFIGURATION_PATH = (
    NB12_ROOT / "configuration" / "sppgan_training_configuration.json"
)

NB12_CONFIGURATION = {}

if NB12_CONFIGURATION_PATH.is_file():
    try:
        with open(
            NB12_CONFIGURATION_PATH,
            "r",
            encoding="utf-8",
        ) as handle:
            NB12_CONFIGURATION = json.load(handle)

        print(
            "\n✓ Loaded Notebook 12 configuration:",
            NB12_CONFIGURATION_PATH,
        )

    except Exception as exc:
        print(f"\n⚠ Could not load Notebook 12 configuration: {exc}")

else:
    print(
        "\n⚠ Notebook 12 configuration snapshot not found:",
        NB12_CONFIGURATION_PATH,
    )

# --------------------------------------------------------------------------------------------------
# 6. Save Artifact Discovery Report
# --------------------------------------------------------------------------------------------------

ARTIFACT_DISCOVERY_DF = pd.DataFrame(ARTIFACT_DISCOVERY_ROWS)

NB13_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DISCOVERY_PATH = (
    NB13_VALIDATION_ROOT / "preprocessor_schema_discovery.csv"
)

ARTIFACT_DISCOVERY_DF.to_csv(
    ARTIFACT_DISCOVERY_PATH,
    index=False,
)

print("\n" + "-" * 100)
print("SECTION 5 DISCOVERY SUMMARY")
print("-" * 100)

display(ARTIFACT_DISCOVERY_DF)

print(f"\n✓ Discovery report saved: {ARTIFACT_DISCOVERY_PATH}")

print(
    "\n⚠ Generation must not proceed until the canonical preprocessing artifacts, "
    "feature schema, and reference data are identified and validated."
)

SECTION 5 — LOAD PREPROCESSORS/SCHEMAS

adult_income
  Preprocessor: LOADED
  Schema      : LOADED
  Reference   : LOADED
  Preprocessor candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors/adult_income/train_fitted_preprocessor.joblib
  Schema candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/adult_income/preprocessing_schema.json
  Reference candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/adult_income/train.csv

bank_marketing
  Preprocessor: LOADED
  Schema      : LOADED
  Reference   : LOADED
  Preprocessor candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors/bank_marketing/train_fitted_preprocessor.joblib
  Schema candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/bank_marketing/preprocessing_schema.json
  Reference candidates:
    - /content/drive/MyDrive/SPP_GAN

,dataset,preprocessor_candidates,preprocessor_path,preprocessor_status,schema_candidates,schema_path,schema_status,reference_candidates,reference_path,reference_status
0,adult_income,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED
1,bank_marketing,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED
2,diabetes_130us,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED



✓ Discovery report saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/preprocessor_schema_discovery.csv

⚠ Generation must not proceed until the canonical preprocessing artifacts, feature schema, and reference data are identified and validated.


In [41]:
# =============================================================================
# SECTION 6 — CONFIGURE GENERATION
# =============================================================================

TARGET_DEFAULTS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

def resolve_target(dataset):
    schema = SCHEMAS.get(dataset, {})
    if isinstance(schema, dict):
        for key in ("target", "target_column", "target_name"):
            value = schema.get(key)
            if isinstance(value, str) and value.strip():
                return value
        for parent_key in ("dataset", "metadata", "configuration", "columns"):
            nested = schema.get(parent_key)
            if isinstance(nested, dict):
                for key in ("target", "target_column", "target_name"):
                    value = nested.get(key)
                    if isinstance(value, str) and value.strip():
                        return value
    return TARGET_DEFAULTS[dataset]

TARGET_COLUMNS = {dataset: resolve_target(dataset) for dataset in DATASET_IDS}

GENERATION_CONFIG = {
    "framework": "SPP-GAN",
    "notebook": "Notebook 13",
    "datasets": DATASET_IDS,
    "master_seed": MASTER_SEED,
    "generation_repetitions": GENERATION_REPETITIONS,
    "sample_size_override": SAMPLE_SIZE_OVERRIDE,
    "batch_size": GENERATION_BATCH_SIZE,
    "save_parquet": SAVE_PARQUET,
    "target_columns": TARGET_COLUMNS,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

GENERATION_CONFIG_PATH = NB13_CONFIG_ROOT / "sppgan_generation_configuration.json"
with open(GENERATION_CONFIG_PATH, "w", encoding="utf-8") as handle:
    json.dump(GENERATION_CONFIG, handle, indent=2, ensure_ascii=False)

print(json.dumps(GENERATION_CONFIG, indent=2))

{
  "framework": "SPP-GAN",
  "notebook": "Notebook 13",
  "datasets": [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
  ],
  "master_seed": 2025,
  "generation_repetitions": 1,
  "sample_size_override": null,
  "batch_size": 1024,
  "save_parquet": false,
  "target_columns": {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
  },
  "created_utc": "2026-10-14T07:24:05.768549+00:00"
}


In [42]:
# =============================================================================
# SECTION 7 — SET GENERATION SEEDS
# =============================================================================

def generation_seed(dataset, repetition=0):
    return MASTER_SEED + DATASET_IDS.index(dataset) * 10000 + repetition

def set_generation_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

SEED_PLAN = pd.DataFrame([
    {
        "dataset": dataset,
        "repetition": repetition,
        "seed": generation_seed(dataset, repetition),
    }
    for dataset in DATASET_IDS
    for repetition in range(GENERATION_REPETITIONS)
])

SEED_PLAN_PATH = NB13_METADATA_ROOT / "generation_seed_plan.csv"
SEED_PLAN.to_csv(SEED_PLAN_PATH, index=False)
display(SEED_PLAN)

,dataset,repetition,seed
0,adult_income,0,2025
1,bank_marketing,0,12025
2,diabetes_130us,0,22025


In [50]:

# ==================================================================================================
# SECTION 8 — GENERATE SYNTHETIC SAMPLES
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — GENERATE SYNTHETIC SAMPLES")
print("=" * 100)

import time
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Generation device: {DEVICE}")

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")
NB08_ARCH_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_08" / "models"
NB13_LOG_ROOT = Path(NB13_LOG_ROOT)
NB13_LOG_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------
# 1. VALIDATE CONFIGURATION
# --------------------------------------------------------------------------------

for name in [
    "DATASET_IDS", "MODEL_ARTIFACTS", "REFERENCE_TRAIN",
    "GENERATION_REPETITIONS", "GENERATION_BATCH_SIZE",
    "SAMPLE_SIZE_OVERRIDE", "MASTER_SEED",
]:
    if name not in globals():
        raise RuntimeError(
            f"{name} is missing. Run the preceding Notebook 13 sections first."
        )

if int(GENERATION_REPETITIONS) < 1:
    raise ValueError("GENERATION_REPETITIONS must be at least 1.")

if int(GENERATION_BATCH_SIZE) < 1:
    raise ValueError("GENERATION_BATCH_SIZE must be at least 1.")

# The canonical output contracts established by Notebook 08 / Notebook 12.
# Cardinalities are ordered by the original categorical feature order.
ARCHITECTURE_FALLBACKS = {
    "adult_income": {
        "latent_dim": 128,
        "hidden_dims": [256, 256],
        "num_numerical": 6,
        "categorical_cardinalities": [8, 16, 7, 14, 6, 5, 2, 41],
        "transformed_dim": 105,
    },
    "bank_marketing": {
        "latent_dim": 128,
        "hidden_dims": [256, 256],
        "num_numerical": 7,
        "categorical_cardinalities": [12, 3, 4, 2, 2, 2, 3, 12, 4],
        "transformed_dim": 51,
    },
    "diabetes_130us": {
        "latent_dim": 128,
        "hidden_dims": [256, 256],
        "num_numerical": 11,
        "categorical_cardinalities": [
            5, 3, 10, 9, 16, 69, 690, 704, 734,
            3, 3, 4, 4, 4, 4, 4, 1, 4, 4, 2,
            4, 4, 4, 4, 2, 2, 1, 1, 4, 4,
            2, 1, 2, 2, 2, 2,
        ],
        "transformed_dim": 2329,
    },
}

def normalize_key(value):
    return str(value).strip().lower()

def resolve_dataset_entry(container, dataset, container_name):
    if not isinstance(container, dict):
        raise TypeError(
            f"{container_name} must be a dictionary; "
            f"received {type(container).__name__}."
        )

    for key, value in container.items():
        if normalize_key(key) == normalize_key(dataset):
            return value

    raise KeyError(
        f"{dataset!r} not found in {container_name}. "
        f"Available keys: {list(container.keys())}"
    )

def resolve_model_path(dataset, repetition_id):
    dataset_artifacts = resolve_dataset_entry(
        MODEL_ARTIFACTS, dataset, "MODEL_ARTIFACTS"
    )

    if isinstance(dataset_artifacts, (str, Path)):
        if int(GENERATION_REPETITIONS) != 1:
            raise TypeError(
                f"{dataset}: one checkpoint path was provided, but multiple "
                "repetitions were requested."
            )
        return Path(dataset_artifacts)

    if not isinstance(dataset_artifacts, dict):
        raise TypeError(
            f"Unsupported model registry entry for {dataset}: "
            f"{type(dataset_artifacts).__name__}"
        )

    for key, value in dataset_artifacts.items():
        if str(key).strip() == str(repetition_id):
            return Path(value)

    raise KeyError(
        f"Repetition {repetition_id} not found for {dataset}. "
        f"Available: {list(dataset_artifacts.keys())}"
    )

def resolve_seed(dataset, repetition_id):
    plan = globals().get("SEED_PLAN")

    if isinstance(plan, dict):
        for key, value in plan.items():
            if normalize_key(key) != normalize_key(dataset):
                continue

            if isinstance(value, dict):
                for rep_key in (
                    repetition_id, str(repetition_id),
                    repetition_id - 1, str(repetition_id - 1),
                ):
                    if rep_key in value:
                        seed_value = value[rep_key]
                        if isinstance(seed_value, dict):
                            seed_value = seed_value.get("seed")
                        if seed_value is not None:
                            return int(seed_value)

            elif isinstance(value, (int, np.integer)):
                return int(value)

    dataset_index = next(
        i for i, item in enumerate(DATASET_IDS)
        if normalize_key(item) == normalize_key(dataset)
    )

    return (
        int(MASTER_SEED)
        + dataset_index * 10000
        + repetition_id - 1
    )

# --------------------------------------------------------------------------------
# 2. LOAD AND CHECK CANONICAL ARCHITECTURE CONTRACT
# --------------------------------------------------------------------------------

def load_architecture(dataset):
    architecture_path = (
        NB08_ARCH_ROOT / dataset / "sppgan_architecture.json"
    )

    if architecture_path.is_file():
        with architecture_path.open("r", encoding="utf-8") as file:
            architecture = json.load(file)

        # Use canonical metadata only when its expected fields can be resolved.
        # The validated fallback below matches the persisted contract dimensions.
        def search_nested(obj, wanted):
            if isinstance(obj, dict):
                for key, value in obj.items():
                    if key in wanted and value is not None:
                        return value
                for value in obj.values():
                    found = search_nested(value, wanted)
                    if found is not None:
                        return found
            elif isinstance(obj, list):
                for value in obj:
                    found = search_nested(value, wanted)
                    if found is not None:
                        return found
            return None

        latent = search_nested(architecture, {"latent_dim"})
        hidden = search_nested(
            architecture, {"generator_hidden_dims", "hidden_dims"}
        )
        input_rep = search_nested(
            architecture, {"input_representation"}
        )

        if isinstance(input_rep, dict):
            cards = input_rep.get("categorical_cardinalities")
            num_numeric = input_rep.get("num_numerical")
            logical_dim = input_rep.get("logical_feature_dim")
        else:
            cards = None
            num_numeric = None
            logical_dim = None

        contract = dict(ARCHITECTURE_FALLBACKS[dataset])

        if latent is not None:
            contract["latent_dim"] = int(latent)

        if isinstance(hidden, list) and hidden:
            contract["hidden_dims"] = [int(x) for x in hidden]

        if isinstance(cards, list) and cards:
            contract["categorical_cardinalities"] = [
                int(x) for x in cards
            ]

        if num_numeric is not None:
            contract["num_numerical"] = int(num_numeric)

        if logical_dim is not None:
            contract["transformed_dim"] = int(logical_dim)

        contract["architecture_path"] = str(architecture_path)

    else:
        contract = dict(ARCHITECTURE_FALLBACKS[dataset])
        contract["architecture_path"] = None

    calculated_dim = (
        contract["num_numerical"]
        + sum(contract["categorical_cardinalities"])
    )

    if calculated_dim != contract["transformed_dim"]:
        raise ValueError(
            f"{dataset}: architecture contract is inconsistent. "
            f"Numerical + categorical width={calculated_dim}; "
            f"declared transformed dimension={contract['transformed_dim']}."
        )

    return contract

# --------------------------------------------------------------------------------
# 3. DEFINE CHECKPOINT-COMPATIBLE GENERATOR
# --------------------------------------------------------------------------------
# The parameter names must match the saved generator_state_dict.
# Numerical/categorical decoding is applied after the forward pass.

class SPPGANGenerator(nn.Module):
    def __init__(self, latent_dim, hidden_dims, transformed_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(latent_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], transformed_dim),
        )

    def forward(self, z):
        return self.network(z)

# --------------------------------------------------------------------------------
# 4. LOAD TRUSTED CHECKPOINT
# --------------------------------------------------------------------------------

def load_checkpoint(checkpoint_path):
    # This fallback is appropriate only for checkpoint files created by your
    # own trusted Notebook 12 workflow. weights_only=False permits pickle
    # deserialization and must not be used on untrusted files.
    try:
        return torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True,
        )
    except Exception as safe_error:
        print(
            "Safe checkpoint load failed; using trusted-project fallback: "
            f"{type(safe_error).__name__}: {safe_error}"
        )
        return torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )

# --------------------------------------------------------------------------------
# 5. CONVERT RAW GENERATOR OUTPUT TO TRANSFORMED REPRESENTATION
# --------------------------------------------------------------------------------

def decode_generator_output(raw_output, contract, seed):
    """
    Preserve numerical outputs using identity activation.
    Convert each categorical logit block to a one-hot category using softmax
    probabilities and deterministic categorical sampling.
    """
    n_numeric = contract["num_numerical"]
    cardinalities = contract["categorical_cardinalities"]

    numerical = raw_output[:, :n_numeric]
    categorical_logits = raw_output[:, n_numeric:]

    blocks = [numerical]
    offset = 0

    rng = np.random.default_rng(int(seed))

    for cardinality in cardinalities:
        block_logits = categorical_logits[
            :, offset:offset + cardinality
        ]
        offset += cardinality

        if block_logits.shape[1] != cardinality:
            raise ValueError(
                f"Categorical block width mismatch: expected {cardinality}, "
                f"received {block_logits.shape[1]}."
            )

        # Stable softmax probability mapping.
        shifted = block_logits - np.max(
            block_logits, axis=1, keepdims=True
        )
        exp_logits = np.exp(shifted)
        probabilities = exp_logits / np.maximum(
            exp_logits.sum(axis=1, keepdims=True),
            np.finfo(np.float64).tiny,
        )

        # Sample one category per categorical feature.
        sampled_categories = np.array([
            rng.choice(cardinality, p=row / row.sum())
            for row in probabilities
        ])

        one_hot = np.zeros(
            (len(raw_output), cardinality),
            dtype=np.float32,
        )
        one_hot[np.arange(len(raw_output)), sampled_categories] = 1.0
        blocks.append(one_hot)

    transformed = np.concatenate(blocks, axis=1).astype(
        np.float32, copy=False
    )

    if transformed.shape[1] != contract["transformed_dim"]:
        raise ValueError(
            f"Decoded width {transformed.shape[1]} does not match "
            f"contract width {contract['transformed_dim']}."
        )

    if not np.isfinite(transformed).all():
        raise ValueError("Transformed output contains NaN or infinity.")

    return transformed

# --------------------------------------------------------------------------------
# 6. GENERATE SAMPLES
# --------------------------------------------------------------------------------

GENERATED_ARRAYS = {}
GENERATION_RUNTIME = {}
GENERATION_ERRORS = []
GENERATION_METADATA = []

for dataset in DATASET_IDS:
    print("-" * 100)
    print(f"GENERATING: {dataset}")
    print("-" * 100)

    GENERATED_ARRAYS[dataset] = {}

    try:
        reference_df = resolve_dataset_entry(
            REFERENCE_TRAIN, dataset, "REFERENCE_TRAIN"
        )

        if not isinstance(reference_df, pd.DataFrame):
            raise TypeError(
                f"Reference data for {dataset} is not a DataFrame."
            )

        if reference_df.empty:
            raise ValueError(f"Reference training data is empty for {dataset}.")

        contract = load_architecture(dataset)

        sample_count = (
            int(SAMPLE_SIZE_OVERRIDE)
            if SAMPLE_SIZE_OVERRIDE is not None
            else len(reference_df)
        )

        if sample_count <= 0:
            raise ValueError("Sample count must be positive.")

        print(
            f"Native reference columns : {len(reference_df.columns)}\n"
            f"Transformed model width  : {contract['transformed_dim']}\n"
            f"Numerical features       : {contract['num_numerical']}\n"
            f"Categorical features     : "
            f"{len(contract['categorical_cardinalities'])}\n"
            f"Requested samples        : {sample_count}"
        )

        dataset_artifacts = resolve_dataset_entry(
            MODEL_ARTIFACTS, dataset, "MODEL_ARTIFACTS"
        )

        if isinstance(dataset_artifacts, dict):
            repetitions = sorted(
                dataset_artifacts.keys(),
                key=lambda key: int(key),
            )[:int(GENERATION_REPETITIONS)]
        else:
            repetitions = [1]

        if not repetitions:
            raise FileNotFoundError(
                f"No checkpoint repetitions found for {dataset}."
            )

        for repetition_key in repetitions:
            repetition_id = int(repetition_key)
            checkpoint = None
            generator = None

            try:
                model_path = resolve_model_path(
                    dataset, repetition_id
                )

                if not model_path.is_file():
                    raise FileNotFoundError(
                        f"Checkpoint not found: {model_path}"
                    )

                seed = resolve_seed(dataset, repetition_id)

                digest = hashlib.sha256()
                with model_path.open("rb") as handle:
                    for chunk in iter(
                        lambda: handle.read(1024 * 1024), b""
                    ):
                        digest.update(chunk)

                checkpoint_sha256 = digest.hexdigest()
                checkpoint = load_checkpoint(model_path)

                if not isinstance(checkpoint, dict):
                    raise TypeError("Checkpoint must be a dictionary.")

                state_dict = checkpoint.get("generator_state_dict")
                if not isinstance(state_dict, dict):
                    raise KeyError(
                        "Checkpoint lacks generator_state_dict."
                    )

                expected_keys = {
                    "network.0.weight",
                    "network.0.bias",
                    "network.2.weight",
                    "network.2.bias",
                    "network.4.weight",
                    "network.4.bias",
                }

                if set(state_dict.keys()) != expected_keys:
                    raise ValueError(
                        "Checkpoint parameter names differ from the "
                        "expected architecture. Check Notebook 08/12."
                    )

                latent_dim = int(
                    state_dict["network.0.weight"].shape[1]
                )
                hidden_1 = int(
                    state_dict["network.0.weight"].shape[0]
                )
                hidden_2 = int(
                    state_dict["network.2.weight"].shape[0]
                )
                checkpoint_output_dim = int(
                    state_dict["network.4.weight"].shape[0]
                )

                if [hidden_1, hidden_2] != contract["hidden_dims"]:
                    raise ValueError(
                        f"Hidden layer mismatch: checkpoint={[hidden_1, hidden_2]}, "
                        f"contract={contract['hidden_dims']}."
                    )

                if latent_dim != contract["latent_dim"]:
                    raise ValueError(
                        f"Latent dimension mismatch: checkpoint={latent_dim}, "
                        f"contract={contract['latent_dim']}."
                    )

                if checkpoint_output_dim != contract["transformed_dim"]:
                    raise ValueError(
                        f"Transformed output mismatch: checkpoint="
                        f"{checkpoint_output_dim}, "
                        f"architecture={contract['transformed_dim']}."
                    )

                generator = SPPGANGenerator(
                    latent_dim=latent_dim,
                    hidden_dims=contract["hidden_dims"],
                    transformed_dim=contract["transformed_dim"],
                ).to(DEVICE)

                generator.load_state_dict(state_dict, strict=True)
                generator.eval()

                latent_rng = torch.Generator(device="cpu")
                latent_rng.manual_seed(int(seed))

                start_time = time.perf_counter()
                raw_batches = []

                with torch.inference_mode():
                    for start in range(
                        0, sample_count, int(GENERATION_BATCH_SIZE)
                    ):
                        batch_size = min(
                            int(GENERATION_BATCH_SIZE),
                            sample_count - start,
                        )

                        z = torch.randn(
                            batch_size,
                            latent_dim,
                            generator=latent_rng,
                            device="cpu",
                        ).to(DEVICE)

                        raw_batches.append(
                            generator(z).detach().cpu().numpy()
                        )

                raw_output = np.concatenate(raw_batches, axis=0)

                if raw_output.shape != (
                    sample_count, contract["transformed_dim"]
                ):
                    raise ValueError(
                        f"Raw generator shape {raw_output.shape} is invalid."
                    )

                transformed_output = decode_generator_output(
                    raw_output, contract, seed
                )

                elapsed = time.perf_counter() - start_time

                GENERATED_ARRAYS[dataset][repetition_id] = (
                    transformed_output
                )
                GENERATION_RUNTIME[(dataset, repetition_id)] = elapsed

                GENERATION_METADATA.append({
                    "dataset": dataset,
                    "repetition_id": repetition_id,
                    "status": "PASS",
                    "sample_count": sample_count,
                    "native_reference_columns": len(reference_df.columns),
                    "transformed_dim": contract["transformed_dim"],
                    "num_numerical": contract["num_numerical"],
                    "num_categorical": len(
                        contract["categorical_cardinalities"]
                    ),
                    "latent_dim": latent_dim,
                    "seed": int(seed),
                    "device": str(DEVICE),
                    "runtime_seconds": elapsed,
                    "checkpoint_path": str(model_path),
                    "checkpoint_sha256": checkpoint_sha256,
                })

                print(
                    f"PASS | repetition={repetition_id} | "
                    f"transformed shape={transformed_output.shape} | "
                    f"runtime={elapsed:.2f}s"
                )

                del raw_output, transformed_output, raw_batches

            except Exception as exc:
                GENERATION_ERRORS.append({
                    "dataset": dataset,
                    "repetition_id": repetition_id,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                })

                print(
                    f"GENERATION FAILED | repetition={repetition_id} | "
                    f"{type(exc).__name__}: {exc}"
                )

            finally:
                if generator is not None:
                    del generator
                if checkpoint is not None:
                    del checkpoint
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    except Exception as exc:
        GENERATION_ERRORS.append({
            "dataset": dataset,
            "repetition_id": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

        print(f"DATASET FAILED | {type(exc).__name__}: {exc}")

# --------------------------------------------------------------------------------
# 7. SAVE LOGS
# --------------------------------------------------------------------------------

error_log_path = NB13_LOG_ROOT / "generation_errors.csv"
metadata_log_path = NB13_LOG_ROOT / "generation_metadata.csv"

pd.DataFrame(
    GENERATION_ERRORS,
    columns=[
        "dataset", "repetition_id", "error_type", "error_message"
    ],
).to_csv(error_log_path, index=False)

pd.DataFrame(GENERATION_METADATA).to_csv(
    metadata_log_path, index=False
)

# --------------------------------------------------------------------------------
# 8. SUMMARY
# --------------------------------------------------------------------------------

successful_outputs = sum(
    len(outputs) for outputs in GENERATED_ARRAYS.values()
)

print("\n" + "=" * 100)
print("SECTION 8 GENERATION SUMMARY")
print("=" * 100)
print(f"Datasets requested: {len(DATASET_IDS)}")
print(f"Successful dataset/repetition outputs: {successful_outputs}")
print(f"Generation errors: {len(GENERATION_ERRORS)}")
print(f"Error log: {error_log_path}")
print(f"Metadata log: {metadata_log_path}")

for dataset in DATASET_IDS:
    for repetition_id, array in GENERATED_ARRAYS.get(dataset, {}).items():
        print(
            f"{dataset} | repetition={repetition_id} | "
            f"shape={array.shape}"
        )

if successful_outputs == 0:
    print("WARNING: No synthetic data generated. Review generation_errors.csv.")
else:
    print(
        "Generation completed. Continue with inverse transformation and "
        "validate decoded feature semantics before evaluation."
    )


SECTION 8 — GENERATE SYNTHETIC SAMPLES
Generation device: cuda
----------------------------------------------------------------------------------------------------
GENERATING: adult_income
----------------------------------------------------------------------------------------------------
Native reference columns : 16
Transformed model width  : 105
Numerical features       : 6
Categorical features     : 8
Requested samples        : 34189
Safe checkpoint load failed; using trusted-project fallback: UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to loa

In [56]:

# =============================================================================
# SECTION 9 — INVERSE TRANSFORM
# =============================================================================

import json
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 9 — INVERSE TRANSFORM")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Validate Required Artifacts
# -----------------------------------------------------------------------------

REQUIRED_DATASETS = ("adult_income", "bank_marketing", "diabetes_130us")

for name in (
    "GENERATED_ARRAYS",
    "PREPROCESSORS",
    "SCHEMAS",
    "REFERENCE_TRAIN",
    "PROJECT_ROOT",
    "NB02_ROOT",
    "NB13_LOG_ROOT",
):
    if name not in globals():
        raise RuntimeError(f"Required artifact is unavailable: {name}")

if not isinstance(GENERATED_ARRAYS, dict) or not GENERATED_ARRAYS:
    raise RuntimeError("GENERATED_ARRAYS is missing or empty. Run Section 8 first.")

# -----------------------------------------------------------------------------
# 2. Resolve the Generated Array for Each Repetition
# -----------------------------------------------------------------------------

def extract_generated_array(dataset, stored_value):
    """Extract the configured repetition from Section 8 output."""
    value = stored_value

    if isinstance(value, dict):
        repetition_id = 1

        if repetition_id in value:
            value = value[repetition_id]
        elif str(repetition_id) in value:
            value = value[str(repetition_id)]
        elif len(value) == 1:
            repetition_id, value = next(iter(value.items()))
            print(f"{dataset}: using stored repetition {repetition_id!r}.")
        else:
            raise KeyError(
                f"{dataset}: cannot resolve repetition {repetition_id}; "
                f"available keys={list(value.keys())}."
            )

    if isinstance(value, pd.DataFrame):
        array = value.to_numpy()
    elif hasattr(value, "detach") and hasattr(value, "cpu"):
        array = value.detach().cpu().numpy()
    else:
        array = np.asarray(value)

    if array.ndim != 2 or array.shape[0] == 0:
        raise ValueError(
            f"{dataset}: expected a nonempty 2-D array, received {array.shape}."
        )

    array = array.astype(np.float32, copy=False)

    if not np.isfinite(array).all():
        raise ValueError(f"{dataset}: generated array contains NaN or infinity.")

    return array


# -----------------------------------------------------------------------------
# 3. Read Canonical Schema and Validate the Generator Width
# -----------------------------------------------------------------------------

def get_dataset_contract(dataset):
    schema = SCHEMAS.get(dataset)

    if not isinstance(schema, dict):
        raise TypeError(f"{dataset}: Notebook 02 schema is unavailable.")

    modeling = schema.get("modeling_schema", {})
    transformation = schema.get("transformation_schema", {})

    feature_columns = modeling.get("preprocessing_columns")
    numeric_columns = modeling.get("numeric_columns")
    categorical_columns = modeling.get("categorical_columns")
    transformed_names = transformation.get("transformed_feature_names")
    target_column = modeling.get("target_column")

    if not all(
        isinstance(items, list)
        for items in (
            feature_columns,
            numeric_columns,
            categorical_columns,
            transformed_names,
        )
    ):
        raise ValueError(f"{dataset}: required Notebook 02 schema lists are missing.")

    if len(feature_columns) != len(set(feature_columns)):
        raise ValueError(f"{dataset}: duplicate source feature names in schema.")

    if set(numeric_columns).intersection(categorical_columns):
        raise ValueError(f"{dataset}: numeric and categorical features overlap.")

    if set(numeric_columns).union(categorical_columns) != set(feature_columns):
        raise ValueError(
            f"{dataset}: numeric/categorical feature definitions do not cover "
            "the preprocessing columns exactly."
        )

    preprocessor = PREPROCESSORS.get(dataset)

    if preprocessor is None:
        raise RuntimeError(f"{dataset}: fitted preprocessor is unavailable.")

    fitted_inputs = list(preprocessor.feature_names_in_)

    if fitted_inputs != feature_columns:
        raise ValueError(
            f"{dataset}: fitted preprocessor input columns disagree with "
            "Notebook 02 schema."
        )

    fitted_names = list(preprocessor.get_feature_names_out())

    if fitted_names != transformed_names:
        raise ValueError(
            f"{dataset}: fitted transformed feature order disagrees with "
            "Notebook 02 transformation_schema."
        )

    expected_width = int(transformation.get("transformed_feature_count", -1))

    if expected_width != len(fitted_names):
        raise ValueError(
            f"{dataset}: transformed feature count disagrees with schema."
        )

    return {
        "schema": schema,
        "feature_columns": feature_columns,
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
        "transformed_names": transformed_names,
        "target_column": target_column,
        "preprocessor": preprocessor,
        "expected_width": expected_width,
    }


# -----------------------------------------------------------------------------
# 4. Resolve Fitted Numeric and Categorical Transformers
# -----------------------------------------------------------------------------

def get_fitted_components(dataset, contract):
    preprocessor = contract["preprocessor"]
    transformers = {
        name: (transformer, columns)
        for name, transformer, columns in preprocessor.transformers_
    }

    numeric_entry = None
    categorical_entry = None

    for name, (transformer, columns) in transformers.items():
        column_list = list(columns) if not isinstance(columns, str) else [columns]

        if column_list == contract["numeric_columns"]:
            numeric_entry = transformer

        if column_list == contract["categorical_columns"]:
            categorical_entry = transformer

    if numeric_entry is None:
        raise RuntimeError(
            f"{dataset}: could not identify the fitted numeric transformer "
            "from its source columns."
        )

    if categorical_entry is None:
        raise RuntimeError(
            f"{dataset}: could not identify the fitted categorical transformer "
            "from its source columns."
        )

    numeric_steps = getattr(numeric_entry, "named_steps", {})
    categorical_steps = getattr(categorical_entry, "named_steps", {})

    scaler = numeric_steps.get("scaler")
    encoder = categorical_steps.get("encoder")

    if scaler is None or not hasattr(scaler, "inverse_transform"):
        raise RuntimeError(
            f"{dataset}: fitted numeric scaler with inverse_transform() "
            "was not found."
        )

    if encoder is None or not hasattr(encoder, "categories_"):
        raise RuntimeError(
            f"{dataset}: fitted OneHotEncoder categories were not found."
        )

    if len(encoder.categories_) != len(contract["categorical_columns"]):
        raise ValueError(
            f"{dataset}: OneHotEncoder category count does not match "
            "categorical source feature count."
        )

    return scaler, encoder


# -----------------------------------------------------------------------------
# 5. Decode Transformed Output to Native Feature Values
# -----------------------------------------------------------------------------

def inverse_transform_dataset(dataset, stored_value):
    array = extract_generated_array(dataset, stored_value)
    contract = get_dataset_contract(dataset)

    if array.shape[1] != contract["expected_width"]:
        raise ValueError(
            f"{dataset}: generated width {array.shape[1]} does not match "
            f"Notebook 02 width {contract['expected_width']}."
        )

    scaler, encoder = get_fitted_components(dataset, contract)

    numeric_columns = contract["numeric_columns"]
    categorical_columns = contract["categorical_columns"]

    numeric_count = len(numeric_columns)
    numeric_width = numeric_count

    # Notebook 02's verified feature mapping places numeric transformed
    # features first, followed by the categorical one-hot feature blocks.
    numeric_indices = list(range(numeric_width))
    categorical_start = numeric_width

    categorical_positions = {}
    cursor = categorical_start

    for feature, categories in zip(
        categorical_columns,
        encoder.categories_,
    ):
        block_width = len(categories)
        categorical_positions[feature] = (
            cursor,
            cursor + block_width,
        )
        cursor += block_width

    if cursor != array.shape[1]:
        raise ValueError(
            f"{dataset}: numeric/categorical block widths total {cursor}, "
            f"but generator output width is {array.shape[1]}."
        )

    # Invert StandardScaler using the exact fitted Notebook 02 scaler.
    numeric_scaled = array[:, numeric_indices]
    numeric_restored = scaler.inverse_transform(numeric_scaled)

    restored = pd.DataFrame(index=np.arange(len(array)))

    for index, feature in enumerate(numeric_columns):
        restored[feature] = numeric_restored[:, index]

    # Decode each categorical block against the actual fitted encoder categories.
    # argmax is appropriate for the one-hot blocks produced by Section 8.
    for feature in categorical_columns:
        start, end = categorical_positions[feature]
        block = array[:, start:end]
        categories = encoder.categories_[
            categorical_columns.index(feature)
        ]

        if block.shape[1] != len(categories):
            raise ValueError(
                f"{dataset}/{feature}: block width does not match fitted categories."
            )

        category_indices = np.argmax(block, axis=1)
        restored[feature] = np.asarray(categories, dtype=object)[category_indices]

    # Re-establish the source feature order from Notebook 02.
    restored = restored[contract["feature_columns"]].reset_index(drop=True)

    if list(restored.columns) != contract["feature_columns"]:
        raise ValueError(
            f"{dataset}: restored feature order does not match Notebook 02."
        )

    if len(restored) != len(array):
        raise ValueError(
            f"{dataset}: row count changed during inverse transformation."
        )

    # Target is intentionally excluded here; Section 13 restores it.
    target_column = contract["target_column"]
    if target_column and target_column in restored.columns:
        raise ValueError(
            f"{dataset}: target appeared in the feature-only inverse transform."
        )

    metadata = {
        "dataset": dataset,
        "status": "PASS",
        "transformed_rows": int(array.shape[0]),
        "transformed_width": int(array.shape[1]),
        "native_feature_rows": int(restored.shape[0]),
        "native_feature_columns": int(restored.shape[1]),
        "target_deferred_to_section_13": target_column,
        "error": None,
    }

    return restored, metadata


# -----------------------------------------------------------------------------
# 6. Process All Datasets
# -----------------------------------------------------------------------------

SYNTHETIC_DATAFRAMES = {}
INVERSE_TRANSFORM_ERRORS = []
INVERSE_TRANSFORM_METADATA = []

for dataset in REQUIRED_DATASETS:
    print("-" * 100)
    print("DATASET:", dataset)

    if dataset not in GENERATED_ARRAYS:
        message = "Generated output is missing from GENERATED_ARRAYS."
        INVERSE_TRANSFORM_ERRORS.append({
            "dataset": dataset,
            "error": message,
        })
        INVERSE_TRANSFORM_METADATA.append({
            "dataset": dataset,
            "status": "FAIL",
            "error": message,
        })
        print("FAIL:", message)
        continue

    try:
        dataframe, metadata = inverse_transform_dataset(
            dataset,
            GENERATED_ARRAYS[dataset],
        )

        SYNTHETIC_DATAFRAMES[dataset] = dataframe
        INVERSE_TRANSFORM_METADATA.append(metadata)

        print("PASS | Feature-only native shape:", dataframe.shape)
        print("PASS | Column order matches Notebook 02.")
        print("Target restoration is deferred to Section 13.")

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"

        INVERSE_TRANSFORM_ERRORS.append({
            "dataset": dataset,
            "error": message,
        })

        INVERSE_TRANSFORM_METADATA.append({
            "dataset": dataset,
            "status": "FAIL",
            "error": message,
        })

        print("FAIL:", message)


# -----------------------------------------------------------------------------
# 7. Save Section 9 Logs
# -----------------------------------------------------------------------------

pd.DataFrame(
    INVERSE_TRANSFORM_ERRORS,
    columns=["dataset", "error"],
).to_csv(
    NB13_LOG_ROOT / "inverse_transform_errors.csv",
    index=False,
)

pd.DataFrame(INVERSE_TRANSFORM_METADATA).to_csv(
    NB13_LOG_ROOT / "inverse_transform_metadata.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# 8. Completion Summary
# -----------------------------------------------------------------------------

print("=" * 100)
print("SECTION 9 SUMMARY")
print("=" * 100)
print("Expected datasets:", len(REQUIRED_DATASETS))
print("Successful inverse transforms:", len(SYNTHETIC_DATAFRAMES))
print("Failures:", len(INVERSE_TRANSFORM_ERRORS))
print("Error log:", NB13_LOG_ROOT / "inverse_transform_errors.csv")
print("Metadata log:", NB13_LOG_ROOT / "inverse_transform_metadata.csv")

if INVERSE_TRANSFORM_ERRORS:
    print(
        "\nWARNING: Resolve Section 9 failures before running Sections 10–26."
    )
else:
    print(
        "\nPASS: All transformed feature arrays were decoded using the fitted "
        "Notebook 02 scaler and OneHotEncoder categories."
    )
    print(
        "NOTE: This confirms decoding and schema alignment, not the semantic "
        "validity of generated records or end-to-end differential privacy."
    )


SECTION 9 — INVERSE TRANSFORM
----------------------------------------------------------------------------------------------------
DATASET: adult_income
PASS | Feature-only native shape: (34189, 14)
PASS | Column order matches Notebook 02.
Target restoration is deferred to Section 13.
----------------------------------------------------------------------------------------------------
DATASET: bank_marketing
PASS | Feature-only native shape: (31647, 16)
PASS | Column order matches Notebook 02.
Target restoration is deferred to Section 13.
----------------------------------------------------------------------------------------------------
DATASET: diabetes_130us
PASS | Feature-only native shape: (71236, 47)
PASS | Column order matches Notebook 02.
Target restoration is deferred to Section 13.
SECTION 9 SUMMARY
Expected datasets: 3
Successful inverse transforms: 3
Failures: 0
Error log: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/logs/inverse_transform_errors.csv

In [58]:
# =============================================================================
# SECTION 10 — RESTORE FEATURE TYPES
# =============================================================================

print("=" * 100)
print("SECTION 10 — RESTORE FEATURE TYPES")
print("=" * 100)

import pandas as pd
import numpy as np

def restore_feature_types(dataset, dataframe):
    result = dataframe.copy()
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        raise ValueError(
            f"{dataset}: Notebook 02 reference training data is unavailable."
        )

    # Validate feature alignment; the target is restored in Section 13.
    expected_columns = [
        column for column in reference.columns
        if column in result.columns
    ]

    if list(result.columns) != expected_columns:
        raise ValueError(
            f"{dataset}: feature columns do not match the expected order. "
            f"Expected {expected_columns}, got {list(result.columns)}."
        )

    for column in result.columns:
        source_dtype = reference[column].dtype

        try:
            if pd.api.types.is_bool_dtype(source_dtype):
                values = pd.to_numeric(result[column], errors="coerce")
                result[column] = values.round().clip(0, 1).astype("Int64")

            elif pd.api.types.is_integer_dtype(source_dtype):
                values = pd.to_numeric(result[column], errors="coerce")
                result[column] = values.round().astype("Int64")

            elif pd.api.types.is_float_dtype(source_dtype):
                result[column] = pd.to_numeric(
                    result[column], errors="coerce"
                ).astype("float64")

            elif isinstance(source_dtype, pd.CategoricalDtype):
                result[column] = result[column].astype("object")

            elif (
                pd.api.types.is_object_dtype(source_dtype)
                or pd.api.types.is_string_dtype(source_dtype)
            ):
                result[column] = result[column].astype("object")

        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError(
                f"{dataset}/{column}: feature type restoration failed: {exc}"
            ) from exc

    # Confirm restoration did not alter row count or column order.
    if len(result) != len(dataframe):
        raise RuntimeError(f"{dataset}: row count changed during type restoration.")

    if list(result.columns) != list(dataframe.columns):
        raise RuntimeError(f"{dataset}: column order changed during type restoration.")

    return result


TYPE_RESTORATION_ERRORS = []

for dataset in list(SYNTHETIC_DATAFRAMES):
    try:
        before = SYNTHETIC_DATAFRAMES[dataset]

        SYNTHETIC_DATAFRAMES[dataset] = restore_feature_types(
            dataset, before
        )

        after = SYNTHETIC_DATAFRAMES[dataset]

        print(
            f"PASS | {dataset} | "
            f"Rows: {len(after):,} | "
            f"Features: {len(after.columns)}"
        )

    except Exception as exc:
        TYPE_RESTORATION_ERRORS.append({
            "dataset": dataset,
            "error": str(exc)
        })
        print(f"FAIL | {dataset}: {exc}")

print("-" * 100)
print(f"Expected datasets: {len(REFERENCE_TRAIN)}")
print(f"Successful restorations: {len(SYNTHETIC_DATAFRAMES) - len(TYPE_RESTORATION_ERRORS)}")
print(f"Failures: {len(TYPE_RESTORATION_ERRORS)}")

if TYPE_RESTORATION_ERRORS:
    error_path = NB13_LOG_ROOT / "feature_type_restoration_errors.csv"
    pd.DataFrame(TYPE_RESTORATION_ERRORS).to_csv(error_path, index=False)
    print(f"Error log: {error_path}")
    raise RuntimeError(
        "Section 10 failed. Resolve feature type restoration errors before proceeding."
    )

print("PASS: Feature types restored without changing row counts or column order.")
print("NOTE: Semantic validity and target restoration are assessed in later sections.")

SECTION 10 — RESTORE FEATURE TYPES
PASS | adult_income | Rows: 34,189 | Features: 14
PASS | bank_marketing | Rows: 31,647 | Features: 16
PASS | diabetes_130us | Rows: 71,236 | Features: 47
----------------------------------------------------------------------------------------------------
Expected datasets: 3
Successful restorations: 3
Failures: 0
PASS: Feature types restored without changing row counts or column order.
NOTE: Semantic validity and target restoration are assessed in later sections.


In [59]:
# =============================================================================
# SECTION 11 — RESTORE CATEGORIES
# =============================================================================

print("=" * 100)
print("SECTION 11 — RESTORE CATEGORIES")
print("=" * 100)

import pandas as pd

CATEGORY_RESTORATION_ERRORS = []

def restore_categories(dataset, dataframe):
    result = dataframe.copy()
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        raise ValueError(
            f"{dataset}: Notebook 02 reference training data is unavailable."
        )

    if list(result.columns) != [
        column for column in reference.columns if column in result.columns
    ]:
        raise ValueError(
            f"{dataset}: feature column order does not match the reference data."
        )

    for column in result.columns:
        source_dtype = reference[column].dtype

        if isinstance(source_dtype, pd.CategoricalDtype):
            valid_categories = list(reference[column].cat.categories)
            result[column] = pd.Categorical(
                result[column],
                categories=valid_categories,
                ordered=source_dtype.ordered,
            )

        elif (
            pd.api.types.is_object_dtype(source_dtype)
            or pd.api.types.is_string_dtype(source_dtype)
        ):
            # Preserve decoded category labels without inventing new categories.
            result[column] = result[column].astype("object")

    return result


for dataset in list(SYNTHETIC_DATAFRAMES):
    try:
        before = SYNTHETIC_DATAFRAMES[dataset]

        SYNTHETIC_DATAFRAMES[dataset] = restore_categories(
            dataset, before
        )

        after = SYNTHETIC_DATAFRAMES[dataset]

        if len(after) != len(before):
            raise RuntimeError("Row count changed during category restoration.")

        if list(after.columns) != list(before.columns):
            raise RuntimeError("Column order changed during category restoration.")

        print(
            f"PASS | {dataset} | "
            f"Rows: {len(after):,} | "
            f"Features: {len(after.columns)}"
        )

    except Exception as exc:
        CATEGORY_RESTORATION_ERRORS.append({
            "dataset": dataset,
            "error": str(exc),
        })
        print(f"FAIL | {dataset}: {exc}")


print("-" * 100)
print(f"Datasets processed: {len(SYNTHETIC_DATAFRAMES)}")
print(
    "Successful restorations: "
    f"{len(SYNTHETIC_DATAFRAMES) - len(CATEGORY_RESTORATION_ERRORS)}"
)
print(f"Failures: {len(CATEGORY_RESTORATION_ERRORS)}")

if CATEGORY_RESTORATION_ERRORS:
    error_path = NB13_LOG_ROOT / "category_restoration_errors.csv"
    pd.DataFrame(CATEGORY_RESTORATION_ERRORS).to_csv(
        error_path, index=False
    )
    print(f"Error log: {error_path}")
    raise RuntimeError(
        "Section 11 failed. Resolve category restoration errors before proceeding."
    )

print("PASS: Categorical dtypes and reference category definitions restored where applicable.")
print("NOTE: This step does not force generated values into valid categories; category validity is checked separately.")

SECTION 11 — RESTORE CATEGORIES
PASS | adult_income | Rows: 34,189 | Features: 14
PASS | bank_marketing | Rows: 31,647 | Features: 16
PASS | diabetes_130us | Rows: 71,236 | Features: 47
----------------------------------------------------------------------------------------------------
Datasets processed: 3
Successful restorations: 3
Failures: 0
PASS: Categorical dtypes and reference category definitions restored where applicable.
NOTE: This step does not force generated values into valid categories; category validity is checked separately.


In [60]:
# =============================================================================
# SECTION 12 — RESTORE NUMERICAL VALUES
# =============================================================================

print("=" * 100)
print("SECTION 12 — RESTORE NUMERICAL VALUES")
print("=" * 100)

import pandas as pd
import numpy as np

NUMERICAL_RESTORATION_ERRORS = []

def restore_numerical_values(dataset, dataframe):
    result = dataframe.copy()
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        raise ValueError(
            f"{dataset}: Notebook 02 reference training data is unavailable."
        )

    if list(result.columns) != [
        column for column in reference.columns if column in result.columns
    ]:
        raise ValueError(
            f"{dataset}: feature column order does not match reference data."
        )

    for column in result.columns:
        source_dtype = reference[column].dtype

        if (
            pd.api.types.is_numeric_dtype(source_dtype)
            and not pd.api.types.is_bool_dtype(source_dtype)
        ):
            values = pd.to_numeric(result[column], errors="coerce")

            # Restore integer-valued features without silently clipping values.
            if pd.api.types.is_integer_dtype(source_dtype):
                finite_mask = values.notna() & np.isfinite(values)

                if finite_mask.any():
                    values.loc[finite_mask] = values.loc[finite_mask].round()

                # Nullable integer dtype preserves any generated NaN values.
                result[column] = values.astype("Int64")

            else:
                result[column] = values.astype("float64")

    return result


for dataset in list(SYNTHETIC_DATAFRAMES):
    try:
        before = SYNTHETIC_DATAFRAMES[dataset]

        SYNTHETIC_DATAFRAMES[dataset] = restore_numerical_values(
            dataset, before
        )

        after = SYNTHETIC_DATAFRAMES[dataset]

        if len(after) != len(before):
            raise RuntimeError("Row count changed during numerical restoration.")

        if list(after.columns) != list(before.columns):
            raise RuntimeError("Column order changed during numerical restoration.")

        print(
            f"PASS | {dataset} | "
            f"Rows: {len(after):,} | "
            f"Features: {len(after.columns)}"
        )

    except Exception as exc:
        NUMERICAL_RESTORATION_ERRORS.append({
            "dataset": dataset,
            "error": str(exc),
        })
        print(f"FAIL | {dataset}: {exc}")


print("-" * 100)
print(f"Datasets processed: {len(SYNTHETIC_DATAFRAMES)}")
print(
    "Successful restorations: "
    f"{len(SYNTHETIC_DATAFRAMES) - len(NUMERICAL_RESTORATION_ERRORS)}"
)
print(f"Failures: {len(NUMERICAL_RESTORATION_ERRORS)}")

if NUMERICAL_RESTORATION_ERRORS:
    error_path = NB13_LOG_ROOT / "numerical_restoration_errors.csv"
    pd.DataFrame(NUMERICAL_RESTORATION_ERRORS).to_csv(
        error_path, index=False
    )
    print(f"Error log: {error_path}")
    raise RuntimeError(
        "Section 12 failed. Resolve numerical restoration errors before proceeding."
    )

print("PASS: Numeric columns restored using reference dtypes.")
print("NOTE: Integer rounding restores representation only; it does not establish valid ranges or semantic correctness.")

SECTION 12 — RESTORE NUMERICAL VALUES
PASS | adult_income | Rows: 34,189 | Features: 14
PASS | bank_marketing | Rows: 31,647 | Features: 16
PASS | diabetes_130us | Rows: 71,236 | Features: 47
----------------------------------------------------------------------------------------------------
Datasets processed: 3
Successful restorations: 3
Failures: 0
PASS: Numeric columns restored using reference dtypes.
NOTE: Integer rounding restores representation only; it does not establish valid ranges or semantic correctness.


In [72]:

# ==================================================================================================
# NOTEBOOK 13 — SECTION 13: TARGET RESTORATION
# Classifier Comparison, Target Assignment, and Distribution Diagnostics
# ==================================================================================================

import json
import time
import hashlib
import warnings
import traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

print("=" * 100)
print("NOTEBOOK 13 — SECTION 13: TARGET RESTORATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 13.1 — CONFIGURATION
# --------------------------------------------------------------------------------------------------

DATASET_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

EXPECTED_DATASETS = list(DATASET_TARGETS)
MASTER_SEED = int(globals().get("MASTER_SEED", 2025))
VALIDATION_SIZE = 0.20

MAX_ITERATIONS = {
    "adult_income": 2000,
    "bank_marketing": 2000,
    "diabetes_130us": 3000,
}

IDENTIFIER_COLUMNS = {
    "id", "row_id", "record_id", "patient_id", "customer_id",
    "encounter_id", "__original_row_id__",
}

if "NB13_ROOT" not in globals():
    raise NameError("NB13_ROOT is missing. Run the earlier Notebook 13 setup sections.")

if "SYNTHETIC_DATAFRAMES" not in globals():
    raise NameError("SYNTHETIC_DATAFRAMES is missing. Run the feature-generation sections.")

if "REFERENCE_TRAIN" not in globals():
    raise NameError("REFERENCE_TRAIN is missing. Run the earlier data-loading sections.")

NB13_ROOT = Path(NB13_ROOT)
SYNTHETIC_DIR = NB13_ROOT / "synthetic_datasets"
METADATA_DIR = NB13_ROOT / "metadata"
VALIDATION_DIR = NB13_ROOT / "validation"
LOG_DIR = NB13_ROOT / "logs"

for directory in [SYNTHETIC_DIR, METADATA_DIR, VALIDATION_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

OUTPUT_VALIDATION = VALIDATION_DIR / "target_classifier_validation.csv"
OUTPUT_DISTRIBUTIONS = VALIDATION_DIR / "target_distribution_comparison.csv"
OUTPUT_REPORTS = VALIDATION_DIR / "target_classifier_classification_reports.json"
OUTPUT_METADATA = METADATA_DIR / "target_restoration_metadata.json"
OUTPUT_ERRORS = LOG_DIR / "target_restoration_errors.csv"
OUTPUT_VERIFICATION = VALIDATION_DIR / "target_restoration_final_verification.csv"

print("Datasets    :", EXPECTED_DATASETS)
print("Targets     :", DATASET_TARGETS)
print("Output root :", NB13_ROOT)
print("Master seed :", MASTER_SEED)
print("Validation  :", f"{VALIDATION_SIZE:.0%} held-out real data")

# --------------------------------------------------------------------------------------------------
# 13.2 — INPUT VALIDATION
# --------------------------------------------------------------------------------------------------

for dataset in EXPECTED_DATASETS:
    if dataset not in SYNTHETIC_DATAFRAMES:
        raise KeyError(f"Missing synthetic feature dataset: {dataset}")

    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Missing reference dataset: {dataset}")

    synthetic_df = SYNTHETIC_DATAFRAMES[dataset]
    reference_df = REFERENCE_TRAIN[dataset]
    target = DATASET_TARGETS[dataset]

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(f"{dataset}: synthetic data must be a DataFrame.")

    if not isinstance(reference_df, pd.DataFrame):
        raise TypeError(f"{dataset}: reference data must be a DataFrame.")

    if target not in reference_df.columns:
        raise KeyError(f"{dataset}: target column '{target}' not found.")

    if target in synthetic_df.columns:
        raise ValueError(
            f"{dataset}: target '{target}' already exists in synthetic features. "
            "Check earlier sections before continuing."
        )

    print(
        f"{dataset:<18} reference={reference_df.shape} "
        f"synthetic_features={synthetic_df.shape}"
    )

# --------------------------------------------------------------------------------------------------
# 13.3 — HELPER FUNCTIONS
# --------------------------------------------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def dataframe_sha256(df):
    digest = hashlib.sha256()
    digest.update(
        json.dumps(
            {
                "columns": [str(c) for c in df.columns],
                "dtypes": [str(d) for d in df.dtypes],
                "shape": list(df.shape),
            },
            sort_keys=True,
        ).encode("utf-8")
    )
    digest.update(
        pd.util.hash_pandas_object(df, index=True, categorize=True)
        .values.tobytes()
    )
    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return value
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return str(value)


def make_onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float64,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float64,
        )


def prepare_features(reference_df, synthetic_df, target):
    X = reference_df.drop(columns=[target]).copy()
    Xs = synthetic_df.copy()

    X = X.drop(
        columns=[
            c for c in X.columns
            if str(c).strip().lower() in IDENTIFIER_COLUMNS
        ],
        errors="ignore",
    )
    Xs = Xs.drop(
        columns=[
            c for c in Xs.columns
            if str(c).strip().lower() in IDENTIFIER_COLUMNS
        ],
        errors="ignore",
    )

    missing = [c for c in X.columns if c not in Xs.columns]
    extra = [c for c in Xs.columns if c not in X.columns]

    if missing or extra:
        raise ValueError(
            f"Feature schema mismatch. Missing={missing}; extra={extra}"
        )

    Xs = Xs[X.columns]

    if not all(isinstance(c, str) for c in X.columns):
        raise TypeError("Feature column names must be strings.")

    for col in X.columns:
        if (
            pd.api.types.is_object_dtype(X[col])
            or isinstance(X[col].dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(X[col])
        ):
            X[col] = X[col].astype("object")
            Xs[col] = Xs[col].astype("object")

    all_missing = X.columns[X.isna().all()].tolist()
    if all_missing:
        raise ValueError(f"Reference features entirely missing: {all_missing}")

    return X, Xs


def make_pipeline(X, dataset, class_weight):
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        )

    if categorical_cols:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_onehot()),
                    ]
                ),
                categorical_cols,
            )
        )

    if not transformers:
        raise ValueError("No usable predictors remain.")

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
    )

    classifier = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        max_iter=MAX_ITERATIONS[dataset],
        tol=1e-3,
        random_state=MASTER_SEED,
        average=True,
        class_weight=class_weight,
    )

    return (
        Pipeline(
            [
                ("preprocessor", preprocessor),
                ("classifier", classifier),
            ]
        ),
        numeric_cols,
        categorical_cols,
    )


def tv_distance(reference_target, synthetic_target):
    """Total variation distance between empirical target distributions."""
    p = reference_target.astype(str).value_counts(normalize=True)
    q = synthetic_target.astype(str).value_counts(normalize=True)
    labels = p.index.union(q.index)
    p = p.reindex(labels, fill_value=0.0)
    q = q.reindex(labels, fill_value=0.0)
    return float(0.5 * np.abs(p - q).sum())


def distribution_records(dataset, target, reference_y, synthetic_y):
    p = reference_y.astype(str).value_counts()
    q = synthetic_y.astype(str).value_counts()
    pp = reference_y.astype(str).value_counts(normalize=True)
    qq = synthetic_y.astype(str).value_counts(normalize=True)

    labels = sorted(set(p.index) | set(q.index))
    rows = []

    for label in labels:
        p_prop = float(pp.get(label, 0.0))
        q_prop = float(qq.get(label, 0.0))
        rows.append(
            {
                "dataset": dataset,
                "target": target,
                "class_label": label,
                "reference_count": int(p.get(label, 0)),
                "synthetic_count": int(q.get(label, 0)),
                "reference_proportion": p_prop,
                "synthetic_proportion": q_prop,
                "proportion_difference": q_prop - p_prop,
                "absolute_proportion_difference": abs(q_prop - p_prop),
            }
        )
    return rows


# --------------------------------------------------------------------------------------------------
# 13.4 — INITIALIZE REGISTRIES
# --------------------------------------------------------------------------------------------------

TARGET_RESTORATION_RESULTS = {}
TARGET_RESTORATION_METADATA = {}
TARGET_CLASSIFICATION_REPORTS = {}
TARGET_VALIDATION_ROWS = []
TARGET_DISTRIBUTION_ROWS = []
TARGET_RESTORATION_ERRORS = []

# --------------------------------------------------------------------------------------------------
# 13.5 — CLASSIFIER CANDIDATE EVALUATION AND TARGET ASSIGNMENT
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset in enumerate(EXPECTED_DATASETS):

    print("\n" + "=" * 100)
    print(f"DATASET: {dataset}")
    print("=" * 100)

    start_time = time.perf_counter()

    try:
        target = DATASET_TARGETS[dataset]
        seed = MASTER_SEED + dataset_index

        reference = REFERENCE_TRAIN[dataset].copy()
        synthetic_features = SYNTHETIC_DATAFRAMES[dataset].copy()

        reference_rows_before_filter = len(reference)
        reference = reference.loc[reference[target].notna()].copy()

        if reference.empty:
            raise ValueError("Reference data has no non-missing target rows.")

        y = reference[target].copy()

        if y.nunique() < 2:
            raise ValueError("Classification requires at least two target classes.")

        X, Xs = prepare_features(reference, synthetic_features, target)

        if len(X) != len(y):
            raise AssertionError("Reference predictor/target row mismatch.")

        # Use the same held-out real-data split for both candidates.
        counts = y.value_counts()
        stratify = y if counts.min() >= 2 else None

        X_train, X_valid, y_train, y_valid = train_test_split(
            X,
            y,
            test_size=VALIDATION_SIZE,
            random_state=seed,
            stratify=stratify,
        )

        candidates = {
            "unweighted": None,
            "balanced": "balanced",
        }

        candidate_results = {}
        candidate_models = {}

        for candidate_name, class_weight in candidates.items():
            model, numeric_cols, categorical_cols = make_pipeline(
                X_train,
                dataset,
                class_weight,
            )

            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter("always")
                model.fit(X_train, y_train)

            fit_warnings = [str(w.message) for w in captured]
            validation_prediction = model.predict(X_valid)

            report = classification_report(
                y_valid,
                validation_prediction,
                output_dict=True,
                zero_division=0,
            )

            macro_f1 = f1_score(
                y_valid,
                validation_prediction,
                average="macro",
                zero_division=0,
            )

            balanced_accuracy = balanced_accuracy_score(
                y_valid,
                validation_prediction,
            )

            accuracy = accuracy_score(y_valid, validation_prediction)

            candidate_results[candidate_name] = {
                "accuracy": float(accuracy),
                "balanced_accuracy": float(balanced_accuracy),
                "macro_f1": float(macro_f1),
                "weighted_f1": float(
                    f1_score(
                        y_valid,
                        validation_prediction,
                        average="weighted",
                        zero_division=0,
                    )
                ),
                "classification_report": json_safe(report),
                "confusion_matrix": confusion_matrix(
                    y_valid,
                    validation_prediction,
                    labels=list(model.named_steps["classifier"].classes_),
                ).tolist(),
                "class_labels": [
                    str(v) for v in model.named_steps["classifier"].classes_
                ],
                "fit_warnings": fit_warnings,
                "class_weight": class_weight,
            }

            candidate_models[candidate_name] = model

            print(
                f"{candidate_name:<12} "
                f"accuracy={accuracy:.4f} | "
                f"balanced_accuracy={balanced_accuracy:.4f} | "
                f"macro_f1={macro_f1:.4f} | "
                f"warnings={len(fit_warnings)}"
            )

        # Select using held-out validation macro F1.
        # Deterministic tie-breaker: balanced accuracy, then unweighted model.
        selected_name = max(
            candidate_results,
            key=lambda name: (
                candidate_results[name]["macro_f1"],
                candidate_results[name]["balanced_accuracy"],
                name == "unweighted",
            ),
        )

        selected_validation = candidate_results[selected_name]

        print(f"\nSelected candidate: {selected_name}")
        print("Selection criterion: validation macro F1, then balanced accuracy.")

        # Fit the selected model on all available reference rows.
        selected_class_weight = candidates[selected_name]
        final_model, numeric_cols, categorical_cols = make_pipeline(
            X,
            dataset,
            selected_class_weight,
        )

        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            final_model.fit(X, y)

        final_fit_warnings = [str(w.message) for w in captured]

        # Assign synthetic target labels. Do not force class proportions.
        predicted_target = final_model.predict(Xs)

        if len(predicted_target) != len(synthetic_features):
            raise AssertionError("Predicted target count differs from synthetic row count.")

        restored = synthetic_features.copy()
        restored[target] = predicted_target

        if restored[target].isna().any():
            raise ValueError("Target assignment produced missing values.")

        if len(restored) != len(synthetic_features):
            raise AssertionError("Synthetic row count changed during restoration.")

        reference_distribution = y.astype(str).value_counts(normalize=True).to_dict()
        synthetic_distribution = (
            restored[target].astype(str).value_counts(normalize=True).to_dict()
        )

        distance = tv_distance(y, restored[target])

        minority_label = y.value_counts().idxmin()
        minority_reference_proportion = float((y == minority_label).mean())
        minority_synthetic_proportion = float(
            (restored[target] == minority_label).mean()
        )

        zero_prediction_classes = [
            str(label)
            for label in y.unique()
            if not (restored[target] == label).any()
        ]

        # A warning is a diagnostic trigger, not a formal statistical test.
        distribution_review = (
            distance > 0.10
            or bool(zero_prediction_classes)
        )

        distribution_rows = distribution_records(
            dataset,
            target,
            y,
            restored[target],
        )
        TARGET_DISTRIBUTION_ROWS.extend(distribution_rows)

        output_csv = SYNTHETIC_DIR / (
            f"{dataset}_synthetic_with_predicted_target.csv"
        )

        restored.to_csv(output_csv, index=False, encoding="utf-8")
        file_hash = sha256_file(output_csv)
        dataframe_hash = dataframe_sha256(restored)

        runtime = time.perf_counter() - start_time

        TARGET_RESTORATION_RESULTS[dataset] = restored

        TARGET_CLASSIFICATION_REPORTS[dataset] = {
            "selected_candidate": selected_name,
            "candidate_comparison": candidate_results,
            "selected_validation_report": selected_validation[
                "classification_report"
            ],
            "selected_confusion_matrix": selected_validation["confusion_matrix"],
            "selected_confusion_matrix_labels": selected_validation["class_labels"],
            "final_fit_warnings": final_fit_warnings,
        }

        metadata = {
            "dataset": dataset,
            "target": target,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "master_seed": MASTER_SEED,
            "dataset_seed": seed,
            "validation_size": VALIDATION_SIZE,
            "candidate_models": list(candidates.keys()),
            "selection_metric": "validation_macro_f1",
            "selected_candidate": selected_name,
            "selected_class_weight": selected_class_weight,
            "candidate_validation_metrics": {
                name: {
                    "accuracy": result["accuracy"],
                    "balanced_accuracy": result["balanced_accuracy"],
                    "macro_f1": result["macro_f1"],
                    "weighted_f1": result["weighted_f1"],
                }
                for name, result in candidate_results.items()
            },
            "reference_rows_before_target_filter": reference_rows_before_filter,
            "reference_rows_used": len(reference),
            "synthetic_rows": len(restored),
            "predictor_count": X.shape[1],
            "numeric_predictor_count": len(numeric_cols),
            "categorical_predictor_count": len(categorical_cols),
            "reference_target_distribution": reference_distribution,
            "synthetic_target_distribution": synthetic_distribution,
            "target_distribution_tv_distance": distance,
            "minority_reference_class": str(minority_label),
            "minority_reference_proportion": minority_reference_proportion,
            "minority_synthetic_proportion": minority_synthetic_proportion,
            "classes_with_zero_predictions": zero_prediction_classes,
            "distribution_review_flag": bool(distribution_review),
            "selected_validation_warnings": selected_validation["fit_warnings"],
            "final_fit_warnings": final_fit_warnings,
            "reference_dataframe_sha256": dataframe_sha256(reference),
            "synthetic_features_dataframe_sha256": dataframe_sha256(synthetic_features),
            "synthetic_output_dataframe_sha256": dataframe_hash,
            "synthetic_output_file_sha256": file_hash,
            "output_csv": str(output_csv),
            "runtime_seconds": runtime,
            "execution_status": "PASS",
            "target_assignment_method": "Classifier-assigned; not directly generated",
            "end_to_end_differential_privacy_established": False,
            "scikit_learn_version": sklearn.__version__,
            "numpy_version": np.__version__,
            "pandas_version": pd.__version__,
        }

        TARGET_RESTORATION_METADATA[dataset] = metadata

        TARGET_VALIDATION_ROWS.append(
            {
                "dataset": dataset,
                "target": target,
                "selected_candidate": selected_name,
                "reference_rows": len(reference),
                "synthetic_rows": len(restored),
                "predictor_count": X.shape[1],
                "validation_accuracy": selected_validation["accuracy"],
                "validation_balanced_accuracy": selected_validation["balanced_accuracy"],
                "validation_macro_f1": selected_validation["macro_f1"],
                "validation_weighted_f1": selected_validation["weighted_f1"],
                "unweighted_validation_macro_f1": candidate_results["unweighted"]["macro_f1"],
                "balanced_validation_macro_f1": candidate_results["balanced"]["macro_f1"],
                "target_distribution_tv_distance": distance,
                "minority_reference_class": str(minority_label),
                "minority_reference_proportion": minority_reference_proportion,
                "minority_synthetic_proportion": minority_synthetic_proportion,
                "zero_prediction_classes": json.dumps(zero_prediction_classes),
                "distribution_review_flag": bool(distribution_review),
                "validation_warning_count": len(selected_validation["fit_warnings"]),
                "final_fit_warning_count": len(final_fit_warnings),
                "runtime_seconds": runtime,
                "output_csv": str(output_csv),
                "output_sha256": file_hash,
                "execution_status": "PASS",
            }
        )

        print(f"Reference rows           : {len(reference):,}")
        print(f"Synthetic rows           : {len(restored):,}")
        print(f"Target                   : {target}")
        print(f"Predictor count          : {X.shape[1]}")
        print(f"Selected validation F1   : {selected_validation['macro_f1']:.4f}")
        print(f"Selected balanced acc.   : {selected_validation['balanced_accuracy']:.4f}")
        print(f"Distribution TV distance: {distance:.4f}")
        print(f"Minority class           : {minority_label}")
        print(f"Reference minority share : {minority_reference_proportion:.4%}")
        print(f"Synthetic minority share : {minority_synthetic_proportion:.4%}")
        print(f"Zero-prediction classes  : {zero_prediction_classes}")
        print("\nPredicted target distribution:")
        print(restored[target].value_counts(dropna=False))
        print(f"Saved                    : {output_csv}")
        print(f"SHA-256                  : {file_hash}")
        print(f"Runtime                  : {runtime:.2f}s")
        print("EXECUTION STATUS         : PASS")
        print(
            "FIDELITY DIAGNOSTIC      : "
            + ("REVIEW REQUIRED" if distribution_review else "NO AUTOMATIC FLAG")
        )

    except Exception as error:
        TARGET_RESTORATION_ERRORS.append(
            {
                "dataset": dataset,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "traceback": traceback.format_exc(),
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            }
        )
        print(f"STATUS: FAIL — {type(error).__name__}: {error}")
        print(traceback.format_exc())

# --------------------------------------------------------------------------------------------------
# 13.6 — SAVE ARTIFACTS
# --------------------------------------------------------------------------------------------------

validation_df = pd.DataFrame(TARGET_VALIDATION_ROWS)
distribution_df = pd.DataFrame(TARGET_DISTRIBUTION_ROWS)
errors_df = pd.DataFrame(TARGET_RESTORATION_ERRORS)

validation_df.to_csv(OUTPUT_VALIDATION, index=False)
distribution_df.to_csv(OUTPUT_DISTRIBUTIONS, index=False)
errors_df.to_csv(OUTPUT_ERRORS, index=False)

with open(OUTPUT_REPORTS, "w", encoding="utf-8") as handle:
    json.dump(
        json_safe(TARGET_CLASSIFICATION_REPORTS),
        handle,
        indent=2,
        ensure_ascii=False,
    )

metadata_payload = {
    "section": "Notebook 13 — Section 13: Target Restoration",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "master_seed": MASTER_SEED,
    "expected_datasets": EXPECTED_DATASETS,
    "successful_datasets": list(TARGET_RESTORATION_RESULTS.keys()),
    "failed_datasets": [
        d for d in EXPECTED_DATASETS
        if d not in TARGET_RESTORATION_RESULTS
    ],
    "dataset_metadata": TARGET_RESTORATION_METADATA,
    "validation_csv": str(OUTPUT_VALIDATION),
    "distribution_comparison_csv": str(OUTPUT_DISTRIBUTIONS),
    "classification_reports_json": str(OUTPUT_REPORTS),
    "errors_csv": str(OUTPUT_ERRORS),
    "research_limitation": (
        "Targets are classifier-assigned. Classifier validation metrics do not establish "
        "synthetic joint-distribution fidelity or end-to-end differential privacy."
    ),
}

with open(OUTPUT_METADATA, "w", encoding="utf-8") as handle:
    json.dump(
        json_safe(metadata_payload),
        handle,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# 13.7 — FINAL VERIFICATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 13 — FINAL VERIFICATION")
print("=" * 100)

verification_rows = []

for dataset in EXPECTED_DATASETS:
    target = DATASET_TARGETS[dataset]

    if dataset not in TARGET_RESTORATION_RESULTS:
        verification_rows.append(
            {"dataset": dataset, "status": "FAIL", "details": "Restoration failed"}
        )
        print(f"FAIL | {dataset:<18} | no output dataset")
        continue

    restored = TARGET_RESTORATION_RESULTS[dataset]
    output_csv = SYNTHETIC_DIR / f"{dataset}_synthetic_with_predicted_target.csv"

    checks = {
        "target_present": target in restored.columns,
        "target_complete": target in restored.columns and not restored[target].isna().any(),
        "row_count_preserved": len(restored) == len(SYNTHETIC_DATAFRAMES[dataset]),
        "output_exists": output_csv.exists(),
    }

    try:
        reloaded = pd.read_csv(output_csv)
        checks["saved_shape_matches"] = reloaded.shape == restored.shape
        checks["saved_columns_match"] = list(reloaded.columns) == list(restored.columns)
        checks["sha256_matches"] = (
            sha256_file(output_csv)
            == TARGET_RESTORATION_METADATA[dataset]["synthetic_output_file_sha256"]
        )
    except Exception:
        checks["saved_shape_matches"] = False
        checks["saved_columns_match"] = False
        checks["sha256_matches"] = False

    passed = all(checks.values())

    verification_rows.append(
        {
            "dataset": dataset,
            "status": "PASS" if passed else "FAIL",
            "details": json.dumps(checks, sort_keys=True),
        }
    )

    print(
        f"{'PASS' if passed else 'FAIL'} | {dataset:<18} "
        f"| shape={restored.shape} | target={target} "
        f"| missing={int(restored[target].isna().sum())}"
    )

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(OUTPUT_VERIFICATION, index=False)

successful = len(TARGET_RESTORATION_RESULTS)
failed = len(EXPECTED_DATASETS) - successful
verified = int((verification_df["status"] == "PASS").sum()) if len(verification_df) else 0

review_datasets = [
    dataset for dataset, item in TARGET_RESTORATION_METADATA.items()
    if item["distribution_review_flag"]
]

print("\nExpected datasets       :", len(EXPECTED_DATASETS))
print("Successful restorations :", successful)
print("Failed restorations     :", failed)
print("Verification PASS       :", verified)
print("Validation CSV          :", OUTPUT_VALIDATION)
print("Distribution comparison :", OUTPUT_DISTRIBUTIONS)
print("Classification reports  :", OUTPUT_REPORTS)
print("Metadata JSON           :", OUTPUT_METADATA)
print("Error log               :", OUTPUT_ERRORS)
print("Verification CSV        :", OUTPUT_VERIFICATION)

if failed == 0 and verified == len(EXPECTED_DATASETS):
    SECTION_13_STATUS = "PASS — execution and artifacts verified"
else:
    SECTION_13_STATUS = "FAIL — inspect error and verification logs"

print("\nSECTION 13 STATUS:", SECTION_13_STATUS)
print("FIDELITY REVIEW REQUIRED:", review_datasets)

print(
    "\nRESEARCH LIMITATION: Targets are classifier-assigned. "
    "Classifier metrics on held-out real data do not prove synthetic joint-distribution "
    "fidelity or end-to-end differential privacy."
)

SYNTHETIC_DATAFRAMES_WITH_TARGET = TARGET_RESTORATION_RESULTS

print("\nAvailable output variable: SYNTHETIC_DATAFRAMES_WITH_TARGET")
print("=" * 100)


NOTEBOOK 13 — SECTION 13: TARGET RESTORATION
Datasets    : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets     : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Output root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13
Master seed : 2025
Validation  : 20% held-out real data
adult_income       reference=(34189, 16) synthetic_features=(34189, 14)
bank_marketing     reference=(31647, 18) synthetic_features=(31647, 16)
diabetes_130us     reference=(71236, 49) synthetic_features=(71236, 47)

DATASET: adult_income
unweighted   accuracy=0.8511 | balanced_accuracy=0.7653 | macro_f1=0.7818 | warnings=1
balanced     accuracy=0.8099 | balanced_accuracy=0.8187 | macro_f1=0.7715 | warnings=1

Selected candidate: unweighted
Selection criterion: validation macro F1, then balanced accuracy.
Reference rows           : 34,189
Synthetic rows           : 34,189
Target                   : income
Predictor count          : 14
Selected 

In [74]:

# =============================================================================
# SECTION 14 — VALIDATE SCHEMA
# SPP-GAN | Validate restored synthetic datasets against reference schemas
# =============================================================================

import json
import pandas as pd
from pathlib import Path

print("=" * 100)
print("SECTION 14 — VALIDATE SCHEMA")
print("=" * 100)

# -----------------------------------------------------------------------------
# 14.1 — CONFIGURATION
# -----------------------------------------------------------------------------

if "REFERENCE_TRAIN" not in globals():
    raise NameError("REFERENCE_TRAIN is missing. Run the earlier data-loading sections.")

if "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals():
    raise NameError(
        "SYNTHETIC_DATAFRAMES_WITH_TARGET is missing. "
        "Run the updated Section 13 first."
    )

if "DATASET_TARGETS" not in globals():
    DATASET_TARGETS = {
        "adult_income": "income",
        "bank_marketing": "y",
        "diabetes_130us": "readmitted",
    }

if "NB13_VALIDATION_ROOT" not in globals():
    NB13_VALIDATION_ROOT = Path(NB13_ROOT) / "validation"

NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_DATASETS = list(DATASET_TARGETS)

# This identifier is intentionally not generated by the model.
IDENTIFIER_COLUMNS = {"__original_row_id__"}

# -----------------------------------------------------------------------------
# 14.2 — SCHEMA VALIDATION FUNCTION
# -----------------------------------------------------------------------------

def validate_schema(dataset, dataframe):
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        return {
            "dataset": dataset,
            "schema_status": "NO_REFERENCE",
            "expected_columns": None,
            "actual_columns": len(dataframe.columns),
            "missing_columns": None,
            "unexpected_columns": None,
            "duplicate_columns": None,
            "target_present": False,
            "identifier_policy": "Not evaluated",
            "schema_pass": False,
        }

    target = DATASET_TARGETS[dataset]

    expected_columns = list(reference.columns)
    actual_columns = list(dataframe.columns)

    # A source identifier is not expected to be synthesized.
    expected_model_columns = [
        column for column in expected_columns
        if column not in IDENTIFIER_COLUMNS
    ]

    missing = [
        column for column in expected_model_columns
        if column not in actual_columns
    ]

    unexpected = [
        column for column in actual_columns
        if column not in expected_model_columns
    ]

    duplicate_columns = [
        str(column)
        for column in dataframe.columns[dataframe.columns.duplicated()].tolist()
    ]

    target_present = target in actual_columns

    # Verify column order as well as column membership.
    expected_order = expected_model_columns
    actual_order = actual_columns
    column_order_matches = actual_order == expected_order

    schema_pass = (
        not missing
        and not unexpected
        and not duplicate_columns
        and target_present
        and column_order_matches
    )

    return {
        "dataset": dataset,
        "schema_status": "PASS" if schema_pass else "FAIL",
        "expected_columns": len(expected_model_columns),
        "actual_columns": len(actual_columns),
        "missing_columns": json.dumps(missing),
        "unexpected_columns": json.dumps(unexpected),
        "duplicate_columns": json.dumps(duplicate_columns),
        "target_column": target,
        "target_present": target_present,
        "identifier_policy": (
            "Excluded from expected synthetic schema: "
            "__original_row_id__ is not generated"
        ),
        "column_order_matches": column_order_matches,
        "schema_pass": schema_pass,
    }

# -----------------------------------------------------------------------------
# 14.3 — VALIDATE ALL RESTORED DATASETS
# -----------------------------------------------------------------------------

SCHEMA_VALIDATION_ROWS = []

for dataset in EXPECTED_DATASETS:
    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        SCHEMA_VALIDATION_ROWS.append({
            "dataset": dataset,
            "schema_status": "MISSING_OUTPUT",
            "expected_columns": None,
            "actual_columns": None,
            "missing_columns": None,
            "unexpected_columns": None,
            "duplicate_columns": None,
            "target_column": DATASET_TARGETS[dataset],
            "target_present": False,
            "identifier_policy": "Not evaluated",
            "column_order_matches": False,
            "schema_pass": False,
        })
        continue

    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]
    result = validate_schema(dataset, synthetic_df)
    SCHEMA_VALIDATION_ROWS.append(result)

SCHEMA_VALIDATION_DF = pd.DataFrame(SCHEMA_VALIDATION_ROWS)

SCHEMA_VALIDATION_PATH = (
    NB13_VALIDATION_ROOT / "schema_validation.csv"
)

SCHEMA_VALIDATION_DF.to_csv(
    SCHEMA_VALIDATION_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 14.4 — DISPLAY RESULTS
# -----------------------------------------------------------------------------

display(SCHEMA_VALIDATION_DF)

print("\n" + "=" * 100)
print("SECTION 14 — FINAL SUMMARY")
print("=" * 100)

total_datasets = len(SCHEMA_VALIDATION_DF)
passed_datasets = int(SCHEMA_VALIDATION_DF["schema_pass"].sum())
failed_datasets = total_datasets - passed_datasets

print(f"Expected datasets : {len(EXPECTED_DATASETS)}")
print(f"Schema PASS       : {passed_datasets}")
print(f"Schema FAIL       : {failed_datasets}")
print(f"Validation file   : {SCHEMA_VALIDATION_PATH}")

if failed_datasets == 0:
    SECTION_14_STATUS = "PASS — restored synthetic schemas verified"
else:
    SECTION_14_STATUS = "FAIL — inspect missing/unexpected columns and order"

print("\nSECTION 14 STATUS:", SECTION_14_STATUS)
print(
    "\nNOTE: __original_row_id__ is excluded intentionally because it is a source "
    "identifier, not a generated predictor. Target columns must be present."
)
print("=" * 100)


SECTION 14 — VALIDATE SCHEMA


,dataset,schema_status,expected_columns,actual_columns,missing_columns,unexpected_columns,duplicate_columns,target_column,target_present,identifier_policy,column_order_matches,schema_pass
0,adult_income,PASS,15,15,[],[],[],income,True,Excluded from expected synthetic schema: __ori...,True,True
1,bank_marketing,PASS,17,17,[],[],[],y,True,Excluded from expected synthetic schema: __ori...,True,True
2,diabetes_130us,PASS,48,48,[],[],[],readmitted,True,Excluded from expected synthetic schema: __ori...,True,True



SECTION 14 — FINAL SUMMARY
Expected datasets : 3
Schema PASS       : 3
Schema FAIL       : 0
Validation file   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/schema_validation.csv

SECTION 14 STATUS: PASS — restored synthetic schemas verified

NOTE: __original_row_id__ is excluded intentionally because it is a source identifier, not a generated predictor. Target columns must be present.


In [77]:

# ==================================================================================================
# SECTION 15 — RANGE AND DOMAIN VALIDATION
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Validate synthetic numeric values against the real training-data ranges.
#   2. Check known feature-specific domain constraints.
#   3. Detect missing, non-numeric, non-finite, and fractional values.
#   4. Save detailed reports for reproducibility and research auditing.
#   5. Preserve the original synthetic data without silent clipping or correction.
#
# Required existing objects:
#   SYNTHETIC_DATAFRAMES_WITH_TARGET
#   REFERENCE_TRAIN
#   DATASET_IDS
#   NB13_ROOT
#
# Outputs:
#   validation/range_validation.csv
#   validation/domain_validation.csv
#   validation/range_validation_summary.csv
#   validation/section_15_validation_metadata.json
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 15 — RANGE AND DOMAIN VALIDATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED_GLOBALS = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_ROOT",
]

_missing_globals = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _missing_globals:
    raise NameError(
        "Section 15 cannot run. Missing required objects: "
        + ", ".join(_missing_globals)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError(
        "REFERENCE_TRAIN must be a dictionary keyed by dataset ID. "
        "Check the reference-data loading section."
    )

DATASET_IDS = list(DATASET_IDS)
NB13_ROOT = Path(NB13_ROOT)

VALIDATION_ROOT = NB13_ROOT / "validation"
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

RANGE_REPORT_PATH = VALIDATION_ROOT / "range_validation.csv"
DOMAIN_REPORT_PATH = VALIDATION_ROOT / "domain_validation.csv"
SUMMARY_REPORT_PATH = VALIDATION_ROOT / "range_validation_summary.csv"
METADATA_PATH = VALIDATION_ROOT / "section_15_validation_metadata.json"

# --------------------------------------------------------------------------------------------------
# 2. Feature-specific domain rules
# --------------------------------------------------------------------------------------------------
# These are explicit semantic checks, not replacements for the observed reference ranges.
# A rule is applied only when the feature exists in both datasets.
#
# Rule format:
#   (minimum, maximum, allowed_special_values, description)
#
# None means no bound on that side.
# Special values are allowed only when explicitly listed.

DOMAIN_RULES = {
    "adult_income": {
        "age": (0, None, [], "Age must be non-negative."),
        "education_num": (1, 16, [], "Education level code must be between 1 and 16."),
        "capital_gain": (0, None, [], "Capital gain must be non-negative."),
        "capital_loss": (0, None, [], "Capital loss must be non-negative."),
        "hours_per_week": (0, 168, [], "Hours per week must be between 0 and 168."),
    },
    "bank_marketing": {
        "age": (0, None, [], "Age must be non-negative."),
        "day": (1, 31, [], "Day of month must be between 1 and 31."),
        "duration": (0, None, [], "Call duration must be non-negative."),
        "campaign": (1, None, [], "Campaign contact count must be at least 1."),
        "pdays": (-1, None, [-1], "Pdays must be -1 (not previously contacted) or non-negative."),
        "previous": (0, None, [], "Previous contact count must be non-negative."),
    },
    "diabetes_130us": {
        "admission_type_id": (1, 8, [], "Admission type code must be between 1 and 8."),
        "discharge_disposition_id": (1, 28, [], "Discharge disposition code must be between 1 and 28."),
        "admission_source_id": (1, 25, [], "Admission source code must be between 1 and 25."),
        "time_in_hospital": (1, 14, [], "Length of stay must be between 1 and 14 days."),
        "num_lab_procedures": (0, None, [], "Lab procedure count must be non-negative."),
        "num_procedures": (0, None, [], "Procedure count must be non-negative."),
        "num_medications": (0, None, [], "Medication count must be non-negative."),
        "number_outpatient": (0, None, [], "Outpatient visit count must be non-negative."),
        "number_emergency": (0, None, [], "Emergency visit count must be non-negative."),
        "number_inpatient": (0, None, [], "Inpatient visit count must be non-negative."),
        "number_diagnoses": (1, 16, [], "Diagnosis count must be between 1 and 16."),
    },
}

# Common count/code columns expected to be integer-valued.
INTEGER_FEATURES = {
    "adult_income": {
        "age", "fnlwgt", "education_num", "capital_gain", "capital_loss",
        "hours_per_week",
    },
    "bank_marketing": {
        "age", "balance", "day", "duration", "campaign", "pdays", "previous",
    },
    "diabetes_130us": {
        "admission_type_id", "discharge_disposition_id", "admission_source_id",
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses",
    },
}

# --------------------------------------------------------------------------------------------------
# 3. Helpers for retrieving and validating data
# --------------------------------------------------------------------------------------------------

def _get_dataframe(registry, dataset_id, registry_name):
    """Retrieve a DataFrame from a dataset-keyed registry."""
    if dataset_id not in registry:
        raise KeyError(
            f"{registry_name} has no entry for dataset '{dataset_id}'. "
            f"Available keys: {list(registry.keys())}"
        )

    df = registry[dataset_id]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{registry_name}['{dataset_id}'] must be a pandas DataFrame; "
            f"received {type(df).__name__}."
        )

    if df.empty:
        warnings.warn(f"{registry_name}['{dataset_id}'] is empty.")

    return df


def _numeric_series(series):
    """Convert a copy of a series to numeric values; preserve the source DataFrame."""
    return pd.to_numeric(series, errors="coerce")


def _count_outside(values, minimum=None, maximum=None, allowed_special=None):
    """Count values outside bounds, excluding missing/non-finite values and allowed specials."""
    allowed_special = allowed_special or []

    valid = values.notna() & np.isfinite(values)
    checked = values[valid]

    if allowed_special:
        checked = checked[~checked.isin(allowed_special)]

    below = int((checked < minimum).sum()) if minimum is not None else 0
    above = int((checked > maximum).sum()) if maximum is not None else 0

    return below, above, int(below + above), int(len(checked))


# --------------------------------------------------------------------------------------------------
# 4. Run range and domain checks
# --------------------------------------------------------------------------------------------------

range_records = []
domain_records = []
dataset_summaries = []

print("\n" + "-" * 100)
print("VALIDATING SYNTHETIC DATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:
    reference_df = _get_dataframe(REFERENCE_TRAIN, dataset_id, "REFERENCE_TRAIN")
    synthetic_df = _get_dataframe(
        SYNTHETIC_DATAFRAMES_WITH_TARGET,
        dataset_id,
        "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    )

    # Exclude row identifiers from numerical validation.
    identifier_columns = {
        col for col in reference_df.columns
        if str(col).strip().lower() in {
            "__original_row_id__", "original_row_id", "row_id", "record_id"
        }
    }

    # Only common columns can be compared directly.
    common_columns = [
        col for col in reference_df.columns
        if col in synthetic_df.columns and col not in identifier_columns
    ]

    numeric_columns = [
        col for col in common_columns
        if pd.api.types.is_numeric_dtype(reference_df[col])
    ]

    dataset_range_count = 0
    dataset_range_pass = 0
    dataset_range_review = 0
    dataset_domain_count = 0
    dataset_domain_pass = 0
    dataset_domain_review = 0

    # ----------------------------------------------------------------------------------------------
    # 4A. Observed reference-range validation
    # ----------------------------------------------------------------------------------------------

    for column in numeric_columns:
        ref_numeric = _numeric_series(reference_df[column])
        syn_numeric = _numeric_series(synthetic_df[column])

        ref_finite = ref_numeric[ref_numeric.notna() & np.isfinite(ref_numeric)]
        syn_finite = syn_numeric[syn_numeric.notna() & np.isfinite(syn_numeric)]

        ref_min = float(ref_finite.min()) if not ref_finite.empty else np.nan
        ref_max = float(ref_finite.max()) if not ref_finite.empty else np.nan
        syn_min = float(syn_finite.min()) if not syn_finite.empty else np.nan
        syn_max = float(syn_finite.max()) if not syn_finite.empty else np.nan

        below_count = (
            int((syn_finite < ref_min).sum()) if not np.isnan(ref_min) else 0
        )
        above_count = (
            int((syn_finite > ref_max).sum()) if not np.isnan(ref_max) else 0
        )

        missing_count = int(synthetic_df[column].isna().sum())
        nonnumeric_count = int(
            (synthetic_df[column].notna() & syn_numeric.isna()).sum()
        )
        nonfinite_count = int(
            (syn_numeric.notna() & ~np.isfinite(syn_numeric)).sum()
        )

        is_integer_feature = column in INTEGER_FEATURES.get(dataset_id, set())
        fractional_count = (
            int(
                (
                    syn_finite.notna()
                    & ~np.isclose(syn_finite, np.round(syn_finite), atol=1e-8)
                ).sum()
            )
            if is_integer_feature
            else 0
        )

        reasons = []
        if np.isnan(ref_min) or np.isnan(ref_max):
            reasons.append("REFERENCE_HAS_NO_FINITE_VALUES")
        if below_count:
            reasons.append("BELOW_REFERENCE_MIN")
        if above_count:
            reasons.append("ABOVE_REFERENCE_MAX")
        if missing_count:
            reasons.append("MISSING_VALUES")
        if nonnumeric_count:
            reasons.append("NON_NUMERIC_VALUES")
        if nonfinite_count:
            reasons.append("NON_FINITE_VALUES")
        if fractional_count:
            reasons.append("FRACTIONAL_INTEGER_VALUES")

        status = "PASS" if not reasons else "REVIEW"

        range_records.append({
            "dataset_id": dataset_id,
            "column": column,
            "reference_dtype": str(reference_df[column].dtype),
            "synthetic_dtype": str(synthetic_df[column].dtype),
            "reference_rows": int(len(reference_df)),
            "synthetic_rows": int(len(synthetic_df)),
            "reference_min": ref_min,
            "reference_max": ref_max,
            "synthetic_min": syn_min,
            "synthetic_max": syn_max,
            "below_reference_min_count": below_count,
            "above_reference_max_count": above_count,
            "out_of_reference_range_count": below_count + above_count,
            "missing_count": missing_count,
            "nonnumeric_count": nonnumeric_count,
            "nonfinite_count": nonfinite_count,
            "fractional_integer_count": fractional_count,
            "integer_feature_check": bool(is_integer_feature),
            "status": status,
            "review_reasons": ";".join(reasons),
        })

        dataset_range_count += 1
        if status == "PASS":
            dataset_range_pass += 1
        else:
            dataset_range_review += 1

    # ----------------------------------------------------------------------------------------------
    # 4B. Explicit semantic domain validation
    # ----------------------------------------------------------------------------------------------

    dataset_rules = DOMAIN_RULES.get(dataset_id, {})

    for column, rule in dataset_rules.items():
        if column not in reference_df.columns or column not in synthetic_df.columns:
            domain_records.append({
                "dataset_id": dataset_id,
                "column": column,
                "rule_min": rule[0],
                "rule_max": rule[1],
                "allowed_special_values": json.dumps(rule[2]),
                "rule_description": rule[3],
                "synthetic_min": np.nan,
                "synthetic_max": np.nan,
                "below_domain_min_count": np.nan,
                "above_domain_max_count": np.nan,
                "out_of_domain_count": np.nan,
                "missing_count": np.nan,
                "nonnumeric_count": np.nan,
                "nonfinite_count": np.nan,
                "status": "NOT_CHECKED",
                "review_reasons": "COLUMN_MISSING_FROM_REFERENCE_OR_SYNTHETIC_DATA",
            })
            dataset_domain_count += 1
            dataset_domain_review += 1
            continue

        minimum, maximum, allowed_special, description = rule
        raw_series = synthetic_df[column]
        values = _numeric_series(raw_series)
        finite_values = values[values.notna() & np.isfinite(values)]

        below, above, outside_count, checked_count = _count_outside(
            finite_values,
            minimum=minimum,
            maximum=maximum,
            allowed_special=allowed_special,
        )

        missing_count = int(raw_series.isna().sum())
        nonnumeric_count = int((raw_series.notna() & values.isna()).sum())
        nonfinite_count = int(
            (values.notna() & ~np.isfinite(values)).sum()
        )

        reasons = []
        if below:
            reasons.append("BELOW_DOMAIN_MIN")
        if above:
            reasons.append("ABOVE_DOMAIN_MAX")
        if missing_count:
            reasons.append("MISSING_VALUES")
        if nonnumeric_count:
            reasons.append("NON_NUMERIC_VALUES")
        if nonfinite_count:
            reasons.append("NON_FINITE_VALUES")

        status = "PASS" if not reasons else "REVIEW"

        domain_records.append({
            "dataset_id": dataset_id,
            "column": column,
            "rule_min": minimum,
            "rule_max": maximum,
            "allowed_special_values": json.dumps(allowed_special),
            "rule_description": description,
            "synthetic_min": (
                float(finite_values.min()) if not finite_values.empty else np.nan
            ),
            "synthetic_max": (
                float(finite_values.max()) if not finite_values.empty else np.nan
            ),
            "below_domain_min_count": below,
            "above_domain_max_count": above,
            "out_of_domain_count": outside_count,
            "missing_count": missing_count,
            "nonnumeric_count": nonnumeric_count,
            "nonfinite_count": nonfinite_count,
            "status": status,
            "review_reasons": ";".join(reasons),
        })

        dataset_domain_count += 1
        if status == "PASS":
            dataset_domain_pass += 1
        else:
            dataset_domain_review += 1

    dataset_summaries.append({
        "dataset_id": dataset_id,
        "reference_rows": int(len(reference_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "numeric_columns_checked": dataset_range_count,
        "range_pass_columns": dataset_range_pass,
        "range_review_columns": dataset_range_review,
        "domain_rules_checked": dataset_domain_count,
        "domain_pass_rules": dataset_domain_pass,
        "domain_review_rules": dataset_domain_review,
        "range_status": "PASS" if dataset_range_review == 0 else "REVIEW",
        "domain_status": "PASS" if dataset_domain_review == 0 else "REVIEW",
    })

    print(
        f"{dataset_id}: "
        f"range PASS={dataset_range_pass}, REVIEW={dataset_range_review}; "
        f"domain PASS={dataset_domain_pass}, REVIEW={dataset_domain_review}"
    )

# --------------------------------------------------------------------------------------------------
# 5. Build and save validation reports
# --------------------------------------------------------------------------------------------------

range_validation_df = pd.DataFrame(range_records)
domain_validation_df = pd.DataFrame(domain_records)
range_validation_summary_df = pd.DataFrame(dataset_summaries)

range_validation_df.to_csv(RANGE_REPORT_PATH, index=False)
domain_validation_df.to_csv(DOMAIN_REPORT_PATH, index=False)
range_validation_summary_df.to_csv(SUMMARY_REPORT_PATH, index=False)

range_review_rows = (
    int((range_validation_df["status"] != "PASS").sum())
    if not range_validation_df.empty else 0
)
domain_review_rows = (
    int((domain_validation_df["status"] != "PASS").sum())
    if not domain_validation_df.empty else 0
)

metadata = {
    "section": 15,
    "section_name": "Range and Domain Validation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(NB13_ROOT.parent.parent.parent),
    "notebook_13_root": str(NB13_ROOT),
    "dataset_ids": DATASET_IDS,
    "range_validation_rows": int(len(range_validation_df)),
    "range_pass_rows": int(
        (range_validation_df["status"] == "PASS").sum()
    ) if not range_validation_df.empty else 0,
    "range_review_rows": range_review_rows,
    "domain_validation_rows": int(len(domain_validation_df)),
    "domain_pass_rows": int(
        (domain_validation_df["status"] == "PASS").sum()
    ) if not domain_validation_df.empty else 0,
    "domain_review_rows": domain_review_rows,
    "overall_status": (
        "PASS"
        if range_review_rows == 0 and domain_review_rows == 0
        else "REVIEW_REQUIRED"
    ),
    "data_modified": False,
    "correction_applied": False,
    "notes": [
        "This section validates but does not modify synthetic datasets.",
        "Reference min/max checks are empirical comparisons, not proof of semantic validity.",
        "Domain rules are explicit checks and should be reviewed against dataset documentation.",
        "A REVIEW result must be investigated before claiming valid synthetic data.",
    ],
    "artifacts": {
        "range_validation": str(RANGE_REPORT_PATH),
        "domain_validation": str(DOMAIN_REPORT_PATH),
        "summary": str(SUMMARY_REPORT_PATH),
    },
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 6. Print final report
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 15 — FINAL VALIDATION SUMMARY")
print("=" * 100)

print(f"Datasets checked                 : {len(DATASET_IDS)}")
print(f"Numeric range checks             : {len(range_validation_df)}")
print(f"Range checks requiring review    : {range_review_rows}")
print(f"Domain rules checked             : {len(domain_validation_df)}")
print(f"Domain rules requiring review    : {domain_review_rows}")
print(f"Overall status                   : {metadata['overall_status']}")
print("\nSaved artifacts:")
print(f"  Range report    : {RANGE_REPORT_PATH}")
print(f"  Domain report   : {DOMAIN_REPORT_PATH}")
print(f"  Summary         : {SUMMARY_REPORT_PATH}")
print(f"  Metadata        : {METADATA_PATH}")

print("\nPer-dataset summary:")
display(range_validation_summary_df)

if range_review_rows or domain_review_rows:
    print(
        "\nWARNING: Validation issues remain. Do not label the synthetic data "
        "as fully range/domain valid."
    )
    print(
        "Investigate the generator output, transformed-feature ordering, and "
        "inverse-transformation logic before applying any correction."
    )
else:
    print(
        "\nAll configured range and domain checks passed. This does not establish "
        "distributional fidelity, downstream utility, or privacy."
    )

print("\nSECTION 15 EXECUTION COMPLETED.")


SECTION 15 — RANGE AND DOMAIN VALIDATION

----------------------------------------------------------------------------------------------------
VALIDATING SYNTHETIC DATASETS
----------------------------------------------------------------------------------------------------
adult_income: range PASS=0, REVIEW=6; domain PASS=0, REVIEW=5
bank_marketing: range PASS=0, REVIEW=7; domain PASS=1, REVIEW=5
diabetes_130us: range PASS=0, REVIEW=11; domain PASS=0, REVIEW=11

SECTION 15 — FINAL VALIDATION SUMMARY
Datasets checked                 : 3
Numeric range checks             : 24
Range checks requiring review    : 24
Domain rules checked             : 22
Domain rules requiring review    : 21
Overall status                   : REVIEW_REQUIRED

Saved artifacts:
  Range report    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/range_validation.csv
  Domain report   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/domain_validat

,dataset_id,reference_rows,synthetic_rows,numeric_columns_checked,range_pass_columns,range_review_columns,domain_rules_checked,domain_pass_rules,domain_review_rules,range_status,domain_status
0,adult_income,34189,34189,6,0,6,5,0,5,REVIEW,REVIEW
1,bank_marketing,31647,31647,7,0,7,6,1,5,REVIEW,REVIEW
2,diabetes_130us,71236,71236,11,0,11,11,0,11,REVIEW,REVIEW



Investigate the generator output, transformed-feature ordering, and inverse-transformation logic before applying any correction.

SECTION 15 EXECUTION COMPLETED.


In [79]:

# ==================================================================================================
# SECTION 16 — VALIDATE MISSINGNESS
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Compare missingness in synthetic data against the reference training data.
#   2. Validate all shared columns, including the restored target.
#   3. Detect missing columns and unexpected columns.
#   4. Save detailed, summary, and metadata artifacts.
#   5. Preserve the original synthetic data without modifying values.
#
# Required existing objects:
#   SYNTHETIC_DATAFRAMES_WITH_TARGET
#   REFERENCE_TRAIN
#   DATASET_IDS
#   NB13_VALIDATION_ROOT
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 16 — VALIDATE MISSINGNESS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)
NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

MISSINGNESS_REPORT_PATH = (
    NB13_VALIDATION_ROOT / "missingness_validation.csv"
)
MISSINGNESS_SUMMARY_PATH = (
    NB13_VALIDATION_ROOT / "missingness_validation_summary.csv"
)
MISSINGNESS_METADATA_PATH = (
    NB13_VALIDATION_ROOT / "missingness_validation_metadata.json"
)

# --------------------------------------------------------------------------------------------------
# 2. Missingness validation helper
# --------------------------------------------------------------------------------------------------

def validate_missingness(dataset, synthetic_df, reference_df):
    """Compare per-column missingness without changing either DataFrame."""

    rows = []

    reference_columns = set(reference_df.columns)
    synthetic_columns = set(synthetic_df.columns)

    missing_from_synthetic = sorted(reference_columns - synthetic_columns)
    unexpected_in_synthetic = sorted(synthetic_columns - reference_columns)

    # Evaluate all reference columns so absent synthetic columns are explicitly reported.
    for column in reference_df.columns:
        reference_series = reference_df[column]
        reference_rate = float(reference_series.isna().mean())
        reference_count = int(reference_series.isna().sum())

        if column in synthetic_df.columns:
            synthetic_series = synthetic_df[column]
            synthetic_rate = float(synthetic_series.isna().mean())
            synthetic_count = int(synthetic_series.isna().sum())
            rate_difference = synthetic_rate - reference_rate
            absolute_difference = abs(rate_difference)

            status = (
                "PASS"
                if np.isclose(synthetic_rate, reference_rate, atol=1e-12)
                else "REVIEW"
            )
            reason = (
                "MATCH"
                if status == "PASS"
                else "MISSINGNESS_RATE_DIFFERENCE"
            )
        else:
            synthetic_rate = np.nan
            synthetic_count = np.nan
            rate_difference = np.nan
            absolute_difference = np.nan
            status = "SCHEMA_MISMATCH"
            reason = "COLUMN_MISSING_FROM_SYNTHETIC_DATA"

        rows.append({
            "dataset_id": dataset,
            "column": column,
            "column_role": (
                "target"
                if str(column).lower() in {
                    "income", "y", "readmitted"
                }
                else "feature"
            ),
            "reference_missing_count": reference_count,
            "reference_missing_rate": reference_rate,
            "synthetic_missing_count": synthetic_count,
            "synthetic_missing_rate": synthetic_rate,
            "missingness_rate_difference": rate_difference,
            "absolute_missingness_rate_difference": absolute_difference,
            "status": status,
            "review_reason": reason,
        })

    # Explicitly report unexpected synthetic columns.
    for column in unexpected_in_synthetic:
        synthetic_series = synthetic_df[column]

        rows.append({
            "dataset_id": dataset,
            "column": column,
            "column_role": "unexpected",
            "reference_missing_count": np.nan,
            "reference_missing_rate": np.nan,
            "synthetic_missing_count": int(synthetic_series.isna().sum()),
            "synthetic_missing_rate": float(synthetic_series.isna().mean()),
            "missingness_rate_difference": np.nan,
            "absolute_missingness_rate_difference": np.nan,
            "status": "SCHEMA_MISMATCH",
            "review_reason": "UNEXPECTED_SYNTHETIC_COLUMN",
        })

    return rows


# --------------------------------------------------------------------------------------------------
# 3. Run validation for each dataset
# --------------------------------------------------------------------------------------------------

MISSINGNESS_ROWS = []
MISSINGNESS_SUMMARY_ROWS = []

for dataset in DATASET_IDS:
    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Reference training data missing for '{dataset}'.")

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic data with target missing for '{dataset}'. "
            "Run Section 13 first."
        )

    reference_df = REFERENCE_TRAIN[dataset]
    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(reference_df, pd.DataFrame):
        raise TypeError(f"REFERENCE_TRAIN['{dataset}'] is not a DataFrame.")

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(
            f"SYNTHETIC_DATAFRAMES_WITH_TARGET['{dataset}'] is not a DataFrame."
        )

    rows = validate_missingness(dataset, synthetic_df, reference_df)
    MISSINGNESS_ROWS.extend(rows)

    dataset_results = pd.DataFrame(rows)

    checked = dataset_results[
        dataset_results["status"].isin(["PASS", "REVIEW"])
    ]
    pass_count = int((checked["status"] == "PASS").sum())
    review_count = int((checked["status"] == "REVIEW").sum())
    schema_mismatch_count = int(
        (dataset_results["status"] == "SCHEMA_MISMATCH").sum()
    )

    common_columns = (
        set(reference_df.columns) & set(synthetic_df.columns)
    )

    MISSINGNESS_SUMMARY_ROWS.append({
        "dataset_id": dataset,
        "reference_rows": int(len(reference_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "reference_columns": int(len(reference_df.columns)),
        "synthetic_columns": int(len(synthetic_df.columns)),
        "common_columns": int(len(common_columns)),
        "columns_with_matching_missingness": pass_count,
        "columns_with_missingness_difference": review_count,
        "schema_mismatch_columns": schema_mismatch_count,
        "overall_status": (
            "PASS"
            if review_count == 0 and schema_mismatch_count == 0
            else "REVIEW_REQUIRED"
        ),
    })

    print(
        f"{dataset}: PASS={pass_count}, "
        f"missingness REVIEW={review_count}, "
        f"schema mismatches={schema_mismatch_count}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Save reports
# --------------------------------------------------------------------------------------------------

MISSINGNESS_DF = pd.DataFrame(MISSINGNESS_ROWS)
MISSINGNESS_SUMMARY_DF = pd.DataFrame(MISSINGNESS_SUMMARY_ROWS)

MISSINGNESS_DF.to_csv(MISSINGNESS_REPORT_PATH, index=False)
MISSINGNESS_SUMMARY_DF.to_csv(MISSINGNESS_SUMMARY_PATH, index=False)

review_count = int((MISSINGNESS_DF["status"] == "REVIEW").sum())
schema_mismatch_count = int(
    (MISSINGNESS_DF["status"] == "SCHEMA_MISMATCH").sum()
)

metadata = {
    "section": 16,
    "section_name": "Validate Missingness",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "datasets": DATASET_IDS,
    "validation_rows": int(len(MISSINGNESS_DF)),
    "pass_rows": int((MISSINGNESS_DF["status"] == "PASS").sum()),
    "review_rows": review_count,
    "schema_mismatch_rows": schema_mismatch_count,
    "overall_status": (
        "PASS"
        if review_count == 0 and schema_mismatch_count == 0
        else "REVIEW_REQUIRED"
    ),
    "data_modified": False,
    "missingness_tolerance": 1e-12,
    "artifacts": {
        "detail": str(MISSINGNESS_REPORT_PATH),
        "summary": str(MISSINGNESS_SUMMARY_PATH),
    },
    "interpretation": (
        "This section compares observed missingness rates. "
        "Matching rates do not establish that missingness mechanisms, "
        "joint patterns, or statistical fidelity have been preserved."
    ),
}

with open(MISSINGNESS_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 5. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 16 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked                  : {len(DATASET_IDS)}")
print(f"Detailed validation rows          : {len(MISSINGNESS_DF)}")
print(f"Matching missingness rows         : {metadata['pass_rows']}")
print(f"Missingness differences           : {review_count}")
print(f"Schema mismatch rows              : {schema_mismatch_count}")
print(f"Overall status                    : {metadata['overall_status']}")

print("\nSaved artifacts:")
print(f"  Detail  : {MISSINGNESS_REPORT_PATH}")
print(f"  Summary : {MISSINGNESS_SUMMARY_PATH}")
print(f"  Metadata: {MISSINGNESS_METADATA_PATH}")

print("\nPer-dataset summary:")
display(MISSINGNESS_SUMMARY_DF)

print("\nLargest missingness-rate differences:")
display(
    MISSINGNESS_DF[
        MISSINGNESS_DF["status"] == "REVIEW"
    ]
    .sort_values(
        "absolute_missingness_rate_difference",
        ascending=False,
        na_position="last",
    )
    .head(20)
)

print("\nSECTION 16 EXECUTION COMPLETED.")


SECTION 16 — VALIDATE MISSINGNESS
adult_income: PASS=12, missingness REVIEW=3, schema mismatches=1
bank_marketing: PASS=17, missingness REVIEW=0, schema mismatches=1
diabetes_130us: PASS=39, missingness REVIEW=9, schema mismatches=1

SECTION 16 — FINAL SUMMARY
Datasets checked                  : 3
Detailed validation rows          : 83
Matching missingness rows         : 68
Missingness differences           : 12
Schema mismatch rows              : 3
Overall status                    : REVIEW_REQUIRED

Saved artifacts:
  Detail  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/missingness_validation.csv
  Summary : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/missingness_validation_summary.csv
  Metadata: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/missingness_validation_metadata.json

Per-dataset summary:


,dataset_id,reference_rows,synthetic_rows,reference_columns,synthetic_columns,common_columns,columns_with_matching_missingness,columns_with_missingness_difference,schema_mismatch_columns,overall_status
0,adult_income,34189,34189,16,15,15,12,3,1,REVIEW_REQUIRED
1,bank_marketing,31647,31647,18,17,17,17,0,1,REVIEW_REQUIRED
2,diabetes_130us,71236,71236,49,48,48,39,9,1,REVIEW_REQUIRED



Largest missingness-rate differences:


,dataset_id,column,column_role,reference_missing_count,reference_missing_rate,synthetic_missing_count,synthetic_missing_rate,missingness_rate_difference,absolute_missingness_rate_difference,status,review_reason
38,diabetes_130us,weight,feature,69041,0.969187,0.0,0.0,-0.969187,0.969187,REVIEW,MISSINGNESS_RATE_DIFFERENCE
55,diabetes_130us,max_glu_serum,feature,67483,0.947316,0.0,0.0,-0.947316,0.947316,REVIEW,MISSINGNESS_RATE_DIFFERENCE
56,diabetes_130us,A1Cresult,feature,59423,0.834171,0.0,0.0,-0.834171,0.834171,REVIEW,MISSINGNESS_RATE_DIFFERENCE
44,diabetes_130us,medical_specialty,feature,34942,0.490510,0.0,0.0,-0.490510,0.490510,REVIEW,MISSINGNESS_RATE_DIFFERENCE
43,diabetes_130us,payer_code,feature,28245,0.396499,0.0,0.0,-0.396499,0.396499,REVIEW,MISSINGNESS_RATE_DIFFERENCE
7,adult_income,occupation,feature,1997,0.058411,0.0,0.0,-0.058411,0.058411,REVIEW,MISSINGNESS_RATE_DIFFERENCE
2,adult_income,workclass,feature,1992,0.058264,0.0,0.0,-0.058264,0.058264,REVIEW,MISSINGNESS_RATE_DIFFERENCE
35,diabetes_130us,race,feature,1563,0.021941,0.0,0.0,-0.021941,0.021941,REVIEW,MISSINGNESS_RATE_DIFFERENCE
14,adult_income,native_country,feature,610,0.017842,0.0,0.0,-0.017842,0.017842,REVIEW,MISSINGNESS_RATE_DIFFERENCE
53,diabetes_130us,diag_3,feature,1003,0.014080,0.0,0.0,-0.014080,0.014080,REVIEW,MISSINGNESS_RATE_DIFFERENCE



SECTION 16 EXECUTION COMPLETED.


In [81]:

# ==================================================================================================
# SECTION 17 — VALIDATE INVALID VALUES
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Detect non-finite numeric values.
#   2. Detect non-numeric values in reference-numeric columns.
#   3. Detect categorical values absent from the reference training data.
#   4. Detect fractional values in integer-valued numeric features.
#   5. Report missing/unexpected columns and save auditable results.
#
# Important:
#   - Uses target-restored synthetic data.
#   - Does not modify synthetic values.
#   - Range/domain checks are also performed separately in Section 15.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 17 — VALIDATE INVALID VALUES")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: "
        + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)
NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

INVALID_VALUES_PATH = (
    NB13_VALIDATION_ROOT / "invalid_values_validation.csv"
)
INVALID_SUMMARY_PATH = (
    NB13_VALIDATION_ROOT / "invalid_values_validation_summary.csv"
)
INVALID_METADATA_PATH = (
    NB13_VALIDATION_ROOT / "invalid_values_validation_metadata.json"
)

# Exclude these columns from feature/target value validation if present.
IDENTIFIER_COLUMNS = {
    "__original_row_id__",
    "original_row_id",
    "row_id",
    "record_id",
}

# Dataset-specific integer-valued features.
INTEGER_FEATURES = {
    "adult_income": {
        "age", "fnlwgt", "education_num", "capital_gain",
        "capital_loss", "hours_per_week",
    },
    "bank_marketing": {
        "age", "balance", "day", "duration",
        "campaign", "pdays", "previous",
    },
    "diabetes_130us": {
        "admission_type_id", "discharge_disposition_id",
        "admission_source_id", "time_in_hospital",
        "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient",
        "number_emergency", "number_inpatient",
        "number_diagnoses",
    },
}

# --------------------------------------------------------------------------------------------------
# 2. Validation function
# --------------------------------------------------------------------------------------------------

def validate_invalid_values(dataset, synthetic_df, reference_df):
    """Validate generated values against reference column types and categories."""

    records = []

    reference_columns = set(reference_df.columns)
    synthetic_columns = set(synthetic_df.columns)

    missing_columns = reference_columns - synthetic_columns
    unexpected_columns = synthetic_columns - reference_columns

    # Validate every reference column and report schema mismatches explicitly.
    for column in reference_df.columns:
        is_identifier = str(column).strip().lower() in IDENTIFIER_COLUMNS

        if column not in synthetic_df.columns:
            records.append({
                "dataset_id": dataset,
                "column": column,
                "column_role": "identifier" if is_identifier else "feature_or_target",
                "reference_dtype": str(reference_df[column].dtype),
                "synthetic_dtype": None,
                "synthetic_missing_count": np.nan,
                "non_finite_numeric_count": np.nan,
                "nonnumeric_in_numeric_column_count": np.nan,
                "unseen_category_count": np.nan,
                "fractional_integer_value_count": np.nan,
                "invalid_value_count": np.nan,
                "schema_status": "MISSING_COLUMN",
                "value_status": "NOT_CHECKED",
                "overall_status": "SCHEMA_MISMATCH",
                "review_reasons": "COLUMN_MISSING_FROM_SYNTHETIC_DATA",
            })
            continue

        source = reference_df[column]
        series = synthetic_df[column]

        non_finite = 0
        nonnumeric_count = 0
        unseen_categories = 0
        fractional_count = 0

        # Missing values are reported separately because missingness is assessed in Section 16.
        missing_count = int(series.isna().sum())

        # Determine numeric semantics from the reference column's dtype.
        reference_is_numeric = pd.api.types.is_numeric_dtype(source.dtype)
        reference_is_bool = pd.api.types.is_bool_dtype(source.dtype)

        if reference_is_numeric and not reference_is_bool:
            numeric = pd.to_numeric(series, errors="coerce")

            nonnumeric_mask = series.notna() & numeric.isna()
            nonnumeric_count = int(nonnumeric_mask.sum())

            numeric_nonmissing = numeric.dropna()
            non_finite = int(
                (~np.isfinite(numeric_nonmissing.to_numpy(dtype=float))).sum()
            )

            if column in INTEGER_FEATURES.get(dataset, set()):
                finite_values = numeric[
                    numeric.notna() & np.isfinite(numeric)
                ]
                fractional_count = int(
                    (
                        ~np.isclose(
                            finite_values.to_numpy(dtype=float),
                            np.round(finite_values.to_numpy(dtype=float)),
                            atol=1e-8,
                        )
                    ).sum()
                )

        # Validate categorical values against non-missing reference categories.
        # Numeric columns are not checked as categories.
        elif (
            pd.api.types.is_object_dtype(source.dtype)
            or pd.api.types.is_string_dtype(source.dtype)
            or isinstance(source.dtype, pd.CategoricalDtype)
        ):
            allowed = set(source.dropna().astype(str).unique())
            observed = series.dropna().astype(str)
            unseen_categories = int((~observed.isin(allowed)).sum())

        invalid_count = (
            non_finite
            + nonnumeric_count
            + unseen_categories
            + fractional_count
        )

        reasons = []
        if non_finite:
            reasons.append("NON_FINITE_NUMERIC_VALUES")
        if nonnumeric_count:
            reasons.append("NON_NUMERIC_VALUES_IN_NUMERIC_COLUMN")
        if unseen_categories:
            reasons.append("UNSEEN_CATEGORIES")
        if fractional_count:
            reasons.append("FRACTIONAL_INTEGER_VALUES")

        value_status = "PASS" if not reasons else "REVIEW"

        records.append({
            "dataset_id": dataset,
            "column": column,
            "column_role": "identifier" if is_identifier else "feature_or_target",
            "reference_dtype": str(source.dtype),
            "synthetic_dtype": str(series.dtype),
            "synthetic_missing_count": missing_count,
            "non_finite_numeric_count": non_finite,
            "nonnumeric_in_numeric_column_count": nonnumeric_count,
            "unseen_category_count": unseen_categories,
            "fractional_integer_value_count": fractional_count,
            "invalid_value_count": invalid_count,
            "schema_status": "PASS",
            "value_status": value_status,
            "overall_status": value_status,
            "review_reasons": ";".join(reasons) if reasons else "NONE",
        })

    # Report unexpected synthetic columns.
    for column in sorted(unexpected_columns):
        series = synthetic_df[column]
        records.append({
            "dataset_id": dataset,
            "column": column,
            "column_role": "unexpected",
            "reference_dtype": None,
            "synthetic_dtype": str(series.dtype),
            "synthetic_missing_count": int(series.isna().sum()),
            "non_finite_numeric_count": np.nan,
            "nonnumeric_in_numeric_column_count": np.nan,
            "unseen_category_count": np.nan,
            "fractional_integer_value_count": np.nan,
            "invalid_value_count": np.nan,
            "schema_status": "UNEXPECTED_COLUMN",
            "value_status": "NOT_CHECKED",
            "overall_status": "SCHEMA_MISMATCH",
            "review_reasons": "UNEXPECTED_SYNTHETIC_COLUMN",
        })

    return records


# --------------------------------------------------------------------------------------------------
# 3. Run validation
# --------------------------------------------------------------------------------------------------

INVALID_VALUE_ROWS = []
INVALID_SUMMARY_ROWS = []

for dataset in DATASET_IDS:
    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Reference data not found for '{dataset}'.")

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic data with target not found for '{dataset}'. "
            "Run Section 13 first."
        )

    reference_df = REFERENCE_TRAIN[dataset]
    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(reference_df, pd.DataFrame):
        raise TypeError(f"Reference data for '{dataset}' is not a DataFrame.")

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(f"Synthetic data for '{dataset}' is not a DataFrame.")

    records = validate_invalid_values(
        dataset=dataset,
        synthetic_df=synthetic_df,
        reference_df=reference_df,
    )
    INVALID_VALUE_ROWS.extend(records)

    dataset_df = pd.DataFrame(records)

    value_review_count = int(
        (dataset_df["value_status"] == "REVIEW").sum()
    )
    schema_mismatch_count = int(
        (dataset_df["schema_status"] != "PASS").sum()
    )
    pass_count = int(
        (dataset_df["overall_status"] == "PASS").sum()
    )

    INVALID_SUMMARY_ROWS.append({
        "dataset_id": dataset,
        "reference_rows": int(len(reference_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "reference_columns": int(len(reference_df.columns)),
        "synthetic_columns": int(len(synthetic_df.columns)),
        "columns_passed": pass_count,
        "columns_with_invalid_values": value_review_count,
        "schema_mismatch_columns": schema_mismatch_count,
        "overall_status": (
            "PASS"
            if value_review_count == 0 and schema_mismatch_count == 0
            else "REVIEW_REQUIRED"
        ),
    })

    print(
        f"{dataset}: PASS={pass_count}, "
        f"value REVIEW={value_review_count}, "
        f"schema mismatches={schema_mismatch_count}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Save detailed and summary reports
# --------------------------------------------------------------------------------------------------

INVALID_VALUES_DF = pd.DataFrame(INVALID_VALUE_ROWS)
INVALID_VALUES_SUMMARY_DF = pd.DataFrame(INVALID_SUMMARY_ROWS)

INVALID_VALUES_DF.to_csv(INVALID_VALUES_PATH, index=False)
INVALID_VALUES_SUMMARY_DF.to_csv(INVALID_SUMMARY_PATH, index=False)

value_review_count = int(
    (INVALID_VALUES_DF["value_status"] == "REVIEW").sum()
)
schema_mismatch_count = int(
    (INVALID_VALUES_DF["schema_status"] != "PASS").sum()
)

metadata = {
    "section": 17,
    "section_name": "Validate Invalid Values",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": DATASET_IDS,
    "detail_rows": int(len(INVALID_VALUES_DF)),
    "value_review_columns": value_review_count,
    "schema_mismatch_columns": schema_mismatch_count,
    "overall_status": (
        "PASS"
        if value_review_count == 0 and schema_mismatch_count == 0
        else "REVIEW_REQUIRED"
    ),
    "data_modified": False,
    "notes": [
        "Missingness rates are assessed separately in Section 16.",
        "Numeric range and semantic domain constraints are assessed in Section 15.",
        "PASS here does not establish distributional fidelity, utility, or privacy.",
    ],
    "artifacts": {
        "detail": str(INVALID_VALUES_PATH),
        "summary": str(INVALID_SUMMARY_PATH),
    },
}

with open(INVALID_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 5. Display final results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 17 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked               : {len(DATASET_IDS)}")
print(f"Detailed validation rows       : {len(INVALID_VALUES_DF)}")
print(f"Columns with invalid values    : {value_review_count}")
print(f"Schema mismatch columns       : {schema_mismatch_count}")
print(f"Overall status                 : {metadata['overall_status']}")

print("\nSaved artifacts:")
print(f"  Detail  : {INVALID_VALUES_PATH}")
print(f"  Summary : {INVALID_SUMMARY_PATH}")
print(f"  Metadata: {INVALID_METADATA_PATH}")

print("\nPer-dataset summary:")
display(INVALID_VALUES_SUMMARY_DF)

print("\nColumns requiring review:")
display(
    INVALID_VALUES_DF[
        INVALID_VALUES_DF["overall_status"] != "PASS"
    ].head(50)
)

print("\nSECTION 17 EXECUTION COMPLETED.")


SECTION 17 — VALIDATE INVALID VALUES
adult_income: PASS=15, value REVIEW=0, schema mismatches=1
bank_marketing: PASS=17, value REVIEW=0, schema mismatches=1
diabetes_130us: PASS=48, value REVIEW=0, schema mismatches=1

SECTION 17 — FINAL SUMMARY
Datasets checked               : 3
Detailed validation rows       : 83
Columns with invalid values    : 0
Schema mismatch columns       : 3
Overall status                 : REVIEW_REQUIRED

Saved artifacts:
  Detail  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/invalid_values_validation.csv
  Summary : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/invalid_values_validation_summary.csv
  Metadata: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/invalid_values_validation_metadata.json

Per-dataset summary:


,dataset_id,reference_rows,synthetic_rows,reference_columns,synthetic_columns,columns_passed,columns_with_invalid_values,schema_mismatch_columns,overall_status
0,adult_income,34189,34189,16,15,15,0,1,REVIEW_REQUIRED
1,bank_marketing,31647,31647,18,17,17,0,1,REVIEW_REQUIRED
2,diabetes_130us,71236,71236,49,48,48,0,1,REVIEW_REQUIRED



Columns requiring review:


,dataset_id,column,column_role,reference_dtype,synthetic_dtype,synthetic_missing_count,non_finite_numeric_count,nonnumeric_in_numeric_column_count,unseen_category_count,fractional_integer_value_count,invalid_value_count,schema_status,value_status,overall_status,review_reasons
0,adult_income,__original_row_id__,identifier,int64,None,NaN,NaN,NaN,NaN,NaN,NaN,MISSING_COLUMN,NOT_CHECKED,SCHEMA_MISMATCH,COLUMN_MISSING_FROM_SYNTHETIC_DATA
16,bank_marketing,__original_row_id__,identifier,int64,None,NaN,NaN,NaN,NaN,NaN,NaN,MISSING_COLUMN,NOT_CHECKED,SCHEMA_MISMATCH,COLUMN_MISSING_FROM_SYNTHETIC_DATA
34,diabetes_130us,__original_row_id__,identifier,int64,None,NaN,NaN,NaN,NaN,NaN,NaN,MISSING_COLUMN,NOT_CHECKED,SCHEMA_MISMATCH,COLUMN_MISSING_FROM_SYNTHETIC_DATA



SECTION 17 EXECUTION COMPLETED.


In [83]:

# ==================================================================================================
# SECTION 18 — DUPLICATE/EXACT-MATCH CHARACTERIZATION
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Measure duplicate rows within each synthetic dataset.
#   2. Measure synthetic rows that exactly match reference training records.
#   3. Exclude original row identifiers from comparisons.
#   4. Report the comparison scope and any schema differences.
#   5. Save summary, row-level match details, and metadata.
#
# Important:
#   Exact-match analysis is a diagnostic, not a complete privacy evaluation.
#   No synthetic or reference data is modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 18 — DUPLICATE/EXACT-MATCH CHARACTERIZATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)
NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = (
    NB13_VALIDATION_ROOT / "duplicate_exact_match_characterization.csv"
)
ROW_MATCH_PATH = (
    NB13_VALIDATION_ROOT / "duplicate_exact_match_row_details.csv"
)
METADATA_PATH = (
    NB13_VALIDATION_ROOT / "duplicate_exact_match_metadata.json"
)

IDENTIFIER_COLUMNS = {
    "__original_row_id__",
    "original_row_id",
    "row_id",
    "record_id",
}

# --------------------------------------------------------------------------------------------------
# 2. Normalize values consistently for exact row comparison
# --------------------------------------------------------------------------------------------------

def _normalized_frame(dataframe, columns):
    """
    Create a comparison-only copy with consistent string representation.
    Missing values are represented by a reserved token.
    """
    normalized = dataframe.loc[:, columns].copy()

    for column in columns:
        normalized[column] = (
            normalized[column]
            .astype("string")
            .fillna("<NA>")
            .astype(str)
        )

    return normalized


def _row_signatures(dataframe, columns):
    """
    Create row signatures for efficient candidate matching.
    Hashes are used to find candidates; exact normalized rows confirm matches.
    """
    normalized = _normalized_frame(dataframe, columns)
    hashes = pd.util.hash_pandas_object(normalized, index=False)
    return normalized, hashes


# --------------------------------------------------------------------------------------------------
# 3. Dataset-level characterization
# --------------------------------------------------------------------------------------------------

SUMMARY_ROWS = []
ROW_DETAIL_ROWS = []

for dataset in DATASET_IDS:
    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Reference training data missing for '{dataset}'.")

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic data with target missing for '{dataset}'. "
            "Run Section 13 first."
        )

    reference = REFERENCE_TRAIN[dataset]
    synthetic = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(reference, pd.DataFrame):
        raise TypeError(f"Reference data for '{dataset}' is not a DataFrame.")

    if not isinstance(synthetic, pd.DataFrame):
        raise TypeError(f"Synthetic data for '{dataset}' is not a DataFrame.")

    # Exclude row identifiers, which should not be copied into synthetic records.
    reference_columns = [
        col for col in reference.columns
        if str(col).strip().lower() not in IDENTIFIER_COLUMNS
    ]
    synthetic_columns = [
        col for col in synthetic.columns
        if str(col).strip().lower() not in IDENTIFIER_COLUMNS
    ]

    shared_columns = [
        col for col in reference_columns if col in synthetic_columns
    ]
    missing_synthetic_columns = [
        col for col in reference_columns if col not in synthetic_columns
    ]
    unexpected_synthetic_columns = [
        col for col in synthetic_columns if col not in reference_columns
    ]

    # Within-synthetic duplicate detection on all substantive synthetic columns.
    duplicate_mask = synthetic[synthetic_columns].duplicated(keep="first")
    duplicate_count = int(duplicate_mask.sum())
    duplicate_rate = (
        float(duplicate_count / len(synthetic))
        if len(synthetic) else np.nan
    )

    # A full-record comparison is possible only when substantive schemas match.
    schemas_match = (
        set(reference_columns) == set(synthetic_columns)
        and len(reference_columns) == len(synthetic_columns)
    )

    comparison_columns = (
        reference_columns.copy()
        if schemas_match
        else shared_columns.copy()
    )

    if not comparison_columns:
        SUMMARY_ROWS.append({
            "dataset_id": dataset,
            "reference_rows": len(reference),
            "synthetic_rows": len(synthetic),
            "duplicate_rows": duplicate_count,
            "duplicate_rate": duplicate_rate,
            "real_exact_match_count": np.nan,
            "real_exact_match_rate": np.nan,
            "comparison_columns_count": 0,
            "comparison_scope": "NO_SHARED_COLUMNS",
            "schema_match": schemas_match,
            "missing_synthetic_columns": json.dumps(missing_synthetic_columns),
            "unexpected_synthetic_columns": json.dumps(unexpected_synthetic_columns),
            "exact_match_status": "NOT_COMPUTED",
        })
        continue

    # Deduplicate the reference before matching, so a reference record is represented once.
    reference_compare = reference[comparison_columns].drop_duplicates().reset_index(drop=True)
    synthetic_compare = synthetic[comparison_columns].reset_index(drop=True)

    reference_normalized, reference_hashes = _row_signatures(
        reference_compare, comparison_columns
    )
    synthetic_normalized, synthetic_hashes = _row_signatures(
        synthetic_compare, comparison_columns
    )

    # Group reference rows by hash. Confirm candidate matches using full normalized row equality.
    reference_hash_groups = {}
    for ref_idx, hash_value in enumerate(reference_hashes.tolist()):
        reference_hash_groups.setdefault(hash_value, []).append(ref_idx)

    exact_match_mask = np.zeros(len(synthetic_compare), dtype=bool)

    for syn_idx, hash_value in enumerate(synthetic_hashes.tolist()):
        candidate_indices = reference_hash_groups.get(hash_value, [])

        if not candidate_indices:
            continue

        synthetic_row = synthetic_normalized.iloc[syn_idx]

        for ref_idx in candidate_indices:
            reference_row = reference_normalized.iloc[ref_idx]

            if synthetic_row.equals(reference_row):
                exact_match_mask[syn_idx] = True
                break

    exact_match_count = int(exact_match_mask.sum())
    exact_match_rate = (
        float(exact_match_count / len(synthetic_compare))
        if len(synthetic_compare) else np.nan
    )

    comparison_scope = (
        "FULL_RECORD"
        if schemas_match
        else "SHARED_COLUMNS_ONLY"
    )

    # Save details for each synthetic row that exactly matches a reference row.
    for syn_idx in np.flatnonzero(exact_match_mask):
        ROW_DETAIL_ROWS.append({
            "dataset_id": dataset,
            "synthetic_row_position": int(syn_idx),
            "match_type": "EXACT_MATCH",
            "comparison_scope": comparison_scope,
            "comparison_columns_count": len(comparison_columns),
        })

    SUMMARY_ROWS.append({
        "dataset_id": dataset,
        "reference_rows": int(len(reference)),
        "synthetic_rows": int(len(synthetic)),
        "duplicate_rows": duplicate_count,
        "duplicate_rate": duplicate_rate,
        "real_exact_match_count": exact_match_count,
        "real_exact_match_rate": exact_match_rate,
        "comparison_columns_count": len(comparison_columns),
        "comparison_scope": comparison_scope,
        "schema_match": schemas_match,
        "missing_synthetic_columns": json.dumps(missing_synthetic_columns),
        "unexpected_synthetic_columns": json.dumps(unexpected_synthetic_columns),
        "exact_match_status": (
            "COMPUTED_FULL_RECORD"
            if schemas_match
            else "COMPUTED_SHARED_COLUMNS_ONLY"
        ),
    })

    print(
        f"{dataset}: duplicates={duplicate_count}, "
        f"exact matches={exact_match_count}, "
        f"scope={comparison_scope}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Save reports and metadata
# --------------------------------------------------------------------------------------------------

DUPLICATE_DF = pd.DataFrame(SUMMARY_ROWS)
DUPLICATE_ROW_DETAILS_DF = pd.DataFrame(
    ROW_DETAIL_ROWS,
    columns=[
        "dataset_id",
        "synthetic_row_position",
        "match_type",
        "comparison_scope",
        "comparison_columns_count",
    ],
)

DUPLICATE_DF.to_csv(SUMMARY_PATH, index=False)
DUPLICATE_ROW_DETAILS_DF.to_csv(ROW_MATCH_PATH, index=False)

metadata = {
    "section": 18,
    "section_name": "Duplicate/Exact-Match Characterization",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": DATASET_IDS,
    "within_synthetic_duplicate_rows": int(
        DUPLICATE_DF["duplicate_rows"].sum()
    ) if not DUPLICATE_DF.empty else 0,
    "exact_match_rows": int(
        DUPLICATE_DF["real_exact_match_count"].fillna(0).sum()
    ) if not DUPLICATE_DF.empty else 0,
    "data_modified": False,
    "hash_collisions_verified_with_row_comparison": True,
    "interpretation": (
        "Exact-match rates describe record-level equality under the reported "
        "comparison scope. They are not a complete privacy guarantee. "
        "When schemas differ, matches are computed only over shared columns."
    ),
    "artifacts": {
        "summary": str(SUMMARY_PATH),
        "matching_rows": str(ROW_MATCH_PATH),
    },
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 5. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 18 — FINAL SUMMARY")
print("=" * 100)

display(DUPLICATE_DF)

print("\nSaved artifacts:")
print(f"  Summary       : {SUMMARY_PATH}")
print(f"  Matched rows  : {ROW_MATCH_PATH}")
print(f"  Metadata      : {METADATA_PATH}")

print(
    "\nNOTE: Zero exact matches does not establish privacy. "
    "Membership-inference, attribute-inference, and other privacy evaluations "
    "remain separate research requirements."
)

print("\nSECTION 18 EXECUTION COMPLETED.")


SECTION 18 — DUPLICATE/EXACT-MATCH CHARACTERIZATION
adult_income: duplicates=0, exact matches=0, scope=FULL_RECORD
bank_marketing: duplicates=0, exact matches=0, scope=FULL_RECORD
diabetes_130us: duplicates=0, exact matches=0, scope=FULL_RECORD

SECTION 18 — FINAL SUMMARY


,dataset_id,reference_rows,synthetic_rows,duplicate_rows,duplicate_rate,real_exact_match_count,real_exact_match_rate,comparison_columns_count,comparison_scope,schema_match,missing_synthetic_columns,unexpected_synthetic_columns,exact_match_status
0,adult_income,34189,34189,0,0.0,0,0.0,15,FULL_RECORD,True,[],[],COMPUTED_FULL_RECORD
1,bank_marketing,31647,31647,0,0.0,0,0.0,17,FULL_RECORD,True,[],[],COMPUTED_FULL_RECORD
2,diabetes_130us,71236,71236,0,0.0,0,0.0,48,FULL_RECORD,True,[],[],COMPUTED_FULL_RECORD



Saved artifacts:
  Summary       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/duplicate_exact_match_characterization.csv
  Matched rows  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/duplicate_exact_match_row_details.csv
  Metadata      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/duplicate_exact_match_metadata.json

NOTE: Zero exact matches does not establish privacy. Membership-inference, attribute-inference, and other privacy evaluations remain separate research requirements.

SECTION 18 EXECUTION COMPLETED.


In [85]:

# ==================================================================================================
# SECTION 19 — VALIDATE SAMPLE SIZE
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Validate final synthetic row counts against the expected sample size.
#   2. Use SAMPLE_SIZE_OVERRIDE when explicitly configured.
#   3. Otherwise, use the corresponding reference training-data row count.
#   4. Save detailed results, summary, and metadata.
#   5. Never resize or modify the synthetic data silently.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 19 — VALIDATE SAMPLE SIZE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "SAMPLE_SIZE_OVERRIDE",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)
NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE_PATH = (
    NB13_VALIDATION_ROOT / "sample_size_validation.csv"
)
SAMPLE_SIZE_SUMMARY_PATH = (
    NB13_VALIDATION_ROOT / "sample_size_validation_summary.csv"
)
SAMPLE_SIZE_METADATA_PATH = (
    NB13_VALIDATION_ROOT / "sample_size_validation_metadata.json"
)

# --------------------------------------------------------------------------------------------------
# 2. Validate the sample-size override
# --------------------------------------------------------------------------------------------------

if SAMPLE_SIZE_OVERRIDE is not None:
    if (
        isinstance(SAMPLE_SIZE_OVERRIDE, (bool, np.bool_))
        or not isinstance(SAMPLE_SIZE_OVERRIDE, (int, np.integer))
        or int(SAMPLE_SIZE_OVERRIDE) <= 0
    ):
        raise ValueError(
            "SAMPLE_SIZE_OVERRIDE must be None or a positive integer."
        )

# --------------------------------------------------------------------------------------------------
# 3. Resolve expected sample size
# --------------------------------------------------------------------------------------------------

def expected_sample_count(dataset):
    """Resolve the expected synthetic row count for a dataset."""

    if SAMPLE_SIZE_OVERRIDE is not None:
        return int(SAMPLE_SIZE_OVERRIDE), "SAMPLE_SIZE_OVERRIDE"

    if dataset not in REFERENCE_TRAIN:
        raise RuntimeError(
            f"{dataset}: reference training data is unavailable; "
            "cannot determine expected sample size."
        )

    reference = REFERENCE_TRAIN[dataset]

    if not isinstance(reference, pd.DataFrame):
        raise TypeError(
            f"REFERENCE_TRAIN['{dataset}'] must be a pandas DataFrame."
        )

    expected = len(reference)

    if expected <= 0:
        raise RuntimeError(
            f"{dataset}: reference training dataset has no rows."
        )

    return int(expected), "REFERENCE_TRAIN_ROW_COUNT"

# --------------------------------------------------------------------------------------------------
# 4. Validate each final synthetic dataset
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_ROWS = []

for dataset in DATASET_IDS:
    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"{dataset}: final synthetic data with target was not found. "
            "Run Section 13 first."
        )

    dataframe = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"SYNTHETIC_DATAFRAMES_WITH_TARGET['{dataset}'] "
            "must be a pandas DataFrame."
        )

    expected, expected_source = expected_sample_count(dataset)
    actual = len(dataframe)
    difference = actual - expected

    SAMPLE_SIZE_ROWS.append({
        "dataset_id": dataset,
        "expected_rows": expected,
        "actual_rows": int(actual),
        "row_count_difference": int(difference),
        "expected_count_source": expected_source,
        "sample_size_status": "PASS" if actual == expected else "FAIL",
    })

SAMPLE_SIZE_DF = pd.DataFrame(SAMPLE_SIZE_ROWS)

# --------------------------------------------------------------------------------------------------
# 5. Save detailed and summary reports
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_DF.to_csv(SAMPLE_SIZE_PATH, index=False)

pass_count = int((SAMPLE_SIZE_DF["sample_size_status"] == "PASS").sum())
fail_count = int((SAMPLE_SIZE_DF["sample_size_status"] == "FAIL").sum())

SAMPLE_SIZE_SUMMARY_DF = pd.DataFrame([{
    "datasets_checked": int(len(SAMPLE_SIZE_DF)),
    "datasets_passed": pass_count,
    "datasets_failed": fail_count,
    "total_expected_rows": int(SAMPLE_SIZE_DF["expected_rows"].sum()),
    "total_actual_rows": int(SAMPLE_SIZE_DF["actual_rows"].sum()),
    "overall_status": "PASS" if fail_count == 0 else "FAIL",
}])

SAMPLE_SIZE_SUMMARY_DF.to_csv(
    SAMPLE_SIZE_SUMMARY_PATH,
    index=False,
)

metadata = {
    "section": 19,
    "section_name": "Validate Sample Size",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": DATASET_IDS,
    "sample_size_override": (
        int(SAMPLE_SIZE_OVERRIDE)
        if SAMPLE_SIZE_OVERRIDE is not None else None
    ),
    "datasets_checked": int(len(SAMPLE_SIZE_DF)),
    "datasets_passed": pass_count,
    "datasets_failed": fail_count,
    "overall_status": "PASS" if fail_count == 0 else "FAIL",
    "data_modified": False,
    "artifacts": {
        "detail": str(SAMPLE_SIZE_PATH),
        "summary": str(SAMPLE_SIZE_SUMMARY_PATH),
    },
}

with open(SAMPLE_SIZE_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 6. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 19 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked     : {len(SAMPLE_SIZE_DF)}")
print(f"Datasets passed      : {pass_count}")
print(f"Datasets failed      : {fail_count}")
print(f"Overall status       : {metadata['overall_status']}")

print("\nPer-dataset results:")
display(SAMPLE_SIZE_DF)

print("\nSummary:")
display(SAMPLE_SIZE_SUMMARY_DF)

print("\nSaved artifacts:")
print(f"  Detail   : {SAMPLE_SIZE_PATH}")
print(f"  Summary  : {SAMPLE_SIZE_SUMMARY_PATH}")
print(f"  Metadata : {SAMPLE_SIZE_METADATA_PATH}")

if fail_count:
    print(
        "\nWARNING: One or more datasets have incorrect sample sizes. "
        "Investigate the generation configuration before proceeding."
    )
else:
    print(
        "\nAll final synthetic datasets match their expected row counts. "
        "This validates sample size only, not data quality or privacy."
    )

print("\nSECTION 19 EXECUTION COMPLETED.")


SECTION 19 — VALIDATE SAMPLE SIZE

SECTION 19 — FINAL SUMMARY
Datasets checked     : 3
Datasets passed      : 3
Datasets failed      : 0
Overall status       : PASS

Per-dataset results:


,dataset_id,expected_rows,actual_rows,row_count_difference,expected_count_source,sample_size_status
0,adult_income,34189,34189,0,REFERENCE_TRAIN_ROW_COUNT,PASS
1,bank_marketing,31647,31647,0,REFERENCE_TRAIN_ROW_COUNT,PASS
2,diabetes_130us,71236,71236,0,REFERENCE_TRAIN_ROW_COUNT,PASS



Summary:


,datasets_checked,datasets_passed,datasets_failed,total_expected_rows,total_actual_rows,overall_status
0,3,3,0,137072,137072,PASS



Saved artifacts:
  Detail   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/sample_size_validation.csv
  Summary  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/sample_size_validation_summary.csv
  Metadata : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/sample_size_validation_metadata.json

All final synthetic datasets match their expected row counts. This validates sample size only, not data quality or privacy.

SECTION 19 EXECUTION COMPLETED.


In [88]:

# ==================================================================================================
# SECTION 20 — RECORD GENERATION RUNTIME
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 20 — RECORD GENERATION RUNTIME")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "DATASET_IDS",
    "GENERATION_RUNTIME",
    "NB13_METADATA_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
    )

if not isinstance(GENERATION_RUNTIME, dict):
    raise TypeError("GENERATION_RUNTIME must be a dictionary.")

NB13_METADATA_ROOT = Path(NB13_METADATA_ROOT)
NB13_METADATA_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_REPORT_PATH = (
    NB13_METADATA_ROOT / "generation_runtime_summary.csv"
)
RUNTIME_SUMMARY_PATH = (
    NB13_METADATA_ROOT / "generation_runtime_overall_summary.csv"
)
RUNTIME_METADATA_PATH = (
    NB13_METADATA_ROOT / "generation_runtime_metadata.json"
)

# --------------------------------------------------------------------------------------------------
# 2. Normalize runtime records
# --------------------------------------------------------------------------------------------------
# Supported key formats:
#   "adult_income"             -> one runtime measurement
#   ("adult_income", 1)        -> dataset and repetition
#
# If several repetitions exist, report their mean, minimum, maximum, and count.
# Do not treat repeated measurements as independent datasets.

runtime_by_dataset = {dataset: [] for dataset in DATASET_IDS}
unrecognized_runtime_keys = []

for key, raw_seconds in GENERATION_RUNTIME.items():

    if isinstance(key, tuple) and len(key) >= 1:
        dataset = key[0]
        repetition = key[1] if len(key) >= 2 else None
    else:
        dataset = key
        repetition = None

    if dataset not in runtime_by_dataset:
        unrecognized_runtime_keys.append(str(key))
        continue

    try:
        seconds = float(raw_seconds)
    except (TypeError, ValueError):
        continue

    if not np.isfinite(seconds) or seconds < 0:
        continue

    runtime_by_dataset[dataset].append({
        "repetition": repetition,
        "seconds": seconds,
        "original_key": str(key),
    })

# --------------------------------------------------------------------------------------------------
# 3. Build per-dataset runtime records
# --------------------------------------------------------------------------------------------------

RUNTIME_ROWS = []

for dataset in DATASET_IDS:

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic dataset '{dataset}' not found."
        )

    dataframe = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]
    measurements = runtime_by_dataset.get(dataset, [])

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"Synthetic dataset '{dataset}' must be a pandas DataFrame."
        )

    seconds_values = [item["seconds"] for item in measurements]
    generated_rows = int(len(dataframe))

    if seconds_values:
        # The first measurement is the recorded runtime for the single-run case.
        # With multiple repetitions, mean/min/max are reported transparently.
        mean_seconds = float(np.mean(seconds_values))
        min_seconds = float(np.min(seconds_values))
        max_seconds = float(np.max(seconds_values))

        runtime_status = "RECORDED"
        seconds_per_1000 = (
            mean_seconds / generated_rows * 1000
            if generated_rows > 0
            else np.nan
        )
    else:
        mean_seconds = np.nan
        min_seconds = np.nan
        max_seconds = np.nan
        seconds_per_1000 = np.nan
        runtime_status = "MISSING_OR_INVALID"

    RUNTIME_ROWS.append({
        "dataset_id": dataset,
        "generation_runtime_seconds": mean_seconds,
        "runtime_min_seconds": min_seconds,
        "runtime_max_seconds": max_seconds,
        "runtime_measurement_count": len(seconds_values),
        "generated_rows": generated_rows,
        "seconds_per_1000_rows": seconds_per_1000,
        "runtime_status": runtime_status,
        "runtime_source": (
            "GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY"
            if seconds_values
            else "NO_VALID_RECORDED_MEASUREMENT"
        ),
    })

RUNTIME_DF = pd.DataFrame(RUNTIME_ROWS)

# --------------------------------------------------------------------------------------------------
# 4. Save detailed report
# --------------------------------------------------------------------------------------------------

RUNTIME_DF.to_csv(RUNTIME_REPORT_PATH, index=False)

recorded = RUNTIME_DF[
    RUNTIME_DF["runtime_status"] == "RECORDED"
]
missing_count = int(
    (RUNTIME_DF["runtime_status"] != "RECORDED").sum()
)

total_recorded_runtime = (
    float(recorded["generation_runtime_seconds"].sum())
    if not recorded.empty
    else np.nan
)

RUNTIME_SUMMARY_DF = pd.DataFrame([{
    "datasets_checked": len(RUNTIME_DF),
    "datasets_with_recorded_runtime": len(recorded),
    "datasets_with_missing_runtime": missing_count,
    "total_generated_rows": int(RUNTIME_DF["generated_rows"].sum()),
    "sum_of_dataset_mean_runtimes_seconds": total_recorded_runtime,
    "overall_status": (
        "PASS" if missing_count == 0 else "REVIEW_REQUIRED"
    ),
}])

RUNTIME_SUMMARY_DF.to_csv(
    RUNTIME_SUMMARY_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 5. Save metadata
# --------------------------------------------------------------------------------------------------

runtime_metadata = {
    "section": 20,
    "section_name": "Record Generation Runtime",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": list(DATASET_IDS),
    "runtime_key_format": "dataset or (dataset, repetition)",
    "datasets_with_recorded_runtime": int(len(recorded)),
    "datasets_with_missing_runtime": missing_count,
    "overall_status": (
        "PASS" if missing_count == 0 else "REVIEW_REQUIRED"
    ),
    "unrecognized_runtime_keys": unrecognized_runtime_keys,
    "data_modified": False,
    "artifacts": {
        "detail": str(RUNTIME_REPORT_PATH),
        "summary": str(RUNTIME_SUMMARY_PATH),
    },
}

with open(RUNTIME_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(runtime_metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 6. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 20 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked             : {len(RUNTIME_DF)}")
print(f"Runtime measurements present : {len(recorded)}")
print(f"Runtime measurements missing : {missing_count}")
print(f"Overall status               : {runtime_metadata['overall_status']}")

print("\nPer-dataset runtime:")
display(RUNTIME_DF)

print("\nOverall summary:")
display(RUNTIME_SUMMARY_DF)

print("\nSaved artifacts:")
print(f"  Detail   : {RUNTIME_REPORT_PATH}")
print(f"  Summary  : {RUNTIME_SUMMARY_PATH}")
print(f"  Metadata : {RUNTIME_METADATA_PATH}")

if missing_count:
    print("\nWARNING: Some runtime measurements remain missing.")
else:
    print(
        "\nAll datasets have recorded generation runtimes. "
        "These measurements were retrieved from the existing runtime dictionary."
    )

print("\nSECTION 20 EXECUTION COMPLETED.")


SECTION 20 — RECORD GENERATION RUNTIME

SECTION 20 — FINAL SUMMARY
Datasets checked             : 3
Runtime measurements present : 3
Runtime measurements missing : 0
Overall status               : PASS

Per-dataset runtime:


,dataset_id,generation_runtime_seconds,runtime_min_seconds,runtime_max_seconds,runtime_measurement_count,generated_rows,seconds_per_1000_rows,runtime_status,runtime_source
0,adult_income,6.379362,6.379362,6.379362,1,34189,0.186591,RECORDED,GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY
1,bank_marketing,6.404864,6.404864,6.404864,1,31647,0.202385,RECORDED,GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY
2,diabetes_130us,61.684650,61.684650,61.684650,1,71236,0.865920,RECORDED,GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY



Overall summary:


,datasets_checked,datasets_with_recorded_runtime,datasets_with_missing_runtime,total_generated_rows,sum_of_dataset_mean_runtimes_seconds,overall_status
0,3,3,0,137072,74.468876,PASS



Saved artifacts:
  Detail   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/generation_runtime_summary.csv
  Summary  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/generation_runtime_overall_summary.csv
  Metadata : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/generation_runtime_metadata.json

All datasets have recorded generation runtimes. These measurements were retrieved from the existing runtime dictionary.

SECTION 20 EXECUTION COMPLETED.


In [92]:

# =============================================================================
# SECTION 21 — MODEL ARTIFACT FINGERPRINTS
# =============================================================================

import hashlib
import json
from pathlib import Path
import pandas as pd

MODEL_FINGERPRINT_ROWS = []

for dataset, repetitions in MODEL_ARTIFACTS.items():
    if isinstance(repetitions, dict):
        items = repetitions.items()
    else:
        items = [("1", repetitions)]

    for repetition, model_value in items:
        model_path = None
        model_hash = None
        status = "NOT_FOUND"

        # Model metadata may contain a path or a nested dictionary.
        candidates = (
            list(model_value.values())
            if isinstance(model_value, dict)
            else [model_value]
        )

        for candidate in candidates:
            if not isinstance(candidate, (str, Path)):
                continue

            candidate_path = Path(candidate).expanduser()

            if candidate_path.is_file():
                model_path = candidate_path.resolve()
                break

        if model_path is not None:
            try:
                digest = hashlib.sha256()

                with model_path.open("rb") as handle:
                    for chunk in iter(
                        lambda: handle.read(1024 * 1024), b""
                    ):
                        digest.update(chunk)

                model_hash = digest.hexdigest()
                status = "HASHED"

            except OSError as exc:
                status = f"HASH_ERROR: {type(exc).__name__}"

        MODEL_FINGERPRINT_ROWS.append({
            "dataset": dataset,
            "repetition": str(repetition),
            "model_path": str(model_path) if model_path else None,
            "model_sha256": model_hash,
            "model_fingerprint_status": status,
        })

MODEL_FINGERPRINT_DF = pd.DataFrame(MODEL_FINGERPRINT_ROWS)

model_fingerprint_path = (
    Path(NB13_METADATA_ROOT) / "model_fingerprints.csv"
)
model_fingerprint_path.parent.mkdir(parents=True, exist_ok=True)
MODEL_FINGERPRINT_DF.to_csv(model_fingerprint_path, index=False)

print("=" * 100)
print("SECTION 21 — MODEL FINGERPRINT SUMMARY")
print("=" * 100)
print("Model entries checked :", len(MODEL_FINGERPRINT_DF))
print(
    "Model files hashed    :",
    int((MODEL_FINGERPRINT_DF["model_fingerprint_status"] == "HASHED").sum()),
)
print(
    "Model files unresolved:",
    int((MODEL_FINGERPRINT_DF["model_fingerprint_status"] != "HASHED").sum()),
)
print("Saved artifact        :", model_fingerprint_path)

display(MODEL_FINGERPRINT_DF)


SECTION 21 — MODEL FINGERPRINT SUMMARY
Model entries checked : 15
Model files hashed    : 15
Model files unresolved: 0
Saved artifact        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/model_fingerprints.csv


,dataset,repetition,model_path,model_sha256,model_fingerprint_status
0,adult_income,1,/content/drive/MyDrive/SPP_GAN_Research/result...,3147dc0dd607f0b242d35daed9e4c7a17830f9589fe29b...,HASHED
1,adult_income,2,/content/drive/MyDrive/SPP_GAN_Research/result...,60e8d6409ddd8f5f2e9a82cd758829a736e46c4921913e...,HASHED
2,adult_income,3,/content/drive/MyDrive/SPP_GAN_Research/result...,48e62c40e789f84822f4c26b75137464996fbff3274b8e...,HASHED
3,adult_income,4,/content/drive/MyDrive/SPP_GAN_Research/result...,183431ff4d80d2d1228f9adad8470986db81c1d5b664d3...,HASHED
4,adult_income,5,/content/drive/MyDrive/SPP_GAN_Research/result...,aa1828fd4b41c1450c7c447f1c4206c1a95213bf9fe0fa...,HASHED
5,bank_marketing,1,/content/drive/MyDrive/SPP_GAN_Research/result...,5bb3cc6fcea00cb21f417257963552a16c5e7faaa149ba...,HASHED
6,bank_marketing,2,/content/drive/MyDrive/SPP_GAN_Research/result...,e9055c31fab00a8be54c818a8cde714572680908ae9c75...,HASHED
7,bank_marketing,3,/content/drive/MyDrive/SPP_GAN_Research/result...,797dd47e7c97106cce5d02bc52da26bbc9408567a37877...,HASHED
8,bank_marketing,4,/content/drive/MyDrive/SPP_GAN_Research/result...,fed615ef19110561dddae97a6d67d63950dcb67adece08...,HASHED
9,bank_marketing,5,/content/drive/MyDrive/SPP_GAN_Research/result...,9ef74c66d380fbbbb5e33c900c00e23f74dcd7c76e9005...,HASHED


In [93]:

# =============================================================================
# SECTION 22 — SAVE SYNTHETIC DATASETS
# =============================================================================

from pathlib import Path
import gc
import json
import pandas as pd

print("=" * 100)
print("SECTION 22 — SAVE SYNTHETIC DATASETS")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Select final synthetic datasets
# -----------------------------------------------------------------------------

if (
    "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals()
    or not SYNTHETIC_DATAFRAMES_WITH_TARGET
):
    raise RuntimeError(
        "Final target-restored synthetic datasets are unavailable. "
        "Run Section 13 before saving."
    )

FINAL_SYNTHETIC_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

NB13_SYNTHETIC_ROOT = Path(NB13_SYNTHETIC_ROOT)
NB13_SYNTHETIC_ROOT.mkdir(parents=True, exist_ok=True)

SAVED_DATASET_PATHS = {}
SAVE_VALIDATION_ROWS = []

# -----------------------------------------------------------------------------
# 2. Save and verify each final dataset
# -----------------------------------------------------------------------------

for dataset, dataframe in FINAL_SYNTHETIC_DATAFRAMES.items():

    csv_path = (
        NB13_SYNTHETIC_ROOT
        / f"{dataset}_sppgan_synthetic.csv"
    )

    dataframe.to_csv(csv_path, index=False)

    if not csv_path.is_file() or csv_path.stat().st_size == 0:
        raise RuntimeError(
            f"{dataset}: CSV output is missing or empty."
        )

    # Reload for independent structural verification.
    reloaded = pd.read_csv(csv_path, low_memory=False)

    row_match = len(reloaded) == len(dataframe)
    column_match = list(reloaded.columns) == list(dataframe.columns)

    if not row_match:
        raise RuntimeError(
            f"{dataset}: saved CSV row count mismatch."
        )

    if not column_match:
        raise RuntimeError(
            f"{dataset}: saved CSV column mismatch."
        )

    # Confirm the target column is present.
    target_column = (
        DATASET_CONFIG[dataset].get("target")
        if "DATASET_CONFIG" in globals()
        and isinstance(DATASET_CONFIG.get(dataset), dict)
        else None
    )

    if target_column is None:
        target_column = {
            "adult_income": "income",
            "bank_marketing": "y",
            "diabetes_130us": "readmitted",
        }.get(dataset)

    target_present = (
        target_column in reloaded.columns
        if target_column is not None
        else False
    )

    if not target_present:
        raise RuntimeError(
            f"{dataset}: expected target column "
            f"{target_column!r} is absent from the saved dataset."
        )

    # Optional Parquet export.
    parquet_path = None
    parquet_status = "DISABLED"

    if SAVE_PARQUET:
        parquet_path = (
            NB13_SYNTHETIC_ROOT
            / f"{dataset}_sppgan_synthetic.parquet"
        )

        try:
            dataframe.to_parquet(parquet_path, index=False)

            if not parquet_path.is_file() or parquet_path.stat().st_size == 0:
                raise RuntimeError("Parquet file is missing or empty.")

            parquet_status = "SAVED"

        except Exception as exc:
            parquet_status = f"FAILED: {type(exc).__name__}"
            print(
                f"WARNING: Parquet not saved for {dataset}: {exc}"
            )

    SAVED_DATASET_PATHS[dataset] = csv_path

    SAVE_VALIDATION_ROWS.append({
        "dataset": dataset,
        "rows": len(reloaded),
        "columns": len(reloaded.columns),
        "target_column": target_column,
        "target_present": target_present,
        "row_count_match": row_match,
        "column_match": column_match,
        "csv_size_bytes": csv_path.stat().st_size,
        "csv_path": str(csv_path),
        "parquet_status": parquet_status,
        "save_status": "PASS",
    })

    print(
        f"PASS | {dataset}: {reloaded.shape} "
        f"| target={target_column} "
        f"| CSV={csv_path}"
    )

    del reloaded
    gc.collect()

# -----------------------------------------------------------------------------
# 3. Save verification report
# -----------------------------------------------------------------------------

SAVE_VALIDATION_DF = pd.DataFrame(SAVE_VALIDATION_ROWS)

validation_path = (
    Path(NB13_METADATA_ROOT)
    / "synthetic_dataset_save_validation.csv"
)
validation_path.parent.mkdir(parents=True, exist_ok=True)

SAVE_VALIDATION_DF.to_csv(validation_path, index=False)

overall_status = (
    "PASS"
    if len(SAVE_VALIDATION_DF) == len(FINAL_SYNTHETIC_DATAFRAMES)
    and SAVE_VALIDATION_DF["save_status"].eq("PASS").all()
    else "REVIEW"
)

summary = {
    "datasets_expected": len(FINAL_SYNTHETIC_DATAFRAMES),
    "datasets_saved": len(SAVED_DATASET_PATHS),
    "total_rows_saved": int(SAVE_VALIDATION_DF["rows"].sum()),
    "all_targets_present": bool(
        SAVE_VALIDATION_DF["target_present"].all()
    ),
    "overall_status": overall_status,
    "dataset_scope": "FINAL_SYNTHETIC_DATA_WITH_TARGET",
    "validation_report": str(validation_path),
}

summary_path = (
    Path(NB13_METADATA_ROOT)
    / "synthetic_dataset_save_summary.json"
)

with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

# -----------------------------------------------------------------------------
# 4. Final summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 22 — FINAL SUMMARY")
print("=" * 100)

print("Datasets saved       :", summary["datasets_saved"])
print("Total rows saved     :", summary["total_rows_saved"])
print("All targets present  :", summary["all_targets_present"])
print("Overall status       :", summary["overall_status"])
print("Validation report    :", validation_path)
print("Summary metadata     :", summary_path)

display(SAVE_VALIDATION_DF)

print("\nSECTION 22 EXECUTION COMPLETED.")


SECTION 22 — SAVE SYNTHETIC DATASETS
PASS | adult_income: (34189, 15) | target=income | CSV=/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets/adult_income_sppgan_synthetic.csv
PASS | bank_marketing: (31647, 17) | target=y | CSV=/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets/bank_marketing_sppgan_synthetic.csv
PASS | diabetes_130us: (71236, 48) | target=readmitted | CSV=/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets/diabetes_130us_sppgan_synthetic.csv

SECTION 22 — FINAL SUMMARY
Datasets saved       : 3
Total rows saved     : 137072
All targets present  : True
Overall status       : PASS
Validation report    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/synthetic_dataset_save_validation.csv
Summary metadata     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/synthetic_dataset_save_summary.json


,dataset,rows,columns,target_column,target_present,row_count_match,column_match,csv_size_bytes,csv_path,parquet_status,save_status
0,adult_income,34189,15,income,True,True,True,3912319,/content/drive/MyDrive/SPP_GAN_Research/result...,DISABLED,PASS
1,bank_marketing,31647,17,y,True,True,True,2662749,/content/drive/MyDrive/SPP_GAN_Research/result...,DISABLED,PASS
2,diabetes_130us,71236,48,readmitted,True,True,True,13017837,/content/drive/MyDrive/SPP_GAN_Research/result...,DISABLED,PASS



SECTION 22 EXECUTION COMPLETED.


In [94]:

# =============================================================================
# SECTION 23 — SAVE GENERATION METADATA
# =============================================================================

import json
import platform
import hashlib
from datetime import datetime, timezone
from pathlib import Path

print("=" * 100)
print("SECTION 23 — SAVE GENERATION METADATA")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Select final synthetic datasets
# -----------------------------------------------------------------------------

if (
    "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals()
    or not SYNTHETIC_DATAFRAMES_WITH_TARGET
):
    raise RuntimeError(
        "Final synthetic datasets are unavailable. Run Section 13 first."
    )

FINAL_METADATA_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

# -----------------------------------------------------------------------------
# 2. Robust file fingerprint helper
# -----------------------------------------------------------------------------

def metadata_file_sha256(path):
    if not path:
        return None

    try:
        path = Path(path)
        if not path.is_file():
            return None

        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(
                lambda: handle.read(1024 * 1024), b""
            ):
                digest.update(chunk)

        return digest.hexdigest()

    except (OSError, TypeError, ValueError):
        return None


# -----------------------------------------------------------------------------
# 3. Normalize generation runtime lookup
# -----------------------------------------------------------------------------

def get_dataset_runtime(dataset):
    runtime_registry = globals().get("GENERATION_RUNTIME", {})

    values = []

    if isinstance(runtime_registry, dict):
        for key, value in runtime_registry.items():
            matches = (
                key == dataset
                or (
                    isinstance(key, tuple)
                    and len(key) >= 1
                    and key[0] == dataset
                )
            )

            if matches:
                try:
                    runtime = float(value)
                    if np.isfinite(runtime) and runtime >= 0:
                        values.append(runtime)
                except (TypeError, ValueError):
                    continue

    if values:
        return {
            "runtime_seconds": float(np.mean(values)),
            "runtime_measurement_count": len(values),
            "runtime_min_seconds": float(np.min(values)),
            "runtime_max_seconds": float(np.max(values)),
        }

    return {
        "runtime_seconds": None,
        "runtime_measurement_count": 0,
        "runtime_min_seconds": None,
        "runtime_max_seconds": None,
    }


# -----------------------------------------------------------------------------
# 4. Build dataset-level generation metadata
# -----------------------------------------------------------------------------

GENERATION_METADATA = {
    "framework": "SPP-GAN",
    "notebook": str(globals().get("NOTEBOOK_NAME", "Notebook 13")),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "master_seed": MASTER_SEED,
    "generation_repetitions": GENERATION_REPETITIONS,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "device": str(DEVICE),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "torch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "configuration_path": str(GENERATION_CONFIG_PATH),
    "dataset_scope": "FINAL_SYNTHETIC_DATA_WITH_TARGET",
    "datasets": {},
}

for dataset, dataframe in FINAL_METADATA_DATAFRAMES.items():

    # Each dataset can have multiple repetition-specific model artifacts.
    artifact_registry = globals().get("MODEL_ARTIFACTS", {})
    dataset_artifacts = artifact_registry.get(dataset, {})

    model_artifacts = []

    if isinstance(dataset_artifacts, dict):
        model_items = dataset_artifacts.items()
    elif isinstance(dataset_artifacts, (str, Path)):
        model_items = [("1", dataset_artifacts)]
    else:
        model_items = []

    for repetition, artifact in model_items:
        candidate_paths = (
            list(artifact.values())
            if isinstance(artifact, dict)
            else [artifact]
        )

        for candidate in candidate_paths:
            if not isinstance(candidate, (str, Path)):
                continue

            candidate_path = Path(candidate).expanduser()

            if candidate_path.is_file():
                model_artifacts.append({
                    "repetition": str(repetition),
                    "model_path": str(candidate_path.resolve()),
                    "model_sha256": metadata_file_sha256(candidate_path),
                })
                break

    runtime_info = get_dataset_runtime(dataset)

    target_column = TARGET_COLUMNS.get(dataset)

    if not target_column:
        target_column = {
            "adult_income": "income",
            "bank_marketing": "y",
            "diabetes_130us": "readmitted",
        }.get(dataset)

    csv_path = SAVED_DATASET_PATHS.get(dataset)

    preprocessor_path = (
        PREPROCESSOR_PATHS.get(dataset)
        if "PREPROCESSOR_PATHS" in globals()
        else None
    )

    schema_path = (
        SCHEMA_PATHS.get(dataset)
        if "SCHEMA_PATHS" in globals()
        else None
    )

    GENERATION_METADATA["datasets"][dataset] = {
        "rows": int(len(dataframe)),
        "columns": [str(column) for column in dataframe.columns],
        "column_count": int(len(dataframe.columns)),
        "target_column": target_column,
        "target_present": (
            target_column in dataframe.columns
            if target_column else False
        ),
        "model_artifacts": model_artifacts,
        "model_artifact_count": len(model_artifacts),
        "synthetic_dataframe_sha256": dataframe_sha256(dataframe),
        "seed": generation_seed(dataset, 0),
        **runtime_info,
        "synthetic_csv": str(csv_path) if csv_path else None,
        "synthetic_csv_sha256": metadata_file_sha256(csv_path),
        "preprocessor_path": (
            str(preprocessor_path) if preprocessor_path else None
        ),
        "schema_path": str(schema_path) if schema_path else None,
    }

# -----------------------------------------------------------------------------
# 5. Save metadata and verify JSON integrity
# -----------------------------------------------------------------------------

GENERATION_METADATA_PATH = (
    Path(NB13_METADATA_ROOT)
    / "sppgan_synthetic_generation_metadata.json"
)
GENERATION_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

with GENERATION_METADATA_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        GENERATION_METADATA,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# Reload to confirm the metadata is valid JSON.
with GENERATION_METADATA_PATH.open("r", encoding="utf-8") as handle:
    RELOADED_GENERATION_METADATA = json.load(handle)

# -----------------------------------------------------------------------------
# 6. Final summary
# -----------------------------------------------------------------------------

METADATA_SUMMARY_ROWS = []

for dataset, metadata in RELOADED_GENERATION_METADATA["datasets"].items():
    METADATA_SUMMARY_ROWS.append({
        "dataset": dataset,
        "rows": metadata["rows"],
        "columns": metadata["column_count"],
        "target_column": metadata["target_column"],
        "target_present": metadata["target_present"],
        "model_artifacts_found": metadata["model_artifact_count"],
        "synthetic_hash_present": bool(
            metadata["synthetic_dataframe_sha256"]
        ),
        "csv_hash_present": bool(metadata["synthetic_csv_sha256"]),
        "runtime_seconds": metadata["runtime_seconds"],
        "runtime_measurement_count": metadata[
            "runtime_measurement_count"
        ],
    })

METADATA_SUMMARY_DF = pd.DataFrame(METADATA_SUMMARY_ROWS)

print("\nSECTION 23 — FINAL SUMMARY")
print("=" * 100)
print("Datasets documented :", len(METADATA_SUMMARY_DF))
print(
    "Targets present     :",
    int(METADATA_SUMMARY_DF["target_present"].sum()),
    "/",
    len(METADATA_SUMMARY_DF),
)
print(
    "Model artifacts     :",
    int(METADATA_SUMMARY_DF["model_artifacts_found"].sum()),
)
print(
    "Synthetic hashes    :",
    int(METADATA_SUMMARY_DF["synthetic_hash_present"].sum()),
    "/",
    len(METADATA_SUMMARY_DF),
)
print("JSON verified       :", GENERATION_METADATA_PATH.is_file())
print("Metadata saved      :", GENERATION_METADATA_PATH)

display(METADATA_SUMMARY_DF)

print("\nSECTION 23 EXECUTION COMPLETED.")


SECTION 23 — SAVE GENERATION METADATA

SECTION 23 — FINAL SUMMARY
Datasets documented : 3
Targets present     : 3 / 3
Model artifacts     : 15
Synthetic hashes    : 3 / 3
JSON verified       : True
Metadata saved      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/sppgan_synthetic_generation_metadata.json


,dataset,rows,columns,target_column,target_present,model_artifacts_found,synthetic_hash_present,csv_hash_present,runtime_seconds,runtime_measurement_count
0,adult_income,34189,15,income,True,5,True,True,6.379362,1
1,bank_marketing,31647,17,y,True,5,True,True,6.404864,1
2,diabetes_130us,71236,48,readmitted,True,5,True,True,61.684650,1



SECTION 23 EXECUTION COMPLETED.


In [95]:

# =============================================================================
# SECTION 24 — SAVE GENERATION MANIFEST
# =============================================================================

import json
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 24 — SAVE GENERATION MANIFEST")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Select final synthetic datasets
# -----------------------------------------------------------------------------

if (
    "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals()
    or not SYNTHETIC_DATAFRAMES_WITH_TARGET
):
    raise RuntimeError(
        "Final target-restored datasets are unavailable. "
        "Run Section 13 before generating the manifest."
    )

FINAL_MANIFEST_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

# -----------------------------------------------------------------------------
# 2. Resolve model artifacts and runtime measurements
# -----------------------------------------------------------------------------

def manifest_runtime_info(dataset):
    registry = globals().get("GENERATION_RUNTIME", {})
    values = []

    if isinstance(registry, dict):
        for key, value in registry.items():
            if (
                key == dataset
                or (
                    isinstance(key, tuple)
                    and len(key) > 0
                    and key[0] == dataset
                )
            ):
                try:
                    value = float(value)
                    if np.isfinite(value) and value >= 0:
                        values.append(value)
                except (TypeError, ValueError):
                    pass

    if not values:
        return {
            "runtime_seconds": None,
            "runtime_measurement_count": 0,
        }

    return {
        "runtime_seconds": float(np.mean(values)),
        "runtime_measurement_count": len(values),
    }


def manifest_model_artifacts(dataset):
    registry = globals().get("MODEL_ARTIFACTS", {})
    artifact_info = registry.get(dataset, {})
    artifacts = []

    if isinstance(artifact_info, dict):
        items = artifact_info.items()
    elif isinstance(artifact_info, (str, Path)):
        items = [("1", artifact_info)]
    else:
        items = []

    for repetition, value in items:
        candidates = (
            list(value.values())
            if isinstance(value, dict)
            else [value]
        )

        for candidate in candidates:
            if not isinstance(candidate, (str, Path)):
                continue

            path = Path(candidate).expanduser()

            if path.is_file():
                artifacts.append({
                    "repetition": str(repetition),
                    "model_path": str(path.resolve()),
                    "model_sha256": sha256_file(path),
                })
                break

    return artifacts


# -----------------------------------------------------------------------------
# 3. Build manifest rows
# -----------------------------------------------------------------------------

MANIFEST_ROWS = []

for dataset, dataframe in FINAL_MANIFEST_DATAFRAMES.items():

    csv_path = SAVED_DATASET_PATHS.get(dataset)

    if not csv_path or not Path(csv_path).is_file():
        raise RuntimeError(
            f"{dataset}: final synthetic CSV is missing. "
            "Run Section 22 first."
        )

    target_column = TARGET_COLUMNS.get(dataset)

    if not target_column:
        target_column = {
            "adult_income": "income",
            "bank_marketing": "y",
            "diabetes_130us": "readmitted",
        }.get(dataset)

    if not target_column or target_column not in dataframe.columns:
        raise RuntimeError(
            f"{dataset}: target column is missing from the final dataframe."
        )

    runtime_info = manifest_runtime_info(dataset)
    model_artifacts = manifest_model_artifacts(dataset)

    MANIFEST_ROWS.append({
        "dataset": dataset,
        "synthetic_csv": str(Path(csv_path).resolve()),
        "synthetic_csv_sha256": metadata_file_sha256(csv_path),
        "rows": int(len(dataframe)),
        "columns": int(len(dataframe.columns)),
        "column_names_json": json.dumps(
            [str(column) for column in dataframe.columns],
            ensure_ascii=False,
        ),
        "target_column": target_column,
        "target_present": True,
        "model_artifact_count": len(model_artifacts),
        "model_artifacts_json": json.dumps(
            model_artifacts,
            ensure_ascii=False,
        ),
        "synthetic_sha256": dataframe_sha256(dataframe),
        "seed": generation_seed(dataset, 0),
        "runtime_seconds": runtime_info["runtime_seconds"],
        "runtime_measurement_count": (
            runtime_info["runtime_measurement_count"]
        ),
        "status": "GENERATED",
    })

# -----------------------------------------------------------------------------
# 4. Save CSV and JSON manifests
# -----------------------------------------------------------------------------

GENERATION_MANIFEST_DF = pd.DataFrame(MANIFEST_ROWS)

NB13_MANIFEST_ROOT = Path(NB13_MANIFEST_ROOT)
NB13_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV_PATH = (
    NB13_MANIFEST_ROOT / "synthetic_generation_manifest.csv"
)
MANIFEST_JSON_PATH = (
    NB13_MANIFEST_ROOT / "synthetic_generation_manifest.json"
)

GENERATION_MANIFEST_DF.to_csv(
    MANIFEST_CSV_PATH,
    index=False,
)

with MANIFEST_JSON_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        MANIFEST_ROWS,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------
# 5. Verify saved manifests
# -----------------------------------------------------------------------------

CSV_MANIFEST_CHECK = pd.read_csv(MANIFEST_CSV_PATH)

with MANIFEST_JSON_PATH.open("r", encoding="utf-8") as handle:
    JSON_MANIFEST_CHECK = json.load(handle)

if len(CSV_MANIFEST_CHECK) != len(FINAL_MANIFEST_DATAFRAMES):
    raise RuntimeError("CSV manifest dataset count mismatch.")

if len(JSON_MANIFEST_CHECK) != len(FINAL_MANIFEST_DATAFRAMES):
    raise RuntimeError("JSON manifest dataset count mismatch.")

if not CSV_MANIFEST_CHECK["target_present"].all():
    raise RuntimeError("At least one manifest target check failed.")

# -----------------------------------------------------------------------------
# 6. Final summary
# -----------------------------------------------------------------------------

print("\nSECTION 24 — FINAL SUMMARY")
print("=" * 100)
print("Datasets documented :", len(GENERATION_MANIFEST_DF))
print("Total synthetic rows:", int(GENERATION_MANIFEST_DF["rows"].sum()))
print("Model artifacts     :", int(
    GENERATION_MANIFEST_DF["model_artifact_count"].sum()
))
print("Targets verified    :", int(
    GENERATION_MANIFEST_DF["target_present"].sum()
))
print("CSV manifest        :", MANIFEST_CSV_PATH)
print("JSON manifest       :", MANIFEST_JSON_PATH)
print("CSV reload verified :", len(CSV_MANIFEST_CHECK))
print("JSON reload verified:", len(JSON_MANIFEST_CHECK))

display(
    GENERATION_MANIFEST_DF[
        [
            "dataset",
            "rows",
            "columns",
            "target_column",
            "model_artifact_count",
            "synthetic_sha256",
            "runtime_seconds",
            "status",
        ]
    ]
)

print("\nSECTION 24 EXECUTION COMPLETED.")


SECTION 24 — SAVE GENERATION MANIFEST

SECTION 24 — FINAL SUMMARY
Datasets documented : 3
Total synthetic rows: 137072
Model artifacts     : 15
Targets verified    : 3
CSV manifest        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/manifests/synthetic_generation_manifest.csv
JSON manifest       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/manifests/synthetic_generation_manifest.json
CSV reload verified : 3
JSON reload verified: 3


,dataset,rows,columns,target_column,model_artifact_count,synthetic_sha256,runtime_seconds,status
0,adult_income,34189,15,income,5,802924f7a05ace796cfb0eebc6af64770c4bb67a945900...,6.379362,GENERATED
1,bank_marketing,31647,17,y,5,46b873fbf83e2aeba1a9f392ab22101f2c49ac20a0db1e...,6.404864,GENERATED
2,diabetes_130us,71236,48,readmitted,5,3bc11282b3ab1daa0b291b0820b3fd8424f3fc6f8085e3...,61.684650,GENERATED



SECTION 24 EXECUTION COMPLETED.


In [101]:

# ==================================================================================================
# SECTION 25 — FINAL VERIFICATION AND STRUCTURAL QUALITY GATE
# SPP-GAN Synthetic Data Generation | Notebook 13
# ==================================================================================================
# Objectives:
#   1. Verify final synthetic datasets, schemas, target columns, and sample sizes.
#   2. Validate feature/target values using the actual INVALID_VALUES_DF schema.
#   3. Verify saved synthetic CSV artifacts against in-memory dataframes.
#   4. Preserve range, domain, and missingness findings for separate quality review.
#   5. Save reproducible final verification CSV and JSON artifacts.
#
# IMPORTANT:
#   Structural PASS is not a claim of statistical fidelity, domain validity, utility,
#   or end-to-end differential privacy.
#
# Required upstream objects:
#   SYNTHETIC_DATAFRAMES_WITH_TARGET, DATASET_IDS, TARGET_COLUMNS,
#   SAMPLE_SIZE_DF, INVALID_VALUES_DF, SCHEMA_VALIDATION_DF,
#   RANGE_VALIDATION_DF, DOMAIN_VALIDATION_DF, MISSINGNESS_DF,
#   NB13_VALIDATION_ROOT, NB13_SYNTHETIC_ROOT
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 25 — FINAL VERIFICATION AND STRUCTURAL QUALITY GATE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required upstream objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "SAMPLE_SIZE_DF",
    "INVALID_VALUES_DF",
    "SCHEMA_VALIDATION_DF",
    "RANGE_VALIDATION_DF",
    "DOMAIN_VALIDATION_DF",
    "MISSINGNESS_DF",
    "NB13_VALIDATION_ROOT",
    "NB13_SYNTHETIC_ROOT",
]

missing_objects = [name for name in REQUIRED_OBJECTS if name not in globals()]

if missing_objects:
    raise NameError(
        "Run the preceding Notebook 13 sections first. Missing objects: "
        + ", ".join(missing_objects)
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

for report_name in [
    "SAMPLE_SIZE_DF",
    "INVALID_VALUES_DF",
    "SCHEMA_VALIDATION_DF",
    "RANGE_VALIDATION_DF",
    "DOMAIN_VALIDATION_DF",
    "MISSINGNESS_DF",
]:
    if not isinstance(globals()[report_name], pd.DataFrame):
        raise TypeError(f"{report_name} must be a pandas DataFrame.")

DATASET_IDS = list(DATASET_IDS)
TARGET_COLUMNS = dict(TARGET_COLUMNS)

NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_SYNTHETIC_ROOT = Path(NB13_SYNTHETIC_ROOT)

NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
NB13_SYNTHETIC_ROOT.mkdir(parents=True, exist_ok=True)

# Canonical final CSV names used by Section 22.
FINAL_SYNTHETIC_FILES = {
    dataset: NB13_SYNTHETIC_ROOT / f"{dataset}_sppgan_synthetic.csv"
    for dataset in DATASET_IDS
}

# --------------------------------------------------------------------------------------------------
# 2. General helper functions
# --------------------------------------------------------------------------------------------------

def normalize_id(value):
    if pd.isna(value):
        return None
    return str(value).strip().lower()


def find_column(df, candidates):
    """Case-insensitive exact column lookup."""
    if not isinstance(df, pd.DataFrame):
        return None

    lookup = {str(col).strip().lower(): col for col in df.columns}

    for candidate in candidates:
        if candidate.strip().lower() in lookup:
            return lookup[candidate.strip().lower()]

    return None


def dataset_rows(report, dataset):
    """
    Support reports using either 'dataset' or 'dataset_id'.
    Return an empty dataframe if no matching dataset rows exist.
    """
    if not isinstance(report, pd.DataFrame) or report.empty:
        return pd.DataFrame()

    dataset_col = find_column(report, ["dataset", "dataset_id"])

    if dataset_col is None:
        return pd.DataFrame()

    mask = report[dataset_col].map(normalize_id).eq(normalize_id(dataset))
    return report.loc[mask].copy()


def status_bool(value):
    """Convert recognized status labels to True/False; unknown values remain None."""
    if pd.isna(value):
        return None

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, (int, float, np.integer, np.floating)):
        if value == 1:
            return True
        if value == 0:
            return False

    value = str(value).strip().upper()

    if value in {"PASS", "PASSED", "TRUE", "VALID", "OK", "SUCCESS"}:
        return True

    if value in {"FAIL", "FAILED", "FALSE", "INVALID", "ERROR"}:
        return False

    return None


def report_status(report, dataset, status_columns):
    """Evaluate explicit status values from a dataset-specific report."""
    rows = dataset_rows(report, dataset)

    if rows.empty:
        return None

    status_col = find_column(rows, status_columns)

    if status_col is None:
        return None

    values = [status_bool(value) for value in rows[status_col].tolist()]

    if not values or any(value is None for value in values):
        return None

    return bool(all(values))


def report_review_count(report, dataset, status_columns):
    """Count rows explicitly marked REVIEW, WARN, or FAIL."""
    rows = dataset_rows(report, dataset)

    if rows.empty:
        return None

    status_col = find_column(rows, status_columns)

    if status_col is None:
        return None

    statuses = rows[status_col].astype(str).str.strip().str.upper()

    return int(
        statuses.isin(
            ["REVIEW", "WARN", "WARNING", "FAIL", "FAILED", "INVALID", "ERROR"]
        ).sum()
    )


def report_count_sum(report, dataset, count_columns):
    """Sum available numeric violation-count columns for a dataset."""
    rows = dataset_rows(report, dataset)

    if rows.empty:
        return None

    count_cols = [
        col for col in rows.columns
        if str(col).strip().lower() in {name.lower() for name in count_columns}
    ]

    if not count_cols:
        return None

    numeric = rows[count_cols].apply(pd.to_numeric, errors="coerce")

    if numeric.isna().all().all():
        return None

    # Missing components are not evidence of zero violations.
    if numeric.isna().any().any():
        return None

    values = numeric.to_numpy(dtype=float)

    if not np.isfinite(values).all():
        return None

    return int(values.sum())


# --------------------------------------------------------------------------------------------------
# 3. Schema validation
# --------------------------------------------------------------------------------------------------

def get_schema_pass(dataset):
    result = report_status(
        SCHEMA_VALIDATION_DF,
        dataset,
        [
            "schema_pass",
            "schema_status",
            "validation_status",
            "status",
            "overall_status",
        ],
    )

    # The existing Section 25 output already showed successful schema validation.
    # Only use that result if the schema report has no recognized status field.
    if result is not None:
        return result

    prior = globals().get("FINAL_VERIFICATION_DF")

    if isinstance(prior, pd.DataFrame) and not prior.empty:
        rows = dataset_rows(prior, dataset)
        col = find_column(rows, ["schema_pass"])

        if not rows.empty and col is not None:
            values = [status_bool(value) for value in rows[col].tolist()]
            if values and all(value is not None for value in values):
                return bool(all(values))

    return None


# --------------------------------------------------------------------------------------------------
# 4. Sample-size verification — uses actual SAMPLE_SIZE_DF schema
# --------------------------------------------------------------------------------------------------

def get_sample_size_pass(dataset, in_memory_rows):
    rows = dataset_rows(SAMPLE_SIZE_DF, dataset)

    if len(rows) != 1:
        return None

    row = rows.iloc[0]

    status_col = find_column(rows, ["sample_size_status", "status"])
    expected_col = find_column(rows, ["expected_rows", "expected_count"])
    actual_col = find_column(rows, ["actual_rows", "generated_rows"])
    difference_col = find_column(rows, ["row_count_difference"])

    if status_col is None or expected_col is None or actual_col is None:
        return None

    status = status_bool(row[status_col])
    expected = pd.to_numeric(pd.Series([row[expected_col]]), errors="coerce").iloc[0]
    actual = pd.to_numeric(pd.Series([row[actual_col]]), errors="coerce").iloc[0]

    if pd.isna(expected) or pd.isna(actual):
        return None

    if difference_col is not None:
        difference = pd.to_numeric(
            pd.Series([row[difference_col]]), errors="coerce"
        ).iloc[0]

        if pd.isna(difference) or difference != 0:
            return False

    counts_match = bool(
        expected == actual
        and actual == in_memory_rows
    )

    if status is False or not counts_match:
        return False

    if status is True and counts_match:
        return True

    return None


# --------------------------------------------------------------------------------------------------
# 5. Invalid-value verification — uses actual INVALID_VALUES_DF schema
# --------------------------------------------------------------------------------------------------

def get_invalid_value_pass(dataset):
    rows = dataset_rows(INVALID_VALUES_DF, dataset)

    if rows.empty:
        return None, "NO_INVALID_VALUE_REPORT_ROWS"

    # Identifier rows are tracked separately because the generated schema
    # intentionally excludes __original_row_id__.
    role_col = find_column(rows, ["column_role"])

    if role_col is not None:
        identifier_mask = (
            rows[role_col].astype(str).str.strip().str.lower() == "identifier"
        )
        identifier_rows = rows.loc[identifier_mask]
        value_rows = rows.loc[~identifier_mask].copy()
    else:
        identifier_rows = pd.DataFrame()
        value_rows = rows.copy()

    if value_rows.empty:
        return None, "NO_FEATURE_OR_TARGET_ROWS_TO_VALIDATE"

    value_status_col = find_column(
        value_rows, ["value_status", "invalid_status", "validation_status"]
    )

    if value_status_col is None:
        return None, "VALUE_STATUS_COLUMN_MISSING"

    value_results = [status_bool(v) for v in value_rows[value_status_col]]

    if not value_results or any(v is None for v in value_results):
        return None, "UNKNOWN_VALUE_STATUS"

    if not all(value_results):
        return False, "ONE_OR_MORE_FEATURE_OR_TARGET_VALUE_CHECKS_FAILED"

    invalid_count_col = find_column(
        value_rows, ["invalid_value_count", "invalid_count", "violation_count"]
    )

    if invalid_count_col is None:
        return None, "INVALID_VALUE_COUNT_COLUMN_MISSING"

    invalid_counts = pd.to_numeric(
        value_rows[invalid_count_col], errors="coerce"
    )

    if invalid_counts.isna().any() or not np.isfinite(
        invalid_counts.to_numpy(dtype=float)
    ).all():
        return None, "INVALID_VALUE_COUNTS_INCOMPLETE"

    if (invalid_counts != 0).any():
        return False, "NONZERO_INVALID_VALUE_COUNT"

    component_columns = [
        "non_finite_numeric_count",
        "nonnumeric_in_numeric_column_count",
        "unseen_category_count",
        "fractional_integer_value_count",
    ]

    for candidate in component_columns:
        col = find_column(value_rows, [candidate])

        if col is None:
            continue

        counts = pd.to_numeric(value_rows[col], errors="coerce")

        if counts.isna().any():
            return None, f"{candidate}_INCOMPLETE"

        if not np.isfinite(counts.to_numpy(dtype=float)).all():
            return None, f"{candidate}_NONFINITE"

        if (counts != 0).any():
            return False, f"{candidate}_NONZERO"

    identifier_note = "NO_IDENTIFIER_ROWS"

    if not identifier_rows.empty:
        overall_col = find_column(identifier_rows, ["overall_status"])
        if overall_col is not None:
            statuses = (
                identifier_rows[overall_col].astype(str).str.strip().str.upper()
            )

            if statuses.eq("SCHEMA_MISMATCH").any():
                identifier_note = "IDENTIFIER_SCHEMA_MISMATCH_REPORTED_SEPARATELY"
            else:
                identifier_note = "IDENTIFIER_ROWS_RECORDED_SEPARATELY"

    return True, f"FEATURE_TARGET_VALUE_CHECKS_PASS; {identifier_note}"


# --------------------------------------------------------------------------------------------------
# 6. Final dataframe, target, and saved CSV verification
# --------------------------------------------------------------------------------------------------

verification_rows = []

for dataset in DATASET_IDS:
    print("\n" + "-" * 100)
    print(f"VERIFYING DATASET: {dataset}")
    print("-" * 100)

    target_col = TARGET_COLUMNS.get(dataset)
    csv_path = FINAL_SYNTHETIC_FILES[dataset]

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        verification_rows.append({
            "dataset": dataset,
            "synthetic_rows": None,
            "synthetic_columns": None,
            "target_column": target_col,
            "target_present": False,
            "schema_pass": None,
            "sample_size_pass": None,
            "invalid_values_pass": None,
            "invalid_values_note": "FINAL_DATAFRAME_MISSING",
            "saved_file_pass": False,
            "saved_rows_match": False,
            "saved_columns_match": False,
            "saved_target_present": False,
            "saved_column_order_match": False,
            "range_report_available": False,
            "range_review_count": None,
            "domain_report_available": False,
            "domain_review_count": None,
            "domain_violation_count": None,
            "missingness_report_available": False,
            "missingness_review_count": None,
            "required_evidence_complete": False,
            "structural_verification_status": "REVIEW",
            "quality_review_status": "REVIEW",
            "saved_csv_path": str(csv_path),
            "saved_file_error": "FINAL_DATAFRAME_MISSING",
        })
        continue

    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(f"{dataset}: final synthetic object is not a DataFrame.")

    synthetic_rows = int(len(synthetic_df))
    synthetic_columns = int(len(synthetic_df.columns))
    target_present = bool(target_col in synthetic_df.columns) if target_col else False

    schema_pass = get_schema_pass(dataset)
    sample_pass = get_sample_size_pass(dataset, synthetic_rows)
    invalid_pass, invalid_note = get_invalid_value_pass(dataset)

    saved_file_pass = False
    saved_rows_match = False
    saved_columns_match = False
    saved_target_present = False
    saved_column_order_match = False
    saved_file_error = None

    if csv_path.exists():
        try:
            saved_df = pd.read_csv(csv_path, low_memory=False)

            saved_rows_match = bool(len(saved_df) == synthetic_rows)
            saved_columns_match = bool(len(saved_df.columns) == synthetic_columns)
            saved_target_present = (
                bool(target_col in saved_df.columns) if target_col else False
            )
            saved_column_order_match = bool(
                list(saved_df.columns) == list(synthetic_df.columns)
            )

            saved_file_pass = bool(
                saved_rows_match
                and saved_columns_match
                and saved_target_present
                and saved_column_order_match
            )

        except Exception as exc:
            saved_file_error = f"{type(exc).__name__}: {exc}"
    else:
        saved_file_error = "CANONICAL_SYNTHETIC_CSV_NOT_FOUND"

    # ------------------------------------------------------------------------------------------------
    # 7. Separate range, domain, and missingness findings
    # ------------------------------------------------------------------------------------------------

    range_rows = dataset_rows(RANGE_VALIDATION_DF, dataset)
    domain_rows = dataset_rows(DOMAIN_VALIDATION_DF, dataset)
    missingness_rows = dataset_rows(MISSINGNESS_DF, dataset)

    range_review_count = report_review_count(
        RANGE_VALIDATION_DF,
        dataset,
        ["range_status", "validation_status", "status"],
    )

    domain_review_count = report_review_count(
        DOMAIN_VALIDATION_DF,
        dataset,
        ["domain_status", "validation_status", "status"],
    )

    domain_violation_count = report_count_sum(
        DOMAIN_VALIDATION_DF,
        dataset,
        [
            "below_domain_min_count",
            "above_domain_max_count",
            "fractional_value_count",
            "nonnumeric_count",
            "nonfinite_count",
        ],
    )

    missingness_review_count = report_review_count(
        MISSINGNESS_DF,
        dataset,
        ["missingness_status", "validation_status", "status"],
    )

    range_available = not range_rows.empty
    domain_available = not domain_rows.empty
    missingness_available = not missingness_rows.empty

    # ------------------------------------------------------------------------------------------------
    # 8. Structural gate — every required check must be explicitly True
    # ------------------------------------------------------------------------------------------------

    required_checks = [
        schema_pass,
        sample_pass,
        invalid_pass,
        saved_file_pass,
        target_present,
    ]

    required_evidence_complete = all(
        value is not None for value in required_checks
    )

    structural_pass = bool(
        required_evidence_complete
        and all(value is True for value in required_checks)
    )

    structural_status = "PASS" if structural_pass else "REVIEW"

    # Quality-review status is separate and conservative.
    # Any reported review flag or positive violation count retains REVIEW.
    review_counts = [
        range_review_count,
        domain_review_count,
        missingness_review_count,
    ]

    has_review_flags = any(
        value is not None and value > 0 for value in review_counts
    )

    has_domain_violations = (
        domain_violation_count is not None and domain_violation_count > 0
    )

    quality_evidence_complete = bool(
        range_available and domain_available and missingness_available
    )

    quality_status = (
        "PASS_NO_REPORTED_REVIEW_FLAGS"
        if quality_evidence_complete
        and not has_review_flags
        and not has_domain_violations
        else "REVIEW"
    )

    record = {
        "dataset": dataset,
        "synthetic_rows": synthetic_rows,
        "synthetic_columns": synthetic_columns,
        "target_column": target_col,
        "target_present": target_present,
        "schema_pass": schema_pass,
        "sample_size_pass": sample_pass,
        "invalid_values_pass": invalid_pass,
        "invalid_values_note": invalid_note,
        "saved_file_pass": saved_file_pass,
        "saved_rows_match": saved_rows_match,
        "saved_columns_match": saved_columns_match,
        "saved_target_present": saved_target_present,
        "saved_column_order_match": saved_column_order_match,
        "range_report_available": range_available,
        "range_review_count": range_review_count,
        "domain_report_available": domain_available,
        "domain_review_count": domain_review_count,
        "domain_violation_count": domain_violation_count,
        "missingness_report_available": missingness_available,
        "missingness_review_count": missingness_review_count,
        "required_evidence_complete": required_evidence_complete,
        "structural_verification_status": structural_status,
        "quality_review_status": quality_status,
        "saved_csv_path": str(csv_path),
        "saved_file_error": saved_file_error,
    }

    verification_rows.append(record)

    print(f"Rows / columns             : {synthetic_rows:,} / {synthetic_columns:,}")
    print(f"Target present             : {target_present}")
    print(f"Schema check               : {schema_pass}")
    print(f"Sample-size check          : {sample_pass}")
    print(f"Feature/target value check : {invalid_pass}")
    print(f"Saved CSV check            : {saved_file_pass}")
    print(f"Range review rows          : {range_review_count}")
    print(f"Domain review rows         : {domain_review_count}")
    print(f"Domain violation count     : {domain_violation_count}")
    print(f"Missingness review rows    : {missingness_review_count}")
    print(f"Structural status          : {structural_status}")
    print(f"Quality review status      : {quality_status}")

    if saved_file_error:
        print(f"Saved CSV note             : {saved_file_error}")

# --------------------------------------------------------------------------------------------------
# 9. Create and save the final verification report
# --------------------------------------------------------------------------------------------------

FINAL_VERIFICATION_DF = pd.DataFrame(verification_rows)

FINAL_VERIFICATION_CSV = NB13_VALIDATION_ROOT / "final_verification.csv"
FINAL_VERIFICATION_JSON = NB13_VALIDATION_ROOT / "final_verification_summary.json"

FINAL_VERIFICATION_DF.to_csv(FINAL_VERIFICATION_CSV, index=False)

structural_pass_count = int(
    FINAL_VERIFICATION_DF["structural_verification_status"].eq("PASS").sum()
)

structural_review_count = int(
    FINAL_VERIFICATION_DF["structural_verification_status"].eq("REVIEW").sum()
)

quality_review_count = int(
    FINAL_VERIFICATION_DF["quality_review_status"].eq("REVIEW").sum()
)

summary = {
    "section": "Notebook 13 — Section 25",
    "verification_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "datasets_checked": int(len(FINAL_VERIFICATION_DF)),
    "structural_pass_count": structural_pass_count,
    "structural_review_count": structural_review_count,
    "quality_review_count": quality_review_count,
    "structural_verification_scope": [
        "schema validation report",
        "sample-size status and expected/actual row-count consistency",
        "feature/target invalid-value status and violation counts",
        "target-column presence",
        "saved CSV row, column, target, and column-order consistency",
    ],
    "quality_review_scope": [
        "range validation",
        "domain validation",
        "missingness validation",
    ],
    "interpretation": (
        "Structural PASS indicates only that the explicit structural checks passed. "
        "It does not establish statistical fidelity, acceptable domain validity, "
        "downstream utility, or end-to-end differential privacy. Identifier schema "
        "differences are recorded separately. Range, domain, and missingness findings "
        "remain independent quality-review items."
    ),
    "reports_used": {
        "schema": "SCHEMA_VALIDATION_DF",
        "sample_size": "SAMPLE_SIZE_DF",
        "invalid_values": "INVALID_VALUES_DF",
        "range": "RANGE_VALIDATION_DF",
        "domain": "DOMAIN_VALIDATION_DF",
        "missingness": "MISSINGNESS_DF",
    },
    "final_csv": str(FINAL_VERIFICATION_CSV),
    "final_json": str(FINAL_VERIFICATION_JSON),
}

with open(FINAL_VERIFICATION_JSON, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 10. Reload artifacts and verify report integrity
# --------------------------------------------------------------------------------------------------

RELOADED_FINAL_VERIFICATION_DF = pd.read_csv(FINAL_VERIFICATION_CSV)

with open(FINAL_VERIFICATION_JSON, "r", encoding="utf-8") as file:
    RELOADED_FINAL_VERIFICATION_SUMMARY = json.load(file)

assert len(RELOADED_FINAL_VERIFICATION_DF) == len(DATASET_IDS), (
    "Saved verification CSV row count does not match registered datasets."
)

assert RELOADED_FINAL_VERIFICATION_SUMMARY["datasets_checked"] == len(DATASET_IDS), (
    "Saved verification JSON dataset count does not match registered datasets."
)

assert RELOADED_FINAL_VERIFICATION_SUMMARY["structural_pass_count"] == structural_pass_count

# --------------------------------------------------------------------------------------------------
# 11. Final summary
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 25 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked                  : {len(FINAL_VERIFICATION_DF)}")
print(f"Structural checks passed          : {structural_pass_count}")
print(f"Datasets requiring structural review: {structural_review_count}")
print(f"Datasets requiring quality review : {quality_review_count}")
print(f"Final verification CSV            : {FINAL_VERIFICATION_CSV}")
print(f"Final verification JSON           : {FINAL_VERIFICATION_JSON}")

display_columns = [
    "dataset",
    "synthetic_rows",
    "synthetic_columns",
    "schema_pass",
    "sample_size_pass",
    "invalid_values_pass",
    "saved_file_pass",
    "range_review_count",
    "domain_review_count",
    "domain_violation_count",
    "missingness_review_count",
    "required_evidence_complete",
    "structural_verification_status",
    "quality_review_status",
]

display(
    FINAL_VERIFICATION_DF[
        [col for col in display_columns if col in FINAL_VERIFICATION_DF.columns]
    ]
)

print("\nIMPORTANT:")
print(
    "Structural PASS does not certify overall synthetic-data quality, "
    "statistical fidelity, or end-to-end differential privacy."
)
print("Section 25 completed successfully.")


SECTION 25 — FINAL VERIFICATION AND STRUCTURAL QUALITY GATE

----------------------------------------------------------------------------------------------------
VERIFYING DATASET: adult_income
----------------------------------------------------------------------------------------------------
Rows / columns             : 34,189 / 15
Target present             : True
Schema check               : True
Sample-size check          : True
Feature/target value check : True
Saved CSV check            : True
Range review rows          : 6
Domain review rows         : 6
Domain violation count     : 28505
Missingness review rows    : 3
Structural status          : PASS
Quality review status      : REVIEW

----------------------------------------------------------------------------------------------------
VERIFYING DATASET: bank_marketing
----------------------------------------------------------------------------------------------------
Rows / columns             : 31,647 / 17
Target present    

,dataset,synthetic_rows,synthetic_columns,schema_pass,sample_size_pass,invalid_values_pass,saved_file_pass,range_review_count,domain_review_count,domain_violation_count,missingness_review_count,required_evidence_complete,structural_verification_status,quality_review_status
0,adult_income,34189,15,True,True,True,True,6,6,28505,3,True,PASS,REVIEW
1,bank_marketing,31647,17,True,True,True,True,7,5,33641,0,True,PASS,REVIEW
2,diabetes_130us,71236,48,True,True,True,True,11,11,91961,9,True,PASS,REVIEW



IMPORTANT:
Structural PASS does not certify overall synthetic-data quality, statistical fidelity, or end-to-end differential privacy.
Section 25 completed successfully.


In [103]:

# =============================================================================
# SECTION 26 — COMPLETION SUMMARY
# SPP-GAN Synthetic Data Generation | Notebook 13
# =============================================================================
# Purpose:
#   1. Summarize final synthetic datasets, including target columns.
#   2. Cross-check structural verification results from Section 25.
#   3. Record generation errors and unresolved quality-review findings.
#   4. Persist a reproducible completion summary.
#
# Important:
#   GENERATION_COMPLETED does not mean QUALITY_VALIDATED or PRIVACY_CERTIFIED.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 100)
print("NOTEBOOK 13 — COMPLETION SUMMARY")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Validate required objects
# -----------------------------------------------------------------------------

required_objects = [
    "DATASET_IDS",
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "GENERATION_ERRORS",
    "PROJECT_ROOT",
    "NB13_SYNTHETIC_ROOT",
    "NB13_CONFIG_ROOT",
    "NB13_METADATA_ROOT",
    "NB13_MANIFEST_ROOT",
    "NB13_VALIDATION_ROOT",
    "NB13_LOG_ROOT",
    "FINAL_VERIFICATION_DF",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Required Notebook 13 variables are missing: "
        + ", ".join(missing_objects)
    )

DATASET_IDS = list(DATASET_IDS)
FINAL_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

# Ensure the metadata directory exists.
NB13_METADATA_ROOT = Path(NB13_METADATA_ROOT)
NB13_METADATA_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Identify generated, missing, and malformed datasets
# -----------------------------------------------------------------------------

generated_datasets = [
    dataset
    for dataset in DATASET_IDS
    if dataset in FINAL_DATAFRAMES
    and isinstance(FINAL_DATAFRAMES[dataset], pd.DataFrame)
    and not FINAL_DATAFRAMES[dataset].empty
]

failed_or_skipped_datasets = sorted(
    set(DATASET_IDS) - set(generated_datasets)
)

malformed_datasets = []
target_missing_datasets = []

for dataset in generated_datasets:
    df = FINAL_DATAFRAMES[dataset]
    target_col = TARGET_COLUMNS.get(dataset) if "TARGET_COLUMNS" in globals() else None

    if df.columns.duplicated().any():
        malformed_datasets.append(dataset)

    if target_col is None or target_col not in df.columns:
        target_missing_datasets.append(dataset)

# -----------------------------------------------------------------------------
# 3. Summarize final verification results
# -----------------------------------------------------------------------------

if not isinstance(FINAL_VERIFICATION_DF, pd.DataFrame):
    raise TypeError("FINAL_VERIFICATION_DF must be a pandas DataFrame.")

verification_records = (
    FINAL_VERIFICATION_DF.to_dict(orient="records")
    if not FINAL_VERIFICATION_DF.empty
    else []
)

verification_status_col = (
    "structural_verification_status"
    if "structural_verification_status" in FINAL_VERIFICATION_DF.columns
    else None
)

quality_status_col = (
    "quality_review_status"
    if "quality_review_status" in FINAL_VERIFICATION_DF.columns
    else None
)

if verification_status_col:
    structural_pass_count = int(
        FINAL_VERIFICATION_DF[verification_status_col]
        .astype(str).str.upper().eq("PASS").sum()
    )
    structural_review_count = int(
        FINAL_VERIFICATION_DF[verification_status_col]
        .astype(str).str.upper().eq("REVIEW").sum()
    )
else:
    structural_pass_count = 0
    structural_review_count = len(DATASET_IDS)

if quality_status_col:
    quality_review_count = int(
        FINAL_VERIFICATION_DF[quality_status_col]
        .astype(str).str.upper().eq("REVIEW").sum()
    )
else:
    quality_review_count = None

verification_dataset_col = (
    "dataset"
    if "dataset" in FINAL_VERIFICATION_DF.columns
    else "dataset_id" if "dataset_id" in FINAL_VERIFICATION_DF.columns
    else None
)

verified_datasets = set()
if verification_dataset_col:
    verified_datasets = set(
        FINAL_VERIFICATION_DF[verification_dataset_col]
        .dropna().astype(str).str.strip().str.lower()
    )

missing_verification_datasets = sorted(
    set(dataset.lower() for dataset in DATASET_IDS) - verified_datasets
)

all_structurally_verified = bool(
    not missing_verification_datasets
    and structural_pass_count == len(DATASET_IDS)
    and len(FINAL_VERIFICATION_DF) >= len(DATASET_IDS)
)

# -----------------------------------------------------------------------------
# 4. Check final saved CSV artifacts
# -----------------------------------------------------------------------------

saved_artifact_results = {}

for dataset in DATASET_IDS:
    csv_path = (
        Path(NB13_SYNTHETIC_ROOT)
        / f"{dataset}_sppgan_synthetic.csv"
    )

    in_memory_df = FINAL_DATAFRAMES.get(dataset)

    result = {
        "path": str(csv_path),
        "exists": csv_path.exists(),
        "readable": False,
        "rows_match": False,
        "columns_match": False,
        "target_present": False,
        "artifact_status": "REVIEW",
    }

    if csv_path.exists() and isinstance(in_memory_df, pd.DataFrame):
        try:
            saved_df = pd.read_csv(csv_path, low_memory=False)

            result["readable"] = True
            result["rows_match"] = bool(
                len(saved_df) == len(in_memory_df)
            )
            result["columns_match"] = bool(
                list(saved_df.columns) == list(in_memory_df.columns)
            )

            target_col = (
                TARGET_COLUMNS.get(dataset)
                if "TARGET_COLUMNS" in globals()
                else None
            )

            result["target_present"] = bool(
                target_col is not None and target_col in saved_df.columns
            )

            result["artifact_status"] = (
                "PASS"
                if result["readable"]
                and result["rows_match"]
                and result["columns_match"]
                and result["target_present"]
                else "REVIEW"
            )

        except Exception as exc:
            result["read_error"] = f"{type(exc).__name__}: {exc}"

    saved_artifact_results[dataset] = result

saved_artifacts_pass = all(
    item["artifact_status"] == "PASS"
    for item in saved_artifact_results.values()
)

# -----------------------------------------------------------------------------
# 5. Determine generation and verification statuses independently
# -----------------------------------------------------------------------------

generation_completed = bool(
    len(generated_datasets) == len(DATASET_IDS)
    and not failed_or_skipped_datasets
    and not malformed_datasets
    and not target_missing_datasets
)

generation_status = (
    "COMPLETED"
    if generation_completed
    else "PARTIAL_OR_FAILED"
)

verification_status = (
    "STRUCTURALLY_VERIFIED"
    if all_structurally_verified and saved_artifacts_pass
    else "REVIEW_REQUIRED"
)

quality_status = (
    "REVIEW_REQUIRED"
    if quality_review_count is None or quality_review_count > 0
    else "NO_REPORTED_REVIEW_FLAGS"
)

# Generation errors are preserved in their original form but converted to a
# JSON-compatible value when serialized.
generation_errors_serializable = GENERATION_ERRORS

# -----------------------------------------------------------------------------
# 6. Build completion summary
# -----------------------------------------------------------------------------

COMPLETION_SUMMARY = {
    "framework": "SPP-GAN",
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "Notebook 13",
    "completion_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),

    "requested_datasets": DATASET_IDS,
    "generated_datasets": generated_datasets,
    "failed_or_skipped_datasets": failed_or_skipped_datasets,
    "malformed_datasets": malformed_datasets,
    "target_missing_datasets": target_missing_datasets,

    "requested_dataset_count": len(DATASET_IDS),
    "generated_dataset_count": len(generated_datasets),
    "failed_or_skipped_dataset_count": len(failed_or_skipped_datasets),

    "generation_status": generation_status,
    "generation_errors": generation_errors_serializable,

    "structural_verification": {
        "status": verification_status,
        "structural_pass_count": structural_pass_count,
        "structural_review_count": structural_review_count,
        "missing_verification_datasets": missing_verification_datasets,
        "all_datasets_structurally_verified": all_structurally_verified,
    },

    "saved_artifact_verification": {
        "all_saved_artifacts_pass": saved_artifacts_pass,
        "artifacts": saved_artifact_results,
    },

    "quality_review": {
        "status": quality_status,
        "datasets_requiring_quality_review": quality_review_count,
        "interpretation": (
            "Quality review is separate from generation and structural verification. "
            "Range, domain, missingness, fidelity, utility, and privacy require their "
            "own evidence and must not be inferred from generation completion."
        ),
    },

    "final_verification": verification_records,

    "artifact_directories": {
        "synthetic_datasets": str(NB13_SYNTHETIC_ROOT),
        "configuration": str(NB13_CONFIG_ROOT),
        "metadata": str(NB13_METADATA_ROOT),
        "manifests": str(NB13_MANIFEST_ROOT),
        "validation": str(NB13_VALIDATION_ROOT),
        "logs": str(NB13_LOG_ROOT),
    },
}

# -----------------------------------------------------------------------------
# 7. Save completion summary
# -----------------------------------------------------------------------------

COMPLETION_SUMMARY_PATH = (
    NB13_METADATA_ROOT / "notebook_13_completion_summary.json"
)

with open(COMPLETION_SUMMARY_PATH, "w", encoding="utf-8") as handle:
    json.dump(
        COMPLETION_SUMMARY,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------
# 8. Reload and validate the completion summary
# -----------------------------------------------------------------------------

with open(COMPLETION_SUMMARY_PATH, "r", encoding="utf-8") as handle:
    RELOADED_COMPLETION_SUMMARY = json.load(handle)

assert (
    RELOADED_COMPLETION_SUMMARY["requested_dataset_count"]
    == len(DATASET_IDS)
), "Completion summary dataset count does not match."

assert (
    RELOADED_COMPLETION_SUMMARY["generated_dataset_count"]
    == len(generated_datasets)
), "Completion summary generated count does not match."

# -----------------------------------------------------------------------------
# 9. Final output
# -----------------------------------------------------------------------------

print("=" * 100)
print("NOTEBOOK 13 — COMPLETION SUMMARY")
print("=" * 100)

print("Generation status         :", generation_status)
print("Requested datasets        :", len(DATASET_IDS))
print("Generated datasets        :", len(generated_datasets))
print("Failed/skipped datasets   :", len(failed_or_skipped_datasets))
print("Malformed datasets        :", len(malformed_datasets))
print("Missing target datasets   :", len(target_missing_datasets))
print("Structural pass count     :", structural_pass_count)
print("Structural review count   :", structural_review_count)
print("Artifact verification     :", "PASS" if saved_artifacts_pass else "REVIEW")
print("Overall verification      :", verification_status)
print("Quality review status     :", quality_status)
print("Synthetic data directory  :", NB13_SYNTHETIC_ROOT)

print(
    "Manifest                  :",
    globals().get(
        "MANIFEST_JSON_PATH",
        Path(NB13_MANIFEST_ROOT) / "synthetic_generation_manifest.json",
    ),
)
print(
    "Metadata                  :",
    globals().get(
        "GENERATION_METADATA_PATH",
        Path(NB13_METADATA_ROOT) / "sppgan_synthetic_generation_metadata.json",
    ),
)
print(
    "Final verification        :",
    Path(NB13_VALIDATION_ROOT) / "final_verification.csv",
)
print("Completion summary        :", COMPLETION_SUMMARY_PATH)

if failed_or_skipped_datasets:
    print("\nDatasets not generated:")
    for dataset in failed_or_skipped_datasets:
        print(" -", dataset)

if target_missing_datasets:
    print("\nDatasets missing target columns:")
    for dataset in target_missing_datasets:
        print(" -", dataset)

print("\nSaved artifact verification:")
for dataset, result in saved_artifact_results.items():
    print(f"  {dataset}: {result['artifact_status']}")

print(
    "\nNotebook 13 generation summary saved. "
    "Review unresolved quality findings before using the outputs in research."
)


NOTEBOOK 13 — COMPLETION SUMMARY
NOTEBOOK 13 — COMPLETION SUMMARY
Generation status         : COMPLETED
Requested datasets        : 3
Generated datasets        : 3
Failed/skipped datasets   : 0
Malformed datasets        : 0
Missing target datasets   : 0
Structural pass count     : 3
Structural review count   : 0
Artifact verification     : PASS
Overall verification      : STRUCTURALLY_VERIFIED
Quality review status     : REVIEW_REQUIRED
Synthetic data directory  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets
Manifest                  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/manifests/synthetic_generation_manifest.json
Metadata                  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/sppgan_synthetic_generation_metadata.json
Final verification        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/final_verification.csv
Completion summary

In [104]:

# =============================================================================
# NOTEBOOK 13 — FINAL GOOGLE DRIVE PERSISTENCE AUDIT
# =============================================================================

from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")
NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"
NB12_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_12"
NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"

datasets = ["adult_income", "bank_marketing", "diabetes_130us"]

# Key final artifacts expected from Notebook 13.
expected_files = [
    NB13_ROOT / "synthetic_datasets" / f"{d}_sppgan_synthetic.csv"
    for d in datasets
]

expected_files += [
    NB13_ROOT / "metadata" / "sppgan_synthetic_generation_metadata.json",
    NB13_ROOT / "metadata" / "notebook_13_completion_summary.json",
    NB13_ROOT / "manifests" / "synthetic_generation_manifest.csv",
    NB13_ROOT / "manifests" / "synthetic_generation_manifest.json",
    NB13_ROOT / "validation" / "final_verification.csv",
    NB13_ROOT / "validation" / "final_verification_summary.json",
]

audit_rows = []

for path in expected_files:
    exists = path.is_file()
    readable = False
    details = ""

    if exists:
        try:
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path, low_memory=False)
                details = f"rows={len(df)}, columns={len(df.columns)}"
            elif path.suffix.lower() == ".json":
                with open(path, "r", encoding="utf-8") as f:
                    json.load(f)
                details = "valid JSON"
            else:
                details = "file exists"

            readable = True
        except Exception as exc:
            details = f"{type(exc).__name__}: {exc}"

    audit_rows.append({
        "artifact": str(path.relative_to(PROJECT_ROOT)),
        "exists": exists,
        "readable": readable,
        "size_bytes": path.stat().st_size if exists else None,
        "status": "PASS" if exists and readable else "REVIEW",
        "details": details,
    })

# Check for Notebook 12 final model checkpoints.
model_rows = []

for dataset in datasets:
    model_dir = NB12_ROOT / "models" / "final" / dataset
    checkpoints = sorted(model_dir.glob("*")) if model_dir.is_dir() else []
    checkpoints = [p for p in checkpoints if p.is_file()]

    model_rows.append({
        "dataset": dataset,
        "model_directory_exists": model_dir.is_dir(),
        "checkpoint_file_count": len(checkpoints),
        "checkpoint_files": "; ".join(p.name for p in checkpoints),
        "status": "PASS" if checkpoints else "REVIEW",
    })

# Inspect Notebook 02 processed artifacts without assuming filenames.
processed_files = (
    [p for p in NB02_ROOT.rglob("*") if p.is_file()]
    if NB02_ROOT.exists()
    else []
)

print("=" * 100)
print("NOTEBOOK 13 — PERSISTENCE AUDIT")
print("=" * 100)

AUDIT_DF = pd.DataFrame(audit_rows)
MODEL_AUDIT_DF = pd.DataFrame(model_rows)

display(AUDIT_DF)
print("\nNOTEBOOK 12 MODEL CHECKPOINT DISCOVERY")
display(MODEL_AUDIT_DF)

print("\nNOTEBOOK 02 PROCESSED DATA")
print("Processed-data directory exists:", NB02_ROOT.is_dir())
print("Files discovered:", len(processed_files))
print("First 20 files:")
for path in processed_files[:20]:
    print(" -", path.relative_to(PROJECT_ROOT))

print("\nSUMMARY")
print("Notebook 13 expected artifacts passing:",
      int(AUDIT_DF["status"].eq("PASS").sum()), "/", len(AUDIT_DF))
print("Notebook 12 datasets with checkpoint files:",
      int(MODEL_AUDIT_DF["status"].eq("PASS").sum()), "/", len(datasets))

print(
    "\nNOTE: Finding checkpoint files does not prove that the correct model "
    "can be loaded. A fresh-session model-loading test is still required."
)


NOTEBOOK 13 — PERSISTENCE AUDIT


,artifact,exists,readable,size_bytes,status,details
0,results/notebooks/notebook_13/synthetic_datase...,True,True,3912319,PASS,"rows=34189, columns=15"
1,results/notebooks/notebook_13/synthetic_datase...,True,True,2662749,PASS,"rows=31647, columns=17"
2,results/notebooks/notebook_13/synthetic_datase...,True,True,13017837,PASS,"rows=71236, columns=48"
3,results/notebooks/notebook_13/metadata/sppgan_...,True,True,10443,PASS,valid JSON
4,results/notebooks/notebook_13/metadata/noteboo...,True,True,6443,PASS,valid JSON
5,results/notebooks/notebook_13/manifests/synthe...,True,True,6710,PASS,"rows=3, columns=15"
6,results/notebooks/notebook_13/manifests/synthe...,True,True,7564,PASS,valid JSON
7,results/notebooks/notebook_13/validation/final...,True,True,1491,PASS,"rows=3, columns=26"
8,results/notebooks/notebook_13/validation/final...,True,True,1511,PASS,valid JSON



NOTEBOOK 12 MODEL CHECKPOINT DISCOVERY


,dataset,model_directory_exists,checkpoint_file_count,checkpoint_files,status
0,adult_income,True,5,adult_income_sppgan_dp_repetition_01_final.pt;...,PASS
1,bank_marketing,True,5,bank_marketing_sppgan_dp_repetition_01_final.p...,PASS
2,diabetes_130us,True,5,diabetes_130us_sppgan_dp_repetition_01_final.p...,PASS



NOTEBOOK 02 PROCESSED DATA
Processed-data directory exists: True
Files discovered: 38
First 20 files:
 - data/processed/notebook_02/native/native_dataset_manifest.csv
 - data/processed/notebook_02/preprocessors/preprocessor_manifest.csv
 - data/processed/notebook_02/schemas/notebook_02_artifact_summary.csv
 - data/processed/notebook_02/schemas/notebook_02_completion_summary.csv
 - data/processed/notebook_02/schemas/final_integrity_verification.csv
 - data/processed/notebook_02/schemas/final_integrity_metadata.json
 - data/processed/notebook_02/schemas/reload_validation.csv
 - data/processed/notebook_02/schemas/preprocessor_manifest.csv
 - data/processed/notebook_02/schemas/section_24_final_integrity_manifest.json
 - data/processed/notebook_02/schemas/notebook_02_completion_metadata.json
 - data/processed/notebook_02/schemas/notebook_02_dataset_completion.csv
 - data/processed/notebook_02/schemas/notebook_02_preprocessor_artifacts.csv
 - data/processed/notebook_02/schemas/notebook_02_f